# GOKO v2 — architecture POC

Runs the pipeline end to end on a folder of `.txt` notes and prints the payload at every
boundary. Each stage is its own cell that reads globals from the previous stage, so you can
stop anywhere and inspect what the next stage will receive.

**Run order matters.** Cells read state the previous cells built. Run them top to bottom.
Cell 19b (remap) rebuilds the state it owns from scratch on every run rather than appending
to it, so re-running is safe; if you edit a cell in the middle, re-run everything below it.
Cell numbers in the prose are the section numbers in these headings.

**Three providers.** One toggle, `PROVIDER`, in cell 1.

| `PROVIDER` | What it does |
|---|---|
| `"offline"` | Replays recorded extractions from `fixtures/offline_extractions.json` for the bundled notes. No network, no key. A note with no recorded payload fails loudly. |
| `"gemini"` | Paste an API key into `GEMINI_API_KEY` and go — no `settings.env`, no SDK to install. Falls back to `$GEMINI_API_KEY` / `$GOOGLE_API_KEY` if the variable is left empty. |
| `"azure"` | Calls your Azure deployment. Reads `settings.env`, needs the `openai` package. |

Offline mode exists so the *plumbing* can be verified without a deployment, and it is never
selected silently: name a live provider with no key and cell 1 raises, naming the toggle.
Every artifact is stamped with the model that produced it, so an offline run can never be
mistaken for a real one.

Gemini and Azure differ in their structured-output dialects. The notebook keeps one schema
and translates (`to_gemini_schema`, cell 2) rather than maintaining two.

**What this POC does not implement.** The architecture trace describes more than this
notebook runs. Cell 24 lists the gaps explicitly so the run summary cannot be mistaken for a
complete implementation.


## 1 — Configuration

Edit this cell only. It prints which keys your `settings.env` actually contains (values
masked) so you can map them without opening the file.


In [ ]:
import os, json, re, sys
from pathlib import Path

# ---- EDIT THESE ------------------------------------------------------------
PROVIDER   = "offline"          # "offline" | "gemini" | "openai" | "azure"
                                # offline replays fixtures; the live default is "openai"
                                # with gpt-6-luna at reasoning effort medium

# Gemini: paste a key here and set PROVIDER = "gemini". No settings.env needed.
# Falls back to $GEMINI_API_KEY / $GOOGLE_API_KEY if left empty.
GEMINI_API_KEY = ""
GEMINI_MODEL   = "gemini-3.1-pro-preview"   # 2.5-pro is closed to new API users

# OpenAI (api.openai.com directly, not Azure): paste a key or leave empty to use
# $OPENAI_API_KEY.
OPENAI_API_KEY = ""
OPENAI_MODEL   = "gpt-6-luna"   # the live default: extraction + categories

ENV_FILE   = "settings.env"     # only read for the Azure path
NOTES_DIR  = "./notes"          # folder of {claim}_{note_id}.txt files
OUT_DIR    = "./poc_output"
FIXTURES   = "./fixtures/offline_extractions.json"
MODEL      = "gpt-luna-5.6"     # Azure *deployment name*, not the model family
API_VERSION_FALLBACK = "2024-10-21"

# Optional stages. Each can be switched off without breaking anything downstream.
CLEANER    = "none"             # "none" | "clean-1.4"   (cell 7)
CHUNKING   = True               # split long notes before extraction (cell 9b)
CHUNK_MAX_CHARS = 30000         #   a note longer than this is split
CHUNK_OVERLAP_SENTENCES = 2     #   whole sentences repeated across a boundary
EXTRACT_CONCURRENCY = 4         # parallel extraction calls (1 = sequential)
GLINER_ENABLED = False          # Lane B, cell 9; needs `pip install gliner`
# ----------------------------------------------------------------------------

if PROVIDER not in ("offline", "gemini", "openai", "azure"):
    raise ValueError(f"PROVIDER must be offline, gemini, openai or azure (got {PROVIDER!r})")

# Derived. Everything downstream still reads OFFLINE_MODE, so a run stays
# self-describing in the stamped artifacts.
OFFLINE_MODE = (PROVIDER == "offline")

def load_env(path):
    found = {}
    p = Path(path)
    if not p.exists():
        print(f"!! {path} not found in {Path.cwd()}")
        return found
    for line in p.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k, v = k.strip(), v.strip().strip('"').strip("'")
        found[k] = v
        os.environ.setdefault(k, v)
    return found

ENV = load_env(ENV_FILE) if PROVIDER == "azure" else {}

def mask(v):
    if v is None: return None
    return v if len(v) < 12 else v[:6] + "..." + v[-4:]

if PROVIDER == "azure":
    print(f"keys found in {ENV_FILE}: {len(ENV)}")
    for k in sorted(ENV):
        print(f"  {k:<40} = {mask(ENV[k])}")

# Best-effort mapping across the usual Azure naming variants.
def pick(*names):
    for n in names:
        if os.environ.get(n):
            return os.environ[n]
    return None

AZURE_ENDPOINT = pick("AZURE_OPENAI_ENDPOINT", "AZURE_OAI_ENDPOINT",
                      "OPENAI_API_BASE", "AZURE_ENDPOINT")
AZURE_KEY      = pick("AZURE_OPENAI_API_KEY", "AZURE_OAI_KEY", "AZURE_OPENAI_KEY",
                      "OPENAI_API_KEY", "AZURE_API_KEY")
AZURE_VERSION  = pick("AZURE_OPENAI_API_VERSION", "OPENAI_API_VERSION",
                      "AZURE_API_VERSION") or API_VERSION_FALLBACK
DEPLOYMENT     = pick("AZURE_OPENAI_DEPLOYMENT", "AZURE_OAI_DEPLOYMENT",
                      "DEPLOYMENT_NAME") or MODEL

GEMINI_KEY = GEMINI_API_KEY or pick("GEMINI_API_KEY", "GOOGLE_API_KEY",
                                   "GOOGLE_GENAI_API_KEY")
OPENAI_KEY = OPENAI_API_KEY or pick("OPENAI_API_KEY")

print(f"\nprovider   = {PROVIDER}")
if PROVIDER == "azure":
    print(f"  endpoint   = {AZURE_ENDPOINT}")
    print(f"  key        = {mask(AZURE_KEY)}")
    print(f"  api_version= {AZURE_VERSION}")
    print(f"  deployment = {DEPLOYMENT}")
elif PROVIDER == "gemini":
    print(f"  model      = {GEMINI_MODEL}")
    print(f"  key        = {mask(GEMINI_KEY)}"
          f"{'  (from environment)' if GEMINI_KEY and not GEMINI_API_KEY else ''}")
elif PROVIDER == "openai":
    print(f"  model      = {OPENAI_MODEL}")
    print(f"  key        = {mask(OPENAI_KEY)}"
          f"{'  (from environment)' if OPENAI_KEY and not OPENAI_API_KEY else ''}")

# No provider silently falls back to fixtures: an offline run and a live run are
# different experiments and must never be confused in the output.
if PROVIDER == "azure" and not (AZURE_ENDPOINT and AZURE_KEY):
    raise RuntimeError(
        "PROVIDER is 'azure' but no endpoint/key resolved.\n"
        "Set them in settings.env, or assign AZURE_ENDPOINT / AZURE_KEY by hand in this "
        "cell, or set PROVIDER = 'offline' to replay the bundled fixtures.")
if PROVIDER == "openai" and not OPENAI_KEY:
    raise RuntimeError(
        "PROVIDER is 'openai' but no key resolved.\n"
        "Paste one into OPENAI_API_KEY above, or export OPENAI_API_KEY, or set "
        "PROVIDER = 'offline' to replay the bundled fixtures.")
if PROVIDER == "gemini" and not GEMINI_KEY:
    raise RuntimeError(
        "PROVIDER is 'gemini' but no key resolved.\n"
        "Paste one into GEMINI_API_KEY above, or export GEMINI_API_KEY, or set "
        "PROVIDER = 'offline' to replay the bundled fixtures.")

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)


## 2 — Client

Three clients, one surface. All expose `chat.completions.create(...)` and return an object
with `.choices[0].message.content`, `.choices[0].finish_reason` and `.usage`, so no
downstream cell knows or cares which provider answered.

The Gemini path is a short adapter over raw HTTPS rather than an SDK — nothing to install,
and the request stays visible where you can read it. Two translations earn their keep:
`to_gemini_schema` converts the strict-mode schema into Gemini's `responseSchema` dialect
(type unions become `nullable`, `additionalProperties` is dropped, property order is
pinned), and `finishReason: "MAX_TOKENS"` is normalised to `"length"` so cell 10's
truncation check works unchanged across providers.

`USE_STRUCTURED = False` falls back to prompt-instructed JSON plus validation — use it to
compare a deployment that does not support strict structured outputs.


In [ ]:
RUN_MODEL_TAG = "offline-replay" if OFFLINE_MODE else None

class OfflineReplayClient:
    """Replays recorded payloads. Raises for any note it has no recording for.

    The point of this class is that it cannot invent an extraction. A missing fixture is an
    error that lands in the run's failure accounting, not an empty result that looks like a
    note with nothing in it."""

    class _Completions:
        def __init__(self, outer): self.outer = outer
        def create(self, **kw):
            return self.outer._create(**kw)

    class _Chat:
        def __init__(self, outer): self.completions = OfflineReplayClient._Completions(outer)

    class _Usage:
        prompt_tokens = 0
        completion_tokens = 0
        def __repr__(self): return "usage(offline: 0/0)"

    def __init__(self, fixtures_path):
        self.chat = OfflineReplayClient._Chat(self)
        p = Path(fixtures_path)
        if not p.exists():
            raise FileNotFoundError(
                f"OFFLINE_MODE is on but {fixtures_path} is missing. It holds the recorded "
                f"payloads the replay client serves.")
        doc = json.loads(p.read_text())
        self.extractions = doc["extractions"]
        self.categories  = doc.get("categories", {})

    def _wrap(self, payload):
        usage = OfflineReplayClient._Usage()
        msg    = type("M", (), {"content": json.dumps(payload)})()
        choice = type("C", (), {"message": msg, "finish_reason": "stop"})()
        return type("R", (), {"choices": [choice], "usage": usage})()

    def _create(self, **kw):
        user = kw["messages"][-1]["content"]
        if "Reply with the single word" in user:
            msg    = type("M", (), {"content": "ready"})()
            choice = type("C", (), {"message": msg, "finish_reason": "stop"})()
            return type("R", (), {"choices": [choice],
                                  "usage": OfflineReplayClient._Usage()})()
        m = re.search(r"<note_id>(\d+)</note_id>", user)
        if m:
            nid = m.group(1)
            if nid not in self.extractions:
                raise RuntimeError(
                    f"offline_fixture_missing: no recorded extraction for note {nid}. "
                    f"Offline mode replays the bundled notes only — set PROVIDER to "
                    f"'gemini' or 'azure' to process new notes.")
            return self._wrap(self.extractions[nid])
        m = re.search(r'"entity_id":\s*"([^"]+)"', user)
        if m:
            eid = m.group(1)
            if eid not in self.categories:
                raise RuntimeError(
                    f"offline_fixture_missing: no recorded category for entity {eid}.")
            return self._wrap(self.categories[eid])
        raise RuntimeError("offline_fixture_missing: unrecognised request shape")


# --- Gemini -----------------------------------------------------------------
# Adapter, not an SDK: it exposes the same `chat.completions.create(...)` surface
# the rest of the notebook already calls, so no downstream cell knows or cares
# which provider answered. Raw HTTPS via urllib, so there is nothing to install.

GEMINI_ENDPOINT = ("https://generativelanguage.googleapis.com/v1beta/models/"
                   "{model}:generateContent")

# Gemini 2.5+/3.x are thinking models, and thinking tokens are charged against
# maxOutputTokens. Callers here pass max_tokens meaning "room for the answer";
# passing it through unchanged let thinking consume the whole budget before a
# single answer token was written (every category call truncated at 512 on the
# first live run). The adapter adds headroom so max_tokens keeps its meaning on
# every provider. The truncation check still fires if the answer itself overruns.
GEMINI_THINKING_HEADROOM = 8192

# Whole-note extraction of a 100-page filing can run for minutes; a short fixed
# timeout turns a slow success into an extraction failure.
GEMINI_TIMEOUT_S = 900

# Try IPv4 addresses before IPv6. Where a machine's IPv6 route is broken, urllib
# waits out a full timeout per IPv6 address before trying IPv4 -- measured at ~55 s
# added to every call. goko/net.py has the details; IPv6 stays as the fallback.
sys.path.insert(0, str(Path.cwd()))
from goko.net import prefer_ipv4
prefer_ipv4()

# Rate limits are a wait, not a failure. A 429 (quota per minute) or 503 (overloaded)
# is retried after the delay the server asks for, else with exponential backoff; only
# when the retries run out does the call fail, and then it is counted as a failure.
GEMINI_RETRIES = 6
import time as _time
import urllib.request, urllib.error

def gemini_post(req):
    for attempt in range(GEMINI_RETRIES + 1):
        try:
            with urllib.request.urlopen(req, timeout=GEMINI_TIMEOUT_S) as r:
                return json.loads(r.read().decode("utf-8"))
        except urllib.error.HTTPError as e:
            detail = e.read().decode("utf-8", "replace")
            # a per-day quota does not come back in minutes, whatever retryDelay says
            per_day = "per_day" in detail or "PerDay" in detail
            if e.code not in (429, 500, 503) or attempt == GEMINI_RETRIES or per_day:
                raise RuntimeError(f"Gemini HTTP {e.code}"
                                   f"{' (daily quota exhausted)' if per_day else ''}: {detail[:600]}") from None
            m = re.search(r'"retryDelay":\s*"(\d+(?:\.\d+)?)s"', detail)
            wait = float(m.group(1)) + 1 if m else min(60, 2 ** attempt * 2)
            print(f"      (Gemini HTTP {e.code}; retrying in {wait:.0f}s, attempt {attempt + 1})")
            _time.sleep(wait)

# Gemini's responseSchema is a different dialect from OpenAI strict mode. The
# differences are small and total: translate rather than maintain two schemas.
def to_gemini_schema(node):
    """OpenAI strict JSON Schema -> Gemini responseSchema."""
    if isinstance(node, list):
        return [to_gemini_schema(x) for x in node]
    if not isinstance(node, dict):
        return node

    out = {}
    t = node.get("type")
    # OpenAI expresses nullable as a type union; Gemini has a `nullable` flag.
    if isinstance(t, list):
        non_null = [x for x in t if x != "null"]
        if len(non_null) != 1:
            raise ValueError(f"cannot translate type union {t!r} for Gemini")
        out["type"] = non_null[0]
        if "null" in t:
            out["nullable"] = True
    elif t is not None:
        out["type"] = t

    for k in ("description", "enum"):
        if k in node:
            out[k] = node[k]
    if "items" in node:
        out["items"] = to_gemini_schema(node["items"])
    if "properties" in node:
        out["properties"] = {k: to_gemini_schema(v)
                             for k, v in node["properties"].items()}
        # Gemini emits keys in whatever order it likes unless told; pinning the
        # order keeps responses diffable across runs.
        out["propertyOrdering"] = list(node["properties"].keys())
    if "required" in node:
        out["required"] = list(node["required"])
    # `additionalProperties` is required by OpenAI strict mode and rejected by
    # Gemini. Dropped deliberately, not overlooked.
    return out


class GeminiClient:
    """Minimal Gemini adapter with an OpenAI-shaped surface."""

    class _Completions:
        def __init__(self, outer): self.outer = outer
        def create(self, **kw): return self.outer._create(**kw)

    class _Chat:
        def __init__(self, outer): self.completions = GeminiClient._Completions(outer)

    def __init__(self, api_key, model):
        if not api_key:
            raise ValueError("GeminiClient needs an API key")
        self.api_key, self.model = api_key, model
        self.chat = GeminiClient._Chat(self)

    def _create(self, **kw):
        import urllib.request, urllib.error

        msgs   = kw["messages"]
        system = "\n\n".join(m["content"] for m in msgs if m["role"] == "system")
        turns  = [{"role": ("model" if m["role"] == "assistant" else "user"),
                   "parts": [{"text": m["content"]}]}
                  for m in msgs if m["role"] != "system"]

        gen = {"temperature": kw.get("temperature", 0.0),
               "maxOutputTokens": kw.get("max_tokens", 4096) + GEMINI_THINKING_HEADROOM}
        rf = kw.get("response_format") or {}
        if rf.get("type") == "json_schema":
            gen["responseMimeType"] = "application/json"
            gen["responseSchema"] = to_gemini_schema(rf["json_schema"]["schema"])
        elif rf.get("type") == "json_object":
            gen["responseMimeType"] = "application/json"

        body = {"contents": turns, "generationConfig": gen}
        if system:
            body["systemInstruction"] = {"parts": [{"text": system}]}

        req = urllib.request.Request(
            GEMINI_ENDPOINT.format(model=kw.get("model") or self.model),
            data=json.dumps(body).encode("utf-8"),
            headers={"Content-Type": "application/json",
                     "x-goog-api-key": self.api_key},   # key in a header, not the URL
            method="POST")
        payload = gemini_post(req)

        if not payload.get("candidates"):
            # Safety blocks and prompt rejections land here with no candidate.
            raise RuntimeError(f"Gemini returned no candidate: "
                               f"{json.dumps(payload)[:600]}")
        cand = payload["candidates"][0]
        parts = cand.get("content", {}).get("parts", [])
        # Thought parts only appear if thought summaries are requested; skip them
        # defensively so they can never be parsed as the JSON answer.
        text  = "".join(p.get("text", "") for p in parts if not p.get("thought"))

        # Normalise the finish reason to the OpenAI vocabulary so the existing
        # truncation check in cell 10 keeps working unchanged.
        finish = "length" if cand.get("finishReason") == "MAX_TOKENS" else "stop"
        um = payload.get("usageMetadata", {})
        usage  = type("U", (), {"prompt_tokens": um.get("promptTokenCount", 0),
                                "completion_tokens": um.get("candidatesTokenCount", 0),
                                "thinking_tokens": um.get("thoughtsTokenCount", 0),
                                "__repr__": lambda s: f"usage(prompt={s.prompt_tokens}, "
                                                      f"completion={s.completion_tokens}, "
                                                      f"thinking={s.thinking_tokens})"})()
        msg    = type("M", (), {"content": text})()
        choice = type("C", (), {"message": msg, "finish_reason": finish})()
        return type("R", (), {"choices": [choice], "usage": usage})()


# --- OpenAI (api.openai.com, not Azure) ------------------------------------------
# Same idea as the Gemini adapter: raw HTTPS, OpenAI-shaped surface, nothing to install.
# The request body is already in OpenAI's own dialect, so only two things change:
#  - GPT-5 models are reasoning models. They take `max_completion_tokens`, which, like
#    Gemini's budget, is charged for reasoning tokens as well as the answer; the adapter
#    adds headroom so `max_tokens` keeps meaning "room for the answer".
#  - They accept only the default temperature, so it is not sent.
OPENAI_ENDPOINT = "https://api.openai.com/v1/chat/completions"
OPENAI_REASONING_HEADROOM = 16384
OPENAI_REASONING_EFFORT = "medium"   # low drops most actions and identifiers (README)

def openai_post(req):
    for attempt in range(GEMINI_RETRIES + 1):
        try:
            with urllib.request.urlopen(req, timeout=GEMINI_TIMEOUT_S) as r:
                return json.loads(r.read().decode("utf-8"))
        except urllib.error.HTTPError as e:
            detail = e.read().decode("utf-8", "replace")
            quota = "insufficient_quota" in detail      # billing, not a rate: never recovers
            if e.code not in (429, 500, 502, 503) or attempt == GEMINI_RETRIES or quota:
                raise RuntimeError(f"OpenAI HTTP {e.code}: {detail[:600]}") from None
            wait = float(e.headers.get("retry-after") or min(60, 2 ** attempt * 2))
            print(f"      (OpenAI HTTP {e.code}; retrying in {wait:.0f}s, attempt {attempt + 1})")
            _time.sleep(wait)

class OpenAIClient:
    """Minimal OpenAI Chat Completions adapter with the same surface as the others."""

    class _Completions:
        def __init__(self, outer): self.outer = outer
        def create(self, **kw): return self.outer._create(**kw)

    class _Chat:
        def __init__(self, outer): self.completions = OpenAIClient._Completions(outer)

    def __init__(self, api_key, model):
        if not api_key:
            raise ValueError("OpenAIClient needs an API key")
        self.api_key, self.model = api_key, model
        self.chat = OpenAIClient._Chat(self)

    def _create(self, **kw):
        body = {"model": kw.get("model") or self.model, "messages": kw["messages"],
                "max_completion_tokens": kw.get("max_tokens", 4096) + OPENAI_REASONING_HEADROOM}
        # Reasoning families take reasoning_effort and reject temperature. gpt-6 was missing
        # here: it was sent temperature and no effort (found in the model sweep).
        if re.match(r"(gpt-[5-9]|o\d)", body["model"]):
            body["reasoning_effort"] = OPENAI_REASONING_EFFORT
        else:
            body["temperature"] = kw.get("temperature", 0.0)
        if kw.get("response_format"):
            body["response_format"] = kw["response_format"]   # strict json_schema, as written
        req = urllib.request.Request(
            OPENAI_ENDPOINT, data=json.dumps(body).encode("utf-8"),
            headers={"Content-Type": "application/json",
                     "Authorization": f"Bearer {self.api_key}"}, method="POST")
        payload = openai_post(req)
        ch = payload["choices"][0]
        if ch["message"].get("refusal"):
            raise RuntimeError(f"OpenAI refused: {ch['message']['refusal'][:300]}")
        um = payload.get("usage", {})
        usage = type("U", (), {
            "prompt_tokens": um.get("prompt_tokens", 0),
            "completion_tokens": um.get("completion_tokens", 0),
            "thinking_tokens": (um.get("completion_tokens_details") or {}).get("reasoning_tokens", 0),
            "__repr__": lambda s: f"usage(prompt={s.prompt_tokens}, "
                                  f"completion={s.completion_tokens}, reasoning={s.thinking_tokens})"})()
        msg = type("M", (), {"content": ch["message"].get("content") or ""})()
        choice = type("C", (), {"message": msg, "finish_reason": ch.get("finish_reason")})()
        return type("R", (), {"choices": [choice], "usage": usage})()


if OFFLINE_MODE:
    client = OfflineReplayClient(FIXTURES)
    DEPLOYMENT = RUN_MODEL_TAG
    print("!! OFFLINE REPLAY CLIENT — no network call will be made.")
    print("!! Every artifact this run writes is stamped model='offline-replay'.")
    print(f"   fixtures: {FIXTURES}  notes recorded: {sorted(client.extractions)}")
elif PROVIDER == "gemini":
    client = GeminiClient(GEMINI_KEY, GEMINI_MODEL)
    DEPLOYMENT = GEMINI_MODEL
    print(f"client ready -> Gemini {GEMINI_MODEL}")
elif PROVIDER == "openai":
    client = OpenAIClient(OPENAI_KEY, OPENAI_MODEL)
    DEPLOYMENT = OPENAI_MODEL
    print(f"client ready -> OpenAI {OPENAI_MODEL} (api.openai.com)")
else:
    try:
        from openai import AzureOpenAI
    except ImportError:
        print("pip install openai")
        raise
    client = AzureOpenAI(
        azure_endpoint=AZURE_ENDPOINT,
        api_key=AZURE_KEY,
        api_version=AZURE_VERSION,
    )
    print(f"client ready -> {AZURE_ENDPOINT}")

USE_STRUCTURED = True     # flip to False to compare against prompt-only JSON
TEMPERATURE    = 0.0
MAX_TOKENS     = 4096     # extraction of a long note needs room; truncation is detected

print(f"model        -> {DEPLOYMENT}")
print(f"structured   -> {USE_STRUCTURED}")


## 3 — Smoke test

One trivial call. Fail here rather than 200 lines deep.


In [ ]:
r = client.chat.completions.create(
    model=DEPLOYMENT,
    messages=[{"role": "user", "content": "Reply with the single word: ready"}],
    temperature=TEMPERATURE,
    max_tokens=10,
)
print("response:", r.choices[0].message.content)
print("usage   :", r.usage)


## 4 — Extraction schema v0.1

Closed enums for entity and detail types. `action_type` and `subcategory` stay open — that
is the overflow channel you mine later to decide what gets promoted.

`lint_strict_schema` runs before the first call. Strict structured outputs reject a schema
that uses validation keywords like `minItems`, and the error arrives as a 400 at call time —
one per note, inside the extraction loop's exception handler. Checking the schema here turns
a whole run that silently extracts nothing into a failure on the cell that owns the mistake.


In [ ]:
SCHEMA_VERSION = "0.1"

ENTITY_TYPES = ["person", "organization", "vehicle"]
DETAIL_TYPES = ["address", "phone", "tin", "ssn", "npi", "bar_number", "vin",
                "email", "dob", "license_plate", "dea_number", "state_license", "bank_account"]
# What each type means, for the prompt. Policy and claim numbers are deliberately absent:
# they identify a contract or a file, not a party.
DETAIL_GUIDE = {
    "address": "a street address",
    "phone": "a telephone or fax number",
    "tin": "a federal tax id / EIN (NN-NNNNNNN)",
    "ssn": "a social security number",
    "npi": "a National Provider Identifier (10 digits)",
    "bar_number": "an attorney's bar or registration number",
    "vin": "a vehicle identification number (17 characters)",
    "email": "an email address",
    "dob": "a person's date of birth (not any other date)",
    "license_plate": "a vehicle license plate; issuer = the issuing state if stated",
    "dea_number": "a DEA registration number (2 letters + 7 digits)",
    "state_license": "a professional license number (medical, nursing, chiropractic, "
                     "law, ...); issuer = the licensing state if stated. Not a driver's "
                     "license",
    "bank_account": "a bank account number; issuer = the bank's 9-digit ABA routing "
                    "number if stated",
}
STANCES      = ["asserted", "denied", "disputed", "alleged"]
BASES        = ["stated", "inferred"]
# One rule for every quote the model writes (cell 11 places quotes by search, so a quote
# that is edited, stitched or repeated cannot be placed).
QUOTE_RULE = ("One short contiguous stretch of the note (one clause, ideally under 120 "
              "characters), copied character for character including odd spacing, line "
              "breaks, hyphenation, typos and OCR artifacts. Never skip, insert or reorder "
              "words, never substitute names, never join text from two places. Unique in the "
              "note: where the note repeats the words, include a distinctive neighbour such "
              "as the paragraph number.")

EXTRACTION_SCHEMA = {
  "type": "object",
  "additionalProperties": False,
  "required": ["entity_mentions", "detail_mentions", "action_mentions"],
  "properties": {
    "entity_mentions": {
      "type": "array",
      "items": {
        "type": "object", "additionalProperties": False,
        "required": ["mention_id", "quote", "name", "type", "occurrences"],
        "properties": {
          "mention_id": {"type": "string",
            "description": "Note-local id: m1, m2, m3."},
          "quote": {"type": "string",
            "description": QUOTE_RULE + " It contains the party's name."},
          "name": {"type": "string",
            "description": "The party's name or designation alone, exactly as written and "
                           "contained in quote, with no surrounding words. "
                           "E.g. 'Dr. Monroe', 'Lakeshore PT', 'the claimant'."},
          "type": {"type": "string", "enum": ENTITY_TYPES,
            "description": "Do not guess. If the party fits none of these, omit it."},
          "occurrences": {"type": "array", "items": {"type": "string"},
            "description": "Every other verbatim phrase in this note referring to the "
                           "same party, including pronouns and role references."},
        }}},
    "detail_mentions": {
      "type": "array",
      "items": {
        "type": "object", "additionalProperties": False,
        "required": ["detail_id","quote","raw_value","detail_type",
                     "issuer","owner_ref","basis"],
        "properties": {
          "detail_id": {"type": "string"},
          "quote": {"type": "string",
            "description": QUOTE_RULE + " It contains the value."},
          "raw_value": {"type": "string",
            "description": "The value alone, exactly as written. Do not reformat."},
          "detail_type": {"type": "string", "enum": DETAIL_TYPES,
            "description": "; ".join(f"{k}: {v}" for k, v in DETAIL_GUIDE.items())
                           + ". Never a policy or claim number."},
          "issuer": {"type": ["string", "null"],
            "description": "Issuing state or jurisdiction where stated (for bank_account: "
                           "the ABA routing number). Never infer."},
          "owner_ref": {"type": "string",
            "description": "mention_id of the owning party, or the literal string "
                           "UNASSIGNED. Proximity in the text is not ownership."},
          "basis": {"type": "string", "enum": BASES},
        }}},
    "action_mentions": {
      "type": "array",
      "items": {
        "type": "object", "additionalProperties": False,
        "required": ["action_id","quote","action_type","participants",
                     "time_qualifier","stance","basis"],
        "properties": {
          "action_id": {"type": "string"},
          "quote": {"type": "string",
            "description": QUOTE_RULE + " It states the action."},
          "action_type": {"type": "string",
            "description": "Short verb phrase from the text, lowercase with "
                           "underscores, e.g. referred_to, treated_by."},
          "participants": {"type": "array",
            "description": "At least one participant. An action with no participant "
                           "cannot be attached to anything, so omit it instead.",
            "items": {"type": "object", "additionalProperties": False,
              "required": ["mention_id", "role"],
              "properties": {"mention_id": {"type": "string"},
                             "role": {"type": "string"}}}},
          "time_qualifier": {"type": ["string", "null"],
            "description": "As written. Do not resolve relative dates."},
          "stance": {"type": "string", "enum": STANCES},
          "basis": {"type": "string", "enum": BASES},
        }}},
  }}

# Keywords strict structured outputs reject. The API returns a 400 naming the keyword;
# catching it here instead means the failure names the cell that has to change.
STRICT_UNSUPPORTED = {"minItems", "maxItems", "minLength", "maxLength", "pattern",
                      "format", "minimum", "maximum", "default", "oneOf", "allOf", "not",
                      "uniqueItems", "multipleOf", "patternProperties"}

def lint_strict_schema(schema, path="$"):
    """Raises on anything strict mode will reject. Returns the count of objects checked."""
    n = 0
    if isinstance(schema, dict):
        bad = STRICT_UNSUPPORTED & set(schema)
        if bad:
            raise ValueError(
                f"strict schema violation at {path}: {sorted(bad)} not permitted under "
                f"strict structured outputs")
        if schema.get("type") == "object":
            n += 1
            if schema.get("additionalProperties") is not False:
                raise ValueError(f"strict schema violation at {path}: "
                                 f"additionalProperties must be false")
            props, req = set(schema.get("properties", {})), set(schema.get("required", []))
            if props != req:
                raise ValueError(
                    f"strict schema violation at {path}: every property must be required; "
                    f"missing from required: {sorted(props - req)}; "
                    f"required but absent: {sorted(req - props)}")
        for k, v in schema.items():
            n += lint_strict_schema(v, f"{path}.{k}")
    elif isinstance(schema, list):
        for i, v in enumerate(schema):
            n += lint_strict_schema(v, f"{path}[{i}]")
    return n

checked = lint_strict_schema(EXTRACTION_SCHEMA)
print(f"schema v{SCHEMA_VERSION}  (strict lint: {checked} object(s) OK)")
print(f"  entity types : {ENTITY_TYPES}")
print(f"  detail types : {DETAIL_TYPES}")
print(f"  action_type  : OPEN (overflow channel)")


## 5 — Claim id parser

`123456-789012-AB-01` → client, occurrence, coverage, sequence. Parsed strictly, never
sliced positionally — an off-format legacy id would otherwise produce a silently wrong
occurrence grouping.


In [ ]:
CLAIM_RE = re.compile(r"^(\d{6})-(\d{6})-([A-Za-z]{2})-(\d{2})$")
ID_FORMAT_VERSION = "1"

def parse_claim_id(raw):
    m = CLAIM_RE.match(raw.strip())
    if not m:
        return {"raw": raw, "valid": False, "flag": "malformed_claim_id"}
    client_id, occ, cov, seq = m.groups()
    return {
        "claim_id":       f"{client_id}-{occ}-{cov.upper()}-{seq}",
        "client_id":      client_id,
        "occurrence_id":  f"{client_id}-{occ}",
        "occurrence_seq": occ,
        "coverage_code":  cov.upper(),
        "claim_seq":      seq,
        "id_format_version": ID_FORMAT_VERSION,
        "valid": True,
    }

for t in ["123456-789012-AB-01", "123456-789012-ab-01", "LEGACY-4471902"]:
    print(f"{t:<24} -> {json.dumps(parse_claim_id(t))}")


## 6 — Load notes

Filename carries the claim id and note id. The filename pattern is anchored to the claim-id
format rather than a loose `(.+)_(\d{5,6})` — against a greedy prefix, `claim_1882130.txt`
matches with the note id silently truncated to `882130`. Malformed filenames are reported,
not skipped silently, and an empty load stops the run here instead of raising `IndexError`
two cells down.


In [ ]:
NOTE_FILE_RE = re.compile(
    r"^(?P<claim>\d{6}-\d{6}-[A-Za-z]{2}-\d{2})_(?P<note>\d+)\.txt$", re.IGNORECASE)

notes = []
bad_files = []
ndir = Path(NOTES_DIR)
if not ndir.exists():
    raise FileNotFoundError(f"{NOTES_DIR} does not exist (cwd={Path.cwd()})")

for f in sorted(ndir.glob("*.txt")):
    m = NOTE_FILE_RE.match(f.name)
    if not m:
        bad_files.append((f.name, "filename_pattern"))
        continue
    parsed = parse_claim_id(m.group("claim"))
    if not parsed.get("valid"):
        bad_files.append((f.name, "malformed_claim_id"))
        continue
    note_id = m.group("note")
    with open(f, encoding="utf-8", errors="replace", newline="") as fh:
        raw_text = fh.read()
    notes.append({
        "path": str(f),
        "note_id": int(note_id),
        "note_key": f"note:{note_id}",
        **{k: v for k, v in parsed.items() if k != "valid"},
        # newline="" keeps the file's own line endings. Path.read_text() translates
        # \r\n to \n on the way in, so "raw" text was never raw and a span could not
        # point at an exact position in the source file.
        "raw_text": raw_text,
    })

print(f"loaded {len(notes)} notes, {len(bad_files)} rejected")
for n in notes:
    print(f"  {n['note_key']:<14} claim={n['claim_id']}  occ={n['occurrence_id']}  "
          f"raw_len={len(n['raw_text'])}")
for f, why in bad_files:
    print(f"  REJECT {f}  ({why})")

dupes = {k for k in (n["note_key"] for n in notes)
         if [x["note_key"] for x in notes].count(k) > 1}
if dupes:
    raise RuntimeError(f"duplicate note ids across files: {sorted(dupes)} — note_key is a "
                       f"node key downstream and must be unique")

if not notes:
    raise RuntimeError(
        f"no usable notes in {NOTES_DIR}. Expected files named "
        f"{{claim_id}}_{{note_id}}.txt, e.g. 123456-789012-AB-01_188213.txt")

claims = sorted({n["claim_id"] for n in notes})
occurrences = sorted({n["occurrence_id"] for n in notes})
print(f"\n{len(claims)} claims across {len(occurrences)} occurrences")
for o in occurrences:
    cs = sorted({n['claim_id'] for n in notes if n['occurrence_id'] == o})
    print(f"  {o} -> {len(cs)} claim(s): {cs}")


## 7 — Cleaning (optional, off by default)

Cleaning is a pluggable stage chosen by `CLEANER` in cell 1. **It is off by default**
(`"none"`): the model reads the source text exactly as stored, every span is a position in
that text, and there is no second coordinate system to keep in sync.

It is off because it earned nothing and cost a lot. On the court corpus it shortened the text
by under 1%; the whitespace tolerance that actually rescues messy quotes lives in quote
resolution (cell 11) and works either way. Meanwhile the cleaned↔raw translation was the
source of most span bugs found so far, and verifying it was the only step with a real compute
cost (~25 s per pass on 626k characters).

`"clean-1.4"` keeps the old behaviour: collapse whitespace runs, normalise line endings, and
record every edit so a clean-text position can be translated back. The exhaustive check below
runs in both modes; with `"none"` it is trivially true.


In [ ]:
def clean_none(raw):
    """Identity cleaner: the text the model reads is the source text."""
    return raw, []

def clean_with_map(raw):
    """Returns (clean_text, edits). edits let us map clean_idx -> raw_idx."""
    out, edits = [], []
    i, ci = 0, 0
    while i < len(raw):
        if raw.startswith("\r\n", i):
            out.append("\n"); edits.append({"raw_start": i, "raw_len": 2,
                                            "clean_start": ci, "clean_len": 1,
                                            "kind": "crlf_to_lf"})
            i += 2; ci += 1
        elif raw[i] == "\r":                      # lone CR, legacy Mac line ending
            # Length-preserving, so the offsets never drift -- but the character
            # changed, and every change is recorded. An unrecorded substitution is
            # indistinguishable from an offset-map bug to anything checking later.
            out.append("\n"); edits.append({"raw_start": i, "raw_len": 1,
                                            "clean_start": ci, "clean_len": 1,
                                            "kind": "cr_to_lf"})
            i += 1; ci += 1
        elif raw[i] in " \t":
            j = i
            while j < len(raw) and raw[j] in " \t":
                j += 1
            run = j - i
            out.append(" ")
            # A single space is the only run that is not an edit. A single tab is a
            # substitution: offsets hold, the character does not.
            if run > 1 or raw[i] != " ":
                edits.append({"raw_start": i, "raw_len": run,
                              "clean_start": ci, "clean_len": 1,
                              "kind": "collapse_ws" if run > 1 else "tab_to_space"})
            i = j; ci += 1
        else:
            out.append(raw[i]); i += 1; ci += 1
    return "".join(out), edits

def clean_to_raw(clean_idx, edits, clean_len=None):
    """Translate a cleaned-text index back to a raw-text index.

    `edits` is in ascending clean_start order, so the loop can stop at the first edit at or
    after the index. An edit that *starts* at clean_idx has not been passed yet: its drift
    belongs to positions after it, not to the index itself."""
    raw_idx = clean_idx
    for e in edits:
        if e["clean_start"] < clean_idx:
            raw_idx += (e["raw_len"] - e["clean_len"])
        else:
            break
    return raw_idx

CLEANERS = {"none": clean_none, "clean-1.4": clean_with_map}
if CLEANER not in CLEANERS:
    raise ValueError(f"CLEANER must be one of {sorted(CLEANERS)} (got {CLEANER!r})")
CLEAN_POLICY = CLEANER

for n in notes:
    n["clean_text"], n["edits"] = CLEANERS[CLEANER](n["raw_text"])
    n["clean_policy"] = CLEAN_POLICY
print(f"cleaner: {CLEANER}")

# Exhaustive offset-map check. Cheap at POC scale; the property it proves is the one every
# citation depends on.
def check_offset_map(note):
    raw, clean, edits = note["raw_text"], note["clean_text"], note["edits"]
    bad = []
    collapsed = set()
    for e in edits:
        for k in range(e["clean_start"], e["clean_start"] + e["clean_len"]):
            collapsed.add(k)
    for ci in range(len(clean)):
        ri = clean_to_raw(ci, edits, len(clean))
        if ri >= len(raw):
            bad.append((ci, ri, "past end of raw")); continue
        if ci in collapsed:
            continue                      # collapsed run: clean char is a substitution
        if clean[ci] != raw[ri]:
            bad.append((ci, ri, f"{clean[ci]!r} != {raw[ri]!r}"))
    end = clean_to_raw(len(clean), edits, len(clean))
    if end != len(raw):
        bad.append((len(clean), end, f"end maps to {end}, raw_len={len(raw)}"))
    return bad

offset_problems = 0
for n in notes:
    bad = check_offset_map(n)
    offset_problems += len(bad)
    print(f"{n['note_key']}  raw_len={len(n['raw_text'])}  clean_len={len(n['clean_text'])}"
          f"  edits={len(n['edits'])}  drift={len(n['raw_text']) - len(n['clean_text'])}"
          f"  map_check={'OK' if not bad else f'{len(bad)} BAD'}")
    for b in bad[:5]:
        print(f"    !! clean={b[0]} -> raw={b[1]}  {b[2]}")

if offset_problems:
    raise RuntimeError(f"{offset_problems} offset-map error(s). Every span downstream would "
                       f"point at the wrong characters.")

n0 = notes[0]
print("\nfirst 3 edits of", n0["note_key"])
for e in n0["edits"][:3]:
    print("   ", json.dumps(e))


## 8 — Lane A: pattern scan and checksums

High recall on structured identifiers, zero ownership. Checksums catch transposed digits
that look perfectly well-formed to a language model — the failure class the LLM lane cannot
see.

Two pattern changes from the naive version. The phone pattern now requires a separator or
parentheses, so it no longer competes with a bare ten-digit NPI; and a bare ten-digit run is
recorded with `cue: false`, because "ten digits with no NPI label" is a much weaker claim
than "ten digits after the word NPI" and the two should not be indistinguishable downstream.

| Type | Pattern | Check |
|---|---|---|
| `npi` | ten digits, cued by "NPI" or bare (`cue: false`) | Luhn with prefix 80840 |
| `dea_number` | two letters and seven digits, cued by "DEA"; an uncued run is kept only if its check digit works | DEA check digit |
| `bank_account` | digits after "account" / "acct"; a routing number within 120 characters becomes its `issuer` | ABA routing checksum |
| `email` | any address | — |
| `dob` | only a date labelled as a birth date ("DOB", "date of birth", "born") | — |
| `license_plate` | after "plate" / "license plate", with a two-letter state before it if stated; must contain a digit | — |
| `state_license` | after "license" / "medical license" etc., never after "driver's" | — |

Policy and claim numbers are never patterns: they identify a contract or a file, not a
party.


In [ ]:
PATTERNS_VERSION = "1.6"

# A value inside a pattern is the named group `v` when there is one, else group 1, else the
# whole match. `iss` captures an issuer (a plate's state) when the text states it before the
# value. Cues are case-insensitive; the values they capture are not, so "plate was dented"
# never yields a plate "WAS".
_MONTH = r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\.?"
PATTERNS = {
  # cued NPI first: an explicit label is far stronger evidence than ten loose digits
  "npi_cued":   re.compile(r"\b(?:NPI|N\.P\.I\.)[\s#:]*(\d{10})\b", re.I),
  "npi":        re.compile(r"\b\d{10}\b"),
  "phone":      re.compile(r"(?:\(\d{3}\)\s?|\b\d{3}[-.\s])\d{3}[-.\s]?\d{4}\b"),
  "ssn":        re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
  "tin":        re.compile(r"\b\d{2}-\d{7}\b"),
  "vin":        re.compile(r"\b[A-HJ-NPR-Z0-9]{17}\b"),
  "bar_number": re.compile(r"\b(?:bar|ARDC)[\s#:]*(\d{6,8})\b", re.I),
  "address":    re.compile(r"\b\d{1,6}\s+[NSEW]?\.?\s?[A-Z][A-Za-z]+"
                           r"(?:\s+[A-Z][A-Za-z]+)*,?\s+[A-Z][a-z]+\s+[A-Z]{2}\s+\d{5}\b"),
  "email":      re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}\b"),
  # only a date labelled as a birth date: every other date in a note is not an identifier
  "dob":        re.compile(r"(?i:\b(?:DOB|D\.O\.B\.?|date\s+of\s+birth|born(?:\s+on)?))[\s:#,-]*"
                           r"(?P<v>\d{1,2}/\d{1,2}/\d{2,4}|\d{1,2}-\d{1,2}-\d{2,4}|\d{4}-\d{2}-\d{2}"
                           rf"|{_MONTH} \d{{1,2}},? \d{{4}}|\d{{1,2}} {_MONTH} \d{{4}})"),
  "license_plate": re.compile(r"(?:\b(?P<iss>[A-Z]{2})\s+)?(?i:\b(?:license\s+)?plate(?:\s+(?:number|no\.?|#))?)"
                              r"[\s#:]*(?P<v>(?=[A-Z0-9 -]{0,9}\d)[A-Z0-9]{1,4}[ -]?[A-Z0-9]{1,4})\b"),
  "dea_number_cued": re.compile(r"(?i:\bDEA(?:\s+(?:registration|reg\.?))?(?:\s+(?:number|no\.?|#))?)"
                                r"[\s#:]*(?P<v>[A-Z][A-Z9]\d{7})\b"),
  "dea_number": re.compile(r"\b[ABCDEFGHJKLMPRSTUX][A-Z9]\d{7}\b"),
  "state_license": re.compile(r"(?<!driver's )(?<!drivers )(?<!driver’s )"
                              r"(?i:\b(?:(?:medical|professional|nursing|state|chiropractic|dental|"
                              r"physician|attorney)\s+)?licen[cs]e(?:\s+(?:number|no\.?|#))?)[\s#:]*"
                              r"(?P<v>(?=[A-Z0-9-]*\d)[A-Z]{0,3}-?\d{4,10}[A-Z]?)\b"),
  "aba_routing": re.compile(r"(?i:\b(?:routing|ABA|RTN)(?:\s+(?:transit\s+)?(?:number|no\.?|#))?)"
                            r"[\s#:]*(?P<v>\d{9})\b"),
  "bank_account": re.compile(r"(?i:\b(?:bank\s+)?(?:account|acct\.?)(?:\s+(?:number|no\.?|#))?)"
                             r"[\s#:]*(?P<v>\d[\d -]{2,18}\d)\b"),
}

# pattern name -> the detail_type it produces (several patterns can feed one type). A routing
# number is not a party's identifier by itself: it qualifies the account it is stated with.
PATTERN_TYPE = {k: ("npi" if k.startswith("npi") else "dea_number" if k.startswith("dea")
                    else None if k == "aba_routing" else k) for k in PATTERNS}

def luhn_npi(v):
    """NPI check digit: prefix 80840, Luhn over the result."""
    if not (v.isdigit() and len(v) == 10):
        return "malformed"
    digits = [int(c) for c in ("80840" + v)]
    total, parity = 0, len(digits) % 2
    for i, d in enumerate(digits):
        if i % 2 == parity:
            d *= 2
            if d > 9: d -= 9
        total += d
    return "pass" if total % 10 == 0 else "FAIL_LUHN"

def dea_check(v):
    """DEA check digit: (d1+d3+d5) + 2*(d2+d4+d6), last digit equals the 7th digit."""
    v = v.upper()
    if not re.fullmatch(r"[A-Z][A-Z9]\d{7}", v):
        return "malformed"
    d = [int(c) for c in v[2:]]
    return "pass" if (d[0] + d[2] + d[4] + 2 * (d[1] + d[3] + d[5])) % 10 == d[6] else "FAIL_DEA"

def aba_check(v):
    """ABA routing check: 3*(d1+d4+d7) + 7*(d2+d5+d8) + (d3+d6+d9) is a multiple of 10."""
    if not (v.isdigit() and len(v) == 9):
        return "malformed"
    d = [int(c) for c in v]
    s = 3 * (d[0] + d[3] + d[6]) + 7 * (d[1] + d[4] + d[7]) + (d[2] + d[5] + d[8])
    return "pass" if s % 10 == 0 else "FAIL_ABA"

# A find some patterns keep only when it survives a check. An uncued letters-and-digits run
# is a DEA number only if its check digit works: court case numbers look the same otherwise.
PATTERN_KEEP = {
    "dea_number": lambda v: dea_check(v) == "pass",
    "bank_account": lambda v: len(re.sub(r"\D", "", v)) >= 5,
}

# Specific identifier patterns win over generic ones on the same characters.
PRIORITY = ["vin", "email", "ssn", "dea_number_cued", "bar_number", "license_plate",
            "state_license", "aba_routing", "bank_account", "tin", "dea_number",
            "npi_cued", "npi", "dob", "address", "phone"]
ROUTING_REACH = 120        # characters between an account number and the routing number it goes with

def _checksum(dtype, val, issuer=None):
    if dtype == "npi":          return luhn_npi(val)
    if dtype == "dea_number":   return dea_check(val)
    if dtype == "bank_account": return aba_check(issuer) if issuer else "n/a"
    return "n/a"

def pattern_scan(note):
    text, claimed, finds = note["clean_text"], [], []
    for pname in PRIORITY:
        rx = PATTERNS[pname]
        dtype = PATTERN_TYPE[pname]
        for m in rx.finditer(text):
            # When the pattern has a capture group, the SPAN is the group's span too —
            # otherwise a find labelled "bar_number 6224417" carries a span covering
            # "ARDC 6224417" and every citation built from it is off by five characters.
            g = "v" if "v" in rx.groupindex else (1 if rx.groups else 0)
            s, e = m.span(g)
            val = m.group(g)
            if any(s < ce and cs < e for cs, ce in claimed):
                continue                      # lower-priority overlap, suppress
            keep = PATTERN_KEEP.get(pname)
            if keep and not keep(val):
                continue
            claimed.append((s, e))
            iss = m.group("iss") if "iss" in rx.groupindex else None
            finds.append({"lane": "pattern", "pattern": pname, "detail_type": dtype,
                          "raw_value": val, "clean_span": [s, e],
                          "cue": pname.endswith("_cued"), "issuer": iss})
    # a routing number qualifies the nearest account number stated with it
    routings = [f for f in finds if f["pattern"] == "aba_routing"]
    for f in finds:
        if f["detail_type"] == "bank_account" and routings:
            near = min(routings, key=lambda r: abs(r["clean_span"][0] - f["clean_span"][0]))
            if abs(near["clean_span"][0] - f["clean_span"][0]) <= ROUTING_REACH:
                f["issuer"] = near["raw_value"]
    finds = [f for f in finds if f["detail_type"]]
    for f in finds:
        f["checksum"] = _checksum(f["detail_type"], f["raw_value"], f["issuer"])
    return sorted(finds, key=lambda f: f["clean_span"][0])

for n in notes:
    n["lane_pattern"] = pattern_scan(n)

for n in notes:
    print(f"\n{n['note_key']}  ({len(n['lane_pattern'])} finds)")
    for f in n["lane_pattern"]:
        chk = "" if f["checksum"] == "n/a" else f"  checksum={f['checksum']}"
        cue = "" if not f["cue"] else "  cued"
        iss = f"  issuer={f['issuer']}" if f["issuer"] else ""
        print(f"  {f['detail_type']:<13} {f['raw_value']!r:<36} "
              f"span={f['clean_span']}{chk}{cue}{iss}")

fails = [f for n in notes for f in n["lane_pattern"] if f["checksum"].startswith("FAIL")]
print(f"\nchecksum failures: {len(fails)}  <- data-quality signal, not extraction error")
uncued = [f for n in notes for f in n["lane_pattern"]
          if f["detail_type"] == "npi" and not f["cue"]]
print(f"uncued 10-digit runs read as npi: {len(uncued)}  <- weaker claim, see `cue` flag")


## 9 — Lane B: GLiNER (optional)

Zero-shot span detection for unpatterned names. Emits character spans natively, so these
finds skip quote resolution entirely. Gate this on measured LLM entity recall — if recall
against gold data is already adequate, the lane is infrastructure for nothing.

Switched on and off by `GLINER_ENABLED` in cell 1. When enabled, its finds reach
reconciliation (cell 13) and the mention pool.

GLiNER reads a few hundred words per call and **silently truncates** anything longer. The scan
below therefore walks each note in overlapping windows cut at whitespace, shifts every span
back to note coordinates, and keeps the best-scoring copy of any span found twice in an
overlap. Without the windows, a 100-page filing would be judged on its first page.


In [ ]:
GLINER_MODEL     = "urchade/gliner_multi-v2.1"
GLINER_THRESHOLD = 0.60
GLINER_WINDOW_CHARS  = 1000     # inside the model's 384-token limit; 1200 overflowed once
                                # on dense legal text (439 tokens) and lost the tail
GLINER_WINDOW_OVERLAP = 200     # a name cut by one window is whole in the next
GLINER_BATCH     = 8
GLINER_INJECT    = False        # optional injection into the LLM prompt

gliner_model = None
if GLINER_ENABLED:
    try:
        from gliner import GLiNER
        gliner_model = GLiNER.from_pretrained(GLINER_MODEL)
        print(f"GLiNER loaded: {GLINER_MODEL}")
    except Exception as e:
        # Loudly disabled, never silently empty: the summary reports the lane as off.
        print(f"!! GLiNER unavailable ({type(e).__name__}: {e}); lane disabled")
        GLINER_ENABLED = False

def gliner_windows(text, size=GLINER_WINDOW_CHARS, overlap=GLINER_WINDOW_OVERLAP):
    """[(start, end)] windows covering text, cut at whitespace, overlapping."""
    out, start, n = [], 0, len(text)
    while start < n:
        end = min(n, start + size)
        if end < n:
            cut = text.rfind(" ", start + size // 2, end)
            cut = max(cut, text.rfind("\n", start + size // 2, end))
            if cut > start:
                end = cut
        out.append((start, end))
        if end >= n:
            break
        nxt = end - overlap
        sp = text.find(" ", nxt, end)          # start the next window on a word boundary
        start = sp + 1 if sp != -1 else nxt
    return out

def gliner_scan(note):
    if not GLINER_ENABLED or gliner_model is None:
        return []
    text = note["clean_text"]
    wins = gliner_windows(text)
    best = {}
    # `inference` replaced `batch_predict_entities` in recent gliner releases.
    batch_fn = (getattr(gliner_model, "inference", None)
                or getattr(gliner_model, "batch_predict_entities", None))
    for b in range(0, len(wins), GLINER_BATCH):
        chunk = wins[b:b + GLINER_BATCH]
        texts = [text[s:e] for s, e in chunk]
        if batch_fn:
            results = batch_fn(texts, ENTITY_TYPES, threshold=0.0)
        else:
            results = [gliner_model.predict_entities(t, ENTITY_TYPES, threshold=0.0)
                       for t in texts]
        for (s, _), ents in zip(chunk, results):
            for e in ents:
                key = (s + e["start"], s + e["end"], e["label"])
                if key not in best or e["score"] > best[key]["score"]:
                    best[key] = e
    out = []
    for (a, z, label), e in sorted(best.items()):
        score = float(e["score"])
        out.append({"lane": "gliner", "label": label, "text": text[a:z],
                    "clean_span": [a, z], "score": round(score, 3),
                    "kept": score >= GLINER_THRESHOLD})
    note["gliner_windows"] = len(wins)
    return out

import time as _time
_t0 = _time.time()
for n in notes:
    n["lane_gliner"] = gliner_scan(n)
if GLINER_ENABLED:
    print(f"GLiNER scanned {sum(n.get('gliner_windows', 0) for n in notes)} windows "
          f"in {_time.time() - _t0:.1f}s")

if GLINER_ENABLED:
    for n in notes:
        kept = [f for f in n["lane_gliner"] if f["kept"]]
        drop = [f for f in n["lane_gliner"] if not f["kept"]]
        print(f"\n{n['note_key']}  kept={len(kept)} discarded={len(drop)}")
        for f in kept[:15]:
            print(f"  KEEP {f['label']:<13} {f['text']!r:<32} "
                  f"span={f['clean_span']} score={f['score']}")
        if len(kept) > 15:
            print(f"  ... {len(kept) - 15} more kept")
        for f in drop[:5]:
            print(f"  drop {f['label']:<13} {f['text']!r:<32} score={f['score']}")
    print("\ndiscard volume by label is how you tune the threshold against gold data")
else:
    print("Lane B disabled. Reconciliation will show detected_by without 'gliner'.")
    print("Injection path (GLINER_INJECT) destroys per-lane attribution — measure first.")


## 9b — Splitting long notes (optional)

Switched by `CHUNKING` in cell 1. With it off, every note goes to the model whole, exactly as
before; a note too long for that is recorded as a failed extraction, never a crash.

With it on, any note longer than `CHUNK_MAX_CHARS` is split so the model extracts from each
part rather than summarising the whole. On the court corpus, the 93-page complaint sent whole
came back with 3 details.

How a note is split, in order of preference — structure first, arithmetic last:

1. **Structural units.** Page breaks and blank-line paragraph breaks. A dated log entry or a
   numbered pleading paragraph is a unit of meaning; cutting inside one is the worst case.
2. **Sentences**, only inside a unit that is itself too long.
3. **A hard cut at whitespace**, only for a "sentence" longer than a whole chunk (tables,
   run-on OCR).

Units are packed greedily up to the limit. Each chunk after the first re-reads the last
`CHUNK_OVERLAP_SENTENCES` whole sentences of the previous one, so a pronoun near a boundary
still has its antecedent. Anything extracted twice from an overlap is de-duplicated in cell 11.
Spans are always reported in note coordinates.


In [ ]:
_UNIT_BREAK = re.compile(r"\f|\n[ \t]*\n")
# Two fixed-width lookbehinds: Python's re rejects a variable-width one.
_SENT_END = re.compile(r"(?:(?<=[.!?])|(?<=[.!?][\"”’)]))\s+(?=[A-Z0-9“\"(])")

def _sentences(text, s, e):
    """[(start, end)] sentence spans inside text[s:e], in note coordinates. Each sentence
    keeps its trailing whitespace, so the spans tile text[s:e] with no gaps."""
    out, cur = [], s
    for m in _SENT_END.finditer(text, s, e):
        if m.end() > cur:
            out.append((cur, m.end()))
        cur = m.end()
    if cur < e:
        out.append((cur, e))
    return out

def _hard_cut(text, s, e, limit):
    out = []
    while e - s > limit:
        cut = text.rfind(" ", s + limit // 2, s + limit)
        cut = cut if cut > s else s + limit
        out.append((s, cut)); s = cut
    out.append((s, e))
    return out

def split_note(text, limit=CHUNK_MAX_CHARS, overlap_sents=CHUNK_OVERLAP_SENTENCES):
    """[{index, start, end, core_start}] covering text. core_start is where the chunk's own
    material begins; before it is overlap re-read from the previous chunk."""
    n = len(text)
    if not CHUNKING or n <= limit:
        return [{"index": 0, "start": 0, "end": n, "core_start": 0}]
    # 1. structural units
    units, cur = [], 0
    for m in _UNIT_BREAK.finditer(text):
        if m.end() > cur:
            units.append((cur, m.end())); cur = m.end()
    if cur < n:
        units.append((cur, n))
    # 2-3. break oversized units into sentences, and oversized sentences at whitespace
    pieces = []
    for s, e in units:
        if e - s <= limit:
            pieces.append((s, e)); continue
        for ss, se in _sentences(text, s, e):
            pieces.extend(_hard_cut(text, ss, se, limit) if se - ss > limit else [(ss, se)])
    # greedy packing
    cores, s0, e0 = [], None, None
    for s, e in pieces:
        if s0 is None:
            s0, e0 = s, e
        elif e - s0 <= limit:
            e0 = e
        else:
            cores.append((s0, e0)); s0, e0 = s, e
    cores.append((s0, e0))
    # overlap: re-read the last whole sentences of the previous chunk
    chunks = []
    for i, (s, e) in enumerate(cores):
        start = s
        if i and overlap_sents:
            prev = _sentences(text, cores[i - 1][0], cores[i - 1][1])
            if prev:
                start = prev[-min(overlap_sents, len(prev))][0]
        chunks.append({"index": i, "start": start, "end": e, "core_start": s})
    return chunks

for n in notes:
    n["chunks"] = split_note(n["clean_text"])
    n["chunk_policy"] = ("whole_note" if not CHUNKING else
                         f"structure-v1/{CHUNK_MAX_CHARS}c/{CHUNK_OVERLAP_SENTENCES}s")
    if len(n["chunks"]) > 1:
        sizes = [c["end"] - c["start"] for c in n["chunks"]]
        print(f"{n['note_key']:<14} {len(n['clean_text']):>7} chars -> {len(sizes)} chunks "
              f"(largest {max(sizes)}, smallest {min(sizes)})")
split = [n for n in notes if len(n["chunks"]) > 1]
print(f"\nchunking {'ON' if CHUNKING else 'OFF'}: {len(split)} of {len(notes)} notes split, "
      f"{sum(len(n['chunks']) for n in notes)} extraction calls")


## 10 — Lane C: LLM extraction

One call per chunk from cell 9b — one call per note when chunking is off or the note is
short. Calls run `EXTRACT_CONCURRENCY` at a time. The model emits verbatim quotes, never
offsets — it is good at copying text and bad at counting characters, so spans are computed by
search in the next cell, inside the chunk the quote came from.

When a note is split, mention ids are prefixed with their chunk (`c2.m1`) and every reference
(`owner_ref`, action participants) is rewritten to match, so ids stay unique within the note.

A failed extraction is recorded as a failure, not as an empty note. The difference matters:
a 400 on the schema, a truncated response, or unparseable JSON all produce zero mentions,
and without the distinction the run summary reports a clean pipeline over an empty corpus.


In [ ]:
SYSTEM_PROMPT = """You extract structured entity intelligence from insurance claim notes.

Rules that matter more than completeness:
- Quotes are copied, never written. Every `quote` is ONE short contiguous stretch of the
  note (one clause, ideally under 120 characters), copied character for character:
  keep odd spacing, line breaks, hyphenation, typos and OCR artifacts exactly as they
  appear ("busi ness", "Com-\npany", "x -ray").
- Never skip a word, insert a word, change an article, reorder or substitute names, or
  join text from two places. If the exact words you want are not contiguous in the note,
  quote a shorter stretch that is.
- A quote must appear only ONCE in the note. Where the note repeats a sentence (pleadings
  answer paragraph after paragraph with the same words), include distinctive neighbouring
  words such as the paragraph number: "599. Upon information and belief, Nexray is".
- Never emit character offsets or positions. Only quotes.
- For each party, `quote` locates it (padded with context for uniqueness) and `name` is the
  name alone, exactly as written inside that quote. Never pad `name`.
- One entity_mention per PARTY, not per occurrence. Put every other phrase referring to
  that same party (pronouns, role references) into `occurrences`.
- owner_ref is UNASSIGNED unless the text states or clearly implies ownership.
  Physical proximity in the text is NOT ownership.
- basis is "stated" when the text says it, "inferred" when you concluded it.
- stance describes how the text PRESENTS the event, not whether you believe it.
- action_type uses the text's own vocabulary, lowercase_with_underscores.
- Identifiers: email addresses, dates of birth (dob; only a date stated as someone's
  birth date), license plates, DEA numbers, professional license numbers
  (state_license) and bank account numbers are details too. Put the issuing state of a
  plate or license, or the routing number of a bank account, in `issuer`. Policy
  numbers and claim numbers are NOT details: omit them.
- Omit anything that does not fit the allowed types. Do not stretch a type to fit."""

EMPTY_EXTRACTION = {"entity_mentions": [], "detail_mentions": [], "action_mentions": []}

def build_messages(note, chunk):
    text = note["clean_text"][chunk["start"]:chunk["end"]]
    part = ""
    if len(note["chunks"]) > 1:
        part = (f"This is part {chunk['index'] + 1} of {len(note['chunks'])} of a longer "
                f"note. Extract only what this part contains; quote only from this part.\n")
    user = (f"Extract from this claim note.\n{part}"
            f"<note_id>{note['note_id']}</note_id>\n\n<note>\n{text}\n</note>")
    if GLINER_ENABLED and GLINER_INJECT:
        cands = [f["text"] for f in note["lane_gliner"] if f["kept"]
                 and chunk["start"] <= f["clean_span"][0] < chunk["end"]]
        if cands:
            user += ("\n\nCandidate entity spans detected by a separate model "
                     f"(verify each, do not trust blindly): {cands}")
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user}]

def extract_chunk(note, chunk):
    msgs = build_messages(note, chunk)
    kwargs = dict(model=DEPLOYMENT, messages=msgs, temperature=TEMPERATURE,
                  max_tokens=MAX_TOKENS)
    if USE_STRUCTURED:
        kwargs["response_format"] = {
            "type": "json_schema",
            "json_schema": {"name": "note_extraction_v0_1",
                            "strict": True, "schema": EXTRACTION_SCHEMA}}
    else:
        kwargs["response_format"] = {"type": "json_object"}
        kwargs["messages"][0]["content"] += (
            "\n\nReturn ONLY JSON matching this schema:\n"
            + json.dumps(EXTRACTION_SCHEMA))
    r = client.chat.completions.create(**kwargs)
    choice = r.choices[0]
    if getattr(choice, "finish_reason", None) == "length":
        raise RuntimeError(
            f"response truncated at max_tokens={MAX_TOKENS}; the JSON is incomplete. "
            f"Raise MAX_TOKENS or split the note — do not treat this as an empty note.")
    payload = json.loads(choice.message.content)
    missing = [k for k in EMPTY_EXTRACTION if k not in payload]
    if missing:
        raise RuntimeError(f"extraction missing required keys: {missing}")
    return payload, r.usage

def namespace(payload, chunk, multi):
    """Tag every mention with its chunk window. When a note is split, prefix ids with the
    chunk ("c2.m1") and remap every reference to match, so ids stay unique per note."""
    p = "" if not multi else f"c{chunk['index']}."
    ren = lambda x: x if x == "UNASSIGNED" or not p else p + x
    win = [chunk["start"], chunk["end"]]
    for e in payload["entity_mentions"]:
        e["mention_id"] = ren(e["mention_id"]); e["_chunk"] = win
    for d in payload["detail_mentions"]:
        d["detail_id"] = ren(d["detail_id"]); d["owner_ref"] = ren(d["owner_ref"])
        d["_chunk"] = win
    for a in payload["action_mentions"]:
        a["action_id"] = ren(a["action_id"]); a["_chunk"] = win
        for q in a.get("participants", []):
            q["mention_id"] = ren(q["mention_id"])
    return payload

def _run(job):
    note, chunk = job
    try:
        payload, usage = extract_chunk(note, chunk)
        return note, chunk, payload, usage, None
    except Exception as e:
        return note, chunk, None, None, f"{type(e).__name__}: {e}"

from concurrent.futures import ThreadPoolExecutor
jobs = [(n, c) for n in notes for c in n["chunks"]]
with ThreadPoolExecutor(max_workers=max(1, EXTRACT_CONCURRENCY)) as pool:
    results = list(pool.map(_run, jobs))           # map keeps input order

by_note = {}
for note, chunk, payload, usage, err in results:
    by_note.setdefault(note["note_key"], []).append((chunk, payload, usage, err))

for n in notes:
    parts = by_note[n["note_key"]]
    multi = len(parts) > 1
    merged = {k: [] for k in EMPTY_EXTRACTION}
    errors, p_tok, c_tok = [], 0, 0
    for chunk, payload, usage, err in parts:
        if err:
            errors.append(f"chunk {chunk['index']}: {err}"); continue
        for k, v in namespace(payload, chunk, multi).items():
            if k in merged:
                merged[k].extend(v)
        p_tok += usage.prompt_tokens; c_tok += usage.completion_tokens
    n["lane_llm"] = merged
    # A note with any failed chunk is PARTIAL: what succeeded is kept, the gap is counted.
    n["extraction_error"] = "; ".join(errors) or None
    print(f"\n--- {n['note_key']} ({len(n['clean_text'])} chars, "
          f"{len(parts)} chunk{'s' if multi else ''}) ---")
    print(f"  usage: prompt={p_tok} completion={c_tok}")
    print(f"  entities={len(merged['entity_mentions'])} "
          f"details={len(merged['detail_mentions'])} "
          f"actions={len(merged['action_mentions'])}")
    if errors:
        print(f"  !! extraction FAILED for {len(errors)} of {len(parts)} chunk(s): "
              f"{n['extraction_error']}")
    elif not multi:
        print(json.dumps(merged, indent=2)[:1200])

extraction_failures = [n for n in notes if n["extraction_error"]]
print(f"\nextraction failures: {len(extraction_failures)} / {len(notes)}")
if extraction_failures:
    print("!! These notes contributed nothing. Anything downstream is a partial run and")
    print("!! must not be read as a measurement. Failing notes:")
    for n in extraction_failures:
        print(f"   {n['note_key']}: {n['extraction_error']}")


## 11 — Quote resolution

Exact substring search, then normalized retry, then fuzzy. A multi-hit quote disambiguates
against the mention it claims a relationship to — and refuses to guess when no candidate is
clearly nearest.

Both retry paths return spans in the *cleaned text's* coordinates. The naive normalized
retry returns `normalized_text.index(...)`, which is an offset into a different string: for
any note where normalization changed a length before the match, the span silently points
somewhere else. The fuzzy path has the same problem in a different form — a window start is
not a match start. Both are aligned back to real coordinates here.


In [ ]:
from difflib import SequenceMatcher

FUZZY_MIN = 0.92
PROXIMITY_RATIO = 3.0

# PDF text breaks words across lines two ways: a real hyphen ("Com-\npany") and a soft
# hyphen U+00AD ("Com\xad\npany"). A model copies either as "Com-\npany", "Company" or
# "Com- pany". Both sides of the normalized pass drop them, so all of those meet.
_HYPHEN_BREAK = re.compile(r"\xad[ \t]*(?:\r\n|\r|\n)?[ \t]*|-[ \t]*(?:\r\n|\r|\n)[ \t]*")

def _norm(s):
    return re.sub(r"\s+", " ", _HYPHEN_BREAK.sub("", s)).strip().lower()

def _norm_with_map(text):
    """_norm(text), plus norm_idx -> original_idx so a hit can be mapped back."""
    out, idx, i, n = [], [], 0, len(text)
    skip = {}                             # start of a hyphenated break -> its end
    for m in _HYPHEN_BREAK.finditer(text):
        skip[m.start()] = m.end()
    while i < n and text[i].isspace():
        i += 1
    while i < n:
        if i in skip:
            i = skip[i]
        elif text[i].isspace():
            j = i
            while j < n and text[j].isspace():
                j += 1
            if j < n:                     # trailing whitespace is dropped, like .strip()
                out.append(" "); idx.append(i)
            i = j
        else:
            out.append(text[i].lower()); idx.append(i); i += 1
    return "".join(out), idx

# The guarded snap. Letters and digits only, lowercased: "Rutland , in turn" meets
# "Rutland, in turn", "busi ness" meets "business", "x -ray" meets "x-ray". Two strings with
# the same loose form differ ONLY in whitespace, punctuation, hyphenation or case, so no
# name, word, negation or number can differ between them -- except that dropping
# punctuation can fuse digit groups ("3/14" and "31/4" are both "314"), so the digit runs
# must also agree (thousands separators aside).
def _loose_with_map(text):
    out, idx = [], []
    t = _HYPHEN_BREAK.sub(lambda m: "\0" * len(m.group(0)), text)
    for i, ch in enumerate(t):
        if ch.isalnum():
            out.append(ch.lower()); idx.append(i)
    return "".join(out), idx

def _loose(s):
    return _loose_with_map(s)[0]

def _digit_runs(s):
    return re.findall(r"\d+", re.sub(r"(?<=\d),(?=\d{3}\b)", "", _HYPHEN_BREAK.sub("", s)))

def loose_equal(a, b):
    return _loose(a) == _loose(b) and _digit_runs(a) == _digit_runs(b)

# A fuzzy match may absorb small copying slips ("mornings" for "morning"), never a change in
# who, how many, when, or whether: a differing capitalised word, number or negation word
# rejects it. A swapped name is an attribution error to surface, not a typo to fix.
NEGATIONS = {"not", "no", "never", "nor", "none", "neither", "without", "deny", "denies",
             "denied", "cannot", "n't"}

def _guard_diff(quote, found):
    """None when the only word-level differences are harmless; else why the match is refused."""
    qa, fa = re.findall(r"\w+|n't", quote), re.findall(r"\w+|n't", found)
    sm = SequenceMatcher(None, [w.lower() for w in qa], [w.lower() for w in fa], autojunk=False)
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        if op == "equal":
            continue
        for w in qa[i1:i2] + fa[j1:j2]:
            if any(c.isdigit() for c in w):
                return "number_differs"
            if w.lower() in NEGATIONS:
                return "negation_differs"
            if w[:1].isupper():
                return "name_differs"
    return None

def _fuzzy_locate(quote, text):
    """Best approximate span for quote in text, aligned to the match, not to a window."""
    w = len(quote)
    if w == 0 or len(text) == 0:
        return None, 0.0
    step = max(1, w // 4)
    best_r, best_i = 0.0, None
    for i in range(0, max(1, len(text) - w + 1), step):
        r = SequenceMatcher(None, quote, text[i:i + w]).ratio()
        if r > best_r:
            best_r, best_i = r, i
    if best_i is None:
        return None, 0.0
    lo, hi = max(0, best_i - w), min(len(text), best_i + 2 * w)
    window = text[lo:hi]
    blocks = [b for b in SequenceMatcher(None, window, quote).get_matching_blocks()
              if b.size > 0]
    if not blocks:
        return None, best_r
    s = lo + blocks[0].a
    e = lo + blocks[-1].a + blocks[-1].size
    return [s, e], SequenceMatcher(None, quote, text[s:e]).ratio()

def _pick(hits, anchors, method):
    """Several places hold the same text. hits: [(start, end)] in text coordinates.

    With an anchor (the owner's span for a detail, the participants' spans for an action),
    the hit nearest any anchor wins when it is clearly nearest. Otherwise the text is still
    verified -- every hit is the same words -- and only its position is uncertain: keep the
    nearest (or first) hit and say so, with the count, rather than dropping a true quote."""
    if len(hits) == 1:
        return list(hits[0]), method
    if anchors:
        d = sorted((min(abs(h[0] - a) for a in anchors), h) for h in hits)
        if d[0][0] == 0 or d[1][0] / max(d[0][0], 1) >= PROXIMITY_RATIO:
            return list(d[0][1]), "proximity" if method == "exact" else f"{method}:proximity"
        best = d[0][1]
    else:
        best = hits[0]
    return list(best), f"{method}:multi_hit_identical:{len(hits)}"

def resolve_quote(quote, text, anchor_pos=None):
    """-> (span_in_clean_text, method) or (None, reason).

    Passes, in order: exact; normalized (whitespace, case, hyphenated line breaks);
    loose (the guarded snap: only whitespace, punctuation, hyphenation or case may differ);
    fuzzy >= FUZZY_MIN, refused when a name, number or negation differs. A method may end in
    ':multi_hit_identical:N' -- the text is verified, its position among N copies is not."""
    if not quote:
        return None, "empty_quote"
    anchors = ([] if anchor_pos is None else
               [anchor_pos] if isinstance(anchor_pos, int) else [a for a in anchor_pos if a is not None])
    hits = [(m.start(), m.start() + len(quote)) for m in re.finditer(re.escape(quote), text)]
    if hits:
        return _pick(hits, anchors, "exact")

    nq = _norm(quote)
    if nq:
        nt, nmap = _norm_with_map(text)
        nh = [(nmap[m.start()], nmap[m.start() + len(nq) - 1] + 1)   # map the LAST char
              for m in re.finditer(re.escape(nq), nt)]
        if nh:
            return _pick(nh, anchors, "normalized")

    lq = _loose(quote)
    if len(lq) >= 8:                      # too little left to place safely otherwise
        lt, lmap = _loose_with_map(text)
        lh = [(lmap[m.start()], lmap[m.start() + len(lq) - 1] + 1)
              for m in re.finditer(re.escape(lq), lt)]
        lh = [h for h in lh if _digit_runs(text[h[0]:h[1]]) == _digit_runs(quote)]
        if lh:
            return _pick(lh, anchors, "loose")

    span, ratio = _fuzzy_locate(quote, text)
    if span and ratio >= FUZZY_MIN:
        why = _guard_diff(quote, text[span[0]:span[1]])
        if why:
            return None, f"unverifiable_quote:{ratio:.2f}:{why}"
        return span, f"fuzzy:{ratio:.2f}"
    return None, f"unverifiable_quote:{ratio:.2f}"

def quote_matches(method, quote, found):
    """The round-trip test for a span, by the pass that placed it (cell 12 and cell 23)."""
    base = method.split(":")[0]
    if base == "fuzzy":
        return SequenceMatcher(None, _norm(quote), _norm(found)).ratio() >= FUZZY_MIN
    if base in ("loose", "name_fallback"):
        return loose_equal(quote, found)
    return _norm(found) == _norm(quote)

def resolve_note(note):
    text = note["clean_text"]
    resolved, methods, quotes, failed, notices = {}, {}, {}, [], []

    def record(mid, kind, quote, anchor, win=None, quiet_fail=False):
        # Search only the chunk the quote came from: a phrase repeated elsewhere in a long
        # note is not a candidate, and every span is shifted back to note coordinates.
        w0, w1 = win or (0, len(text))
        anc = None if anchor is None else [a - w0 for a in anchor]
        span, how = resolve_quote(quote, text[w0:w1], anc)
        if span:
            span = [span[0] + w0, span[1] + w0]
        quotes[mid] = quote
        if span:
            resolved[mid] = span
            methods[mid] = how
            if "multi_hit_identical" in how:
                notices.append({"id": mid, "kind": kind, "flag": "multi_hit_identical",
                                "count": int(how.rsplit(":", 1)[1]), "quote": quote})
        elif not quiet_fail:
            failed.append({"id": mid, "kind": kind, "quote": quote, "reason": how})
        print(f"  {mid:<5} {kind:<7} {quote[:45]!r:<50} -> {span} {how}")
        return span, how

    for e in note["lane_llm"]["entity_mentions"]:
        # Locate the padded quote, then narrow the span to the bare name inside it:
        # the citation should cover the party, not the words around it.
        name, mid = e.get("name"), e["mention_id"]
        span, how = record(mid, "entity", e["quote"], None, e.get("_chunk"),
                           quiet_fail=bool(name))
        if not span and name:
            # The quote was edited, but the name alone is verifiable text: place the name
            # (first copy in the chunk, flagged when it repeats). Otherwise it is a failure.
            nspan, nhow = record(mid, "entity", name, None, e.get("_chunk"), quiet_fail=True)
            if nspan:
                methods[mid] = "name_fallback" + (nhow[nhow.index(":"):] if ":" in nhow and
                                                  "multi_hit" in nhow else "")
                notices.append({"id": mid, "kind": "entity", "flag": "quote_edited_name_placed",
                                "quote": e["quote"], "reason": how.split(":")[0]})
            else:
                failed.append({"id": mid, "kind": "entity", "quote": e["quote"], "reason": how})
            continue
        if name and mid in resolved:
            s0, e0 = resolved[mid]
            window = text[s0:e0]
            k = window.find(name)
            if k >= 0:
                resolved[mid] = [s0 + k, s0 + k + len(name)]
                quotes[mid] = name
            else:
                # Source text breaks lines and doubles spaces inside names
                # ("WILLIAM A. WEINER, \nD.O."); the model's name does not.
                nw, nmap = _norm_with_map(window)
                nn = _norm(name)
                j = nw.find(nn) if nn else -1
                if j >= 0:
                    resolved[mid] = [s0 + nmap[j], s0 + nmap[j + len(nn) - 1] + 1]
                    quotes[mid] = name
                else:
                    print(f"        !! name {name!r} not inside its quote; keeping quote span")
    for d in note["lane_llm"]["detail_mentions"]:
        owner = resolved.get(d.get("owner_ref"))
        record(d["detail_id"], "detail", d["quote"], [owner[0]] if owner else None, d.get("_chunk"))
    for a in note["lane_llm"]["action_mentions"]:
        pids = [p["mention_id"] for p in a.get("participants", [])]
        anchors = [resolved[p][0] for p in pids if p in resolved]
        record(a["action_id"], "action", a["quote"], anchors or None, a.get("_chunk"))
    note["quote_notices"] = notices
    return resolved, methods, quotes, failed

def dedupe_overlap(note):
    """Drop mentions extracted twice from a chunk overlap, keeping the earlier chunk's copy,
    and point every reference at the survivor. Two mentions are the same extraction when
    they are the same kind, from different chunks, and resolve to overlapping spans with the
    same content (entity name / detail value / action type)."""
    if len(note.get("chunks", [])) < 2:
        return 0
    spans, L = note["spans"], note["lane_llm"]
    def chunk_of(x):
        return x.get("_chunk", [0, 0])[0]
    def overlaps_(a, b):
        return a and b and a[0] < b[1] and b[0] < a[1]
    alias = {}
    groups = [("entity_mentions", "mention_id", lambda m: _norm(m.get("name") or m["quote"])),
              ("detail_mentions", "detail_id", lambda d: (d["detail_type"], _norm(d["raw_value"]))),
              ("action_mentions", "action_id", lambda a: a["action_type"])]
    removed = 0
    for key, idf, sig in groups:
        kept = []
        for m in sorted(L[key], key=chunk_of):
            dup = next((k for k in kept
                        if chunk_of(k) != chunk_of(m) and sig(k) == sig(m)
                        and overlaps_(spans.get(k[idf]), spans.get(m[idf]))), None)
            if dup is not None:
                alias[m[idf]] = dup[idf]
                spans.pop(m[idf], None); removed += 1
            else:
                kept.append(m)
        L[key] = kept
    for d in L["detail_mentions"]:
        d["owner_ref"] = alias.get(d["owner_ref"], d["owner_ref"])
    for a in L["action_mentions"]:
        for q in a.get("participants", []):
            q["mention_id"] = alias.get(q["mention_id"], q["mention_id"])
    return removed

for n in notes:
    print(f"\n--- resolving {n['note_key']} ---")
    n["spans"], n["span_methods"], n["quotes"], n["quote_failures"] = resolve_note(n)
    dup = dedupe_overlap(n)
    print(f"  resolved={len(n['spans'])} failed={len(n['quote_failures'])}"
          + (f" overlap-duplicates removed={dup}" if dup else ""))


## 12 — Round-trip check

Map to raw coordinates, slice the original, compare **against the quote the model emitted**.

The comparison target is the whole point. Slicing raw and comparing it to the clean slice of
the same span re-derives one side from the other: it checks the offset map (already checked
exhaustively in cell 7) and passes any span, including a span pointing at the wrong sentence
entirely. Comparing the raw slice to the quote is what catches a hallucinated quote, a bad
fuzzy alignment, and a boundary bug.

Fuzzy-resolved spans are compared by ratio rather than equality — an approximate match
cannot be exact by definition — and are marked `approximate` so they stay visible.


In [ ]:
def round_trip(note):
    passes, fails = [], []
    for mid, span in note["spans"].items():
        rs  = clean_to_raw(span[0], note["edits"], len(note["clean_text"]))
        re_ = clean_to_raw(span[1], note["edits"], len(note["clean_text"]))
        raw_slice   = note["raw_text"][rs:re_]
        clean_slice = note["clean_text"][span[0]:span[1]]
        quote       = note["quotes"][mid]
        method      = note["span_methods"][mid]

        map_ok = _norm(raw_slice) == _norm(clean_slice)
        # each pass has its own test: fuzzy by ratio, the guarded snap by its loose form
        # (only whitespace, punctuation, hyphenation or case may differ), the rest exact
        # after whitespace and hyphenation normalisation
        quote_ok = quote_matches(method, quote, raw_slice)
        detail   = method.split(":")[0]
        ok = map_ok and quote_ok
        rec = {"id": mid, "clean_span": span, "raw_span": [rs, re_], "method": method,
               "raw_slice": raw_slice[:60], "map_ok": map_ok, "quote_ok": quote_ok,
               "approximate": method.startswith("fuzzy"), "ok": ok}
        (passes if ok else fails).append(rec)
        flag = "PASS" if ok else "FAIL"
        approx = "  (approximate)" if rec["approximate"] else ""
        print(f"  {flag} {mid:<5} clean={span} raw=[{rs}, {re_}]  "
              f"{raw_slice[:42]!r}{approx}")
        if not ok:
            print(f"       map_ok={map_ok} quote_ok={quote_ok} ({detail})")
            print(f"       quote was {quote[:60]!r}")
    return passes, fails

for n in notes:
    print(f"\n--- round-trip {n['note_key']} ---")
    n["rt_pass"], n["rt_fail"] = round_trip(n)
    n["raw_spans"] = {r["id"]: r["raw_span"] for r in n["rt_pass"]}
    # a span that fails the round trip is not a citation, so it is not carried forward
    for r in n["rt_fail"]:
        n["spans"].pop(r["id"], None)
    print(f"  pass={len(n['rt_pass'])} fail={len(n['rt_fail'])}")

total_fail = sum(len(n["rt_fail"]) for n in notes)
print(f"\nround-trip failures across all notes: {total_fail}")
print("Any failure here means a citation in the finished dossier would point at the wrong "
      "text, so the span is dropped and the mention goes to review.")


## 13 — Reconcile lanes

Match on normalized value **and overlapping span**, which is what makes the match a
statement about one occurrence of a value rather than about the value. Value-only matching
pairs the first pattern find with every LLM mention of the same value, so a note that says
the same phone number twice emits one reconciled detail and one phantom "recall gap".

Four outcomes: all lanes, pattern-only (the recall gap this lane exists for), GLiNER-only,
and LLM-only. GLiNER finds are reconciled against entity mentions, not details — the lane
produces spans for names, and names are not details.


In [ ]:
import datetime as _dt

US_STATES = {
    "ALABAMA": "AL", "ALASKA": "AK", "ARIZONA": "AZ", "ARKANSAS": "AR", "CALIFORNIA": "CA",
    "COLORADO": "CO", "CONNECTICUT": "CT", "DELAWARE": "DE", "DISTRICT OF COLUMBIA": "DC",
    "FLORIDA": "FL", "GEORGIA": "GA", "HAWAII": "HI", "IDAHO": "ID", "ILLINOIS": "IL",
    "INDIANA": "IN", "IOWA": "IA", "KANSAS": "KS", "KENTUCKY": "KY", "LOUISIANA": "LA",
    "MAINE": "ME", "MARYLAND": "MD", "MASSACHUSETTS": "MA", "MICHIGAN": "MI", "MINNESOTA": "MN",
    "MISSISSIPPI": "MS", "MISSOURI": "MO", "MONTANA": "MT", "NEBRASKA": "NE", "NEVADA": "NV",
    "NEW HAMPSHIRE": "NH", "NEW JERSEY": "NJ", "NEW MEXICO": "NM", "NEW YORK": "NY",
    "NORTH CAROLINA": "NC", "NORTH DAKOTA": "ND", "OHIO": "OH", "OKLAHOMA": "OK", "OREGON": "OR",
    "PENNSYLVANIA": "PA", "RHODE ISLAND": "RI", "SOUTH CAROLINA": "SC", "SOUTH DAKOTA": "SD",
    "TENNESSEE": "TN", "TEXAS": "TX", "UTAH": "UT", "VERMONT": "VT", "VIRGINIA": "VA",
    "WASHINGTON": "WA", "WEST VIRGINIA": "WV", "WISCONSIN": "WI", "WYOMING": "WY",
    "PUERTO RICO": "PR", "GUAM": "GU", "VIRGIN ISLANDS": "VI"}
STATE_CODES = set(US_STATES.values())

def norm_state(s):
    """'New York', 'N.Y.', 'NY', 'New York State Education Department' -> 'NY'; else ''."""
    if not s:
        return ""
    t = re.sub(r"\s+", " ", re.sub(r"[^A-Z ]", "", s.upper().replace(".", ""))).strip()
    if t in STATE_CODES:
        return t
    if t in US_STATES:
        return US_STATES[t]
    if t.split()[0] in STATE_CODES:         # "NY State Education Department"
        return t.split()[0]
    # the longest state name inside the string ("WEST VIRGINIA" before "VIRGINIA")
    for name in sorted(US_STATES, key=len, reverse=True):
        if re.search(rf"\b{name}\b", t):
            return US_STATES[name]
    return ""

# Types whose value only identifies a party together with its issuer: a plate or a license
# number is unique within a state, an account number within a bank. The normalized value is
# "ISSUER:NUMBER" when the issuer is stated, "NUMBER" when it is not.
QUALIFIED_TYPES = {"license_plate", "state_license", "bank_account"}
# Types that propose candidate pairs and count as shared details across claims. A date of
# birth is shared by one person in ~29,000: evidence when two names already agree, never a
# reason on its own to compare two parties.
JOIN_TYPES = [t for t in DETAIL_TYPES if t != "dob"]

def split_qualified(v):
    """'NY:ABC1234' -> ('NY', 'ABC1234'); 'ABC1234' -> ('', 'ABC1234')."""
    return tuple(v.split(":", 1)) if ":" in v else ("", v)

def id_number(dtype, v):
    """The part of a normalized value that must agree for two values to be one identifier."""
    return split_qualified(v)[1] if dtype in QUALIFIED_TYPES and v else v

def qualifiers_agree(dtype, v1, v2):
    """Same number, and the issuers agree or at least one is unstated."""
    if dtype not in QUALIFIED_TYPES:
        return v1 == v2
    (q1, n1), (q2, n2) = split_qualified(v1), split_qualified(v2)
    return n1 == n2 and (not q1 or not q2 or q1 == q2)

_MONTHS = {m: i for i, m in enumerate(
    ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"], 1)}

def parse_date_iso(v, today=None):
    """A date as written -> 'YYYY-MM-DD', or None. Numeric dates are read month first (US
    notes). A two-digit year pivots on today's: '80' is 1980, '05' is 2005."""
    today = today or _dt.date.today()
    s = v.strip().rstrip(".,")
    y = mo = d = None
    if m := re.fullmatch(r"(\d{4})-(\d{1,2})-(\d{1,2})", s):
        y, mo, d = map(int, m.groups())
    elif m := re.fullmatch(r"(\d{4})(\d{2})(\d{2})", s):
        y, mo, d = map(int, m.groups())
    elif m := re.fullmatch(r"(\d{1,2})[/-](\d{1,2})[/-](\d{2}|\d{4})", s):
        mo, d, y = map(int, m.groups())
        if len(m.group(3)) == 2:
            y += 2000 if y <= today.year % 100 else 1900
    elif m := re.fullmatch(r"([A-Za-z]+)\.? (\d{1,2}),? (\d{4})", s):
        mo, d, y = _MONTHS.get(m.group(1)[:3].lower()), int(m.group(2)), int(m.group(3))
    elif m := re.fullmatch(r"(\d{1,2}) ([A-Za-z]+)\.? (\d{4})", s):
        d, mo, y = int(m.group(1)), _MONTHS.get(m.group(2)[:3].lower()), int(m.group(3))
    if not (y and mo and d):
        return None
    try:
        date = _dt.date(y, mo, d)
    except ValueError:
        return None
    return date.isoformat() if date <= today else None

EMAIL_RE = re.compile(r"[a-z0-9._%+-]+@[a-z0-9-]+(?:\.[a-z0-9-]+)*\.[a-z]{2,}")

def normalize_detail(dtype, val, issuer=None):
    v = val.strip()
    if dtype == "phone":
        d = re.sub(r"\D", "", v)
        if len(d) == 10: return "+1" + d
        if len(d) == 11 and d.startswith("1"): return "+" + d
        return None
    if dtype in ("npi", "ssn", "tin"):
        d = re.sub(r"\D", "", v)
        return d or None
    if dtype == "bar_number":
        # Alphanumeric in many jurisdictions (E.D.N.Y. bar codes look like MW7455).
        # Keeping only digits turns MW7455 and XY7455 into the same join key.
        d = re.sub(r"[^0-9A-Za-z]", "", v).upper()
        return d or None
    if dtype == "vin":
        return v.upper()
    if dtype == "address":
        return re.sub(r"\s+", " ", re.sub(r"[.,]", "", v)).lower().replace(" ", "|")
    if dtype == "email":
        e = re.sub(r"^mailto:", "", v.lower()).strip("<>()[].,;: ")
        return e if EMAIL_RE.fullmatch(e) else None
    if dtype == "dob":
        return parse_date_iso(v)
    if dtype == "dea_number":
        d = re.sub(r"[^A-Z0-9]", "", v.upper())
        return d if re.fullmatch(r"[A-Z][A-Z9]\d{7}", d) else None
    if dtype == "license_plate":
        p = re.sub(r"[^A-Z0-9]", "", v.upper())
        if not (2 <= len(p) <= 8):
            return None
        st = norm_state(issuer)
        return f"{st}:{p}" if st else p
    if dtype == "state_license":
        n = re.sub(r"[^A-Z0-9]", "", v.upper())
        if not re.search(r"\d", n):
            return None
        st = norm_state(issuer)
        return f"{st}:{n}" if st else n
    if dtype == "bank_account":
        # The model may put the routing number in raw_value too ("routing 021000021,
        # account 4417 2291 08"): a 9-digit group that passes the ABA check is the routing.
        groups = [re.sub(r"\D", "", g) for g in re.findall(r"\d[\d -]*\d", v)]
        routing = re.sub(r"\D", "", issuer or "")
        if len(groups) > 1:
            r = next((g for g in groups if len(g) == 9 and aba_check(g) == "pass"), None)
            if r:
                routing = routing or r
                groups = [g for g in groups if g != r]
        n = "".join(groups)
        if not 4 <= len(n) <= 17:
            return None
        return f"{routing}:{n}" if len(routing) == 9 else n
    return v.lower()

def checksum_of(dtype, normalized):
    """The check a value of this type carries, computed from the normalized value."""
    if not normalized:
        return "n/a"
    if dtype == "npi":
        return luhn_npi(normalized)
    if dtype == "dea_number":
        return dea_check(normalized)
    if dtype == "bank_account":
        routing = split_qualified(normalized)[0]
        return aba_check(routing) if routing else "n/a"
    return "n/a"

def overlaps(a, b):
    return bool(a) and bool(b) and a[0] < b[1] and b[0] < a[1]

def reconcile(note):
    """LLM details x pattern finds. A pattern find is consumed by at most one mention."""
    out = []
    used_pattern = set()
    pnorm = [normalize_detail(p["detail_type"], p["raw_value"], p.get("issuer"))
             for p in note["lane_pattern"]]
    for d in note["lane_llm"]["detail_mentions"]:
        norm = normalize_detail(d["detail_type"], d["raw_value"], d.get("issuer"))
        span = note["spans"].get(d["detail_id"])
        detected, checksum, cue = ["llm"], "n/a", None
        # Prefer a span-overlapping find; fall back to value-only when the LLM quote could
        # not be resolved (no span to compare) and exactly one find is still unclaimed.
        # Values match on their number: a plate the model read with its state and the
        # pattern read without is one plate.
        cands = [i for i, p in enumerate(note["lane_pattern"])
                 if i not in used_pattern
                 and p["detail_type"] == d["detail_type"] and norm and pnorm[i]
                 and qualifiers_agree(d["detail_type"], pnorm[i], norm)]
        match = next((i for i in cands
                      if overlaps(span, note["lane_pattern"][i]["clean_span"])), None)
        if match is None and span is None and len(cands) == 1:
            match = cands[0]
        if match is not None:
            p = note["lane_pattern"][match]
            used_pattern.add(match)
            detected.append("pattern")
            cue = p["cue"]
            if d["detail_type"] in QUALIFIED_TYPES and not split_qualified(norm)[0]:
                norm = pnorm[match]          # the pattern saw the issuer the model left out
        checksum = checksum_of(d["detail_type"], norm)
        out.append({"detail_type": d["detail_type"], "raw_value": d["raw_value"],
                    "normalized": norm, "clean_span": span,
                    "owner_ref": d["owner_ref"], "basis": d["basis"],
                    "issuer": d.get("issuer"), "detected_by": detected,
                    "checksum": checksum, "cue": cue, "recall_gap": False})
    for i, p in enumerate(note["lane_pattern"]):
        if i in used_pattern:
            continue
        out.append({"detail_type": p["detail_type"], "raw_value": p["raw_value"],
                    "normalized": pnorm[i],
                    "clean_span": p["clean_span"], "owner_ref": "UNASSIGNED",
                    "basis": "stated", "issuer": p.get("issuer"), "detected_by": ["pattern"],
                    "checksum": p["checksum"], "cue": p["cue"], "recall_gap": True})
    return out

def reconcile_entities(note):
    """GLiNER spans against LLM entity mentions. Returns (per-mention lanes, gliner-only)."""
    lanes = {e["mention_id"]: ["llm"] for e in note["lane_llm"]["entity_mentions"]}
    kept = [f for f in note["lane_gliner"] if f["kept"]]
    claimed = set()
    for e in note["lane_llm"]["entity_mentions"]:
        span = note["spans"].get(e["mention_id"])
        for i, f in enumerate(kept):
            if i in claimed:
                continue
            if overlaps(span, f["clean_span"]):
                lanes[e["mention_id"]].append("gliner")
                claimed.add(i)
                break
    gliner_only = [{"entity_candidate": True, "type": f["label"], "text": f["text"],
                    "clean_span": f["clean_span"], "score": f["score"],
                    "detected_by": ["gliner"], "loaded": False}
                   for i, f in enumerate(kept) if i not in claimed]
    return lanes, gliner_only

for n in notes:
    print(f"\n--- reconcile {n['note_key']} ---")
    n["details"] = reconcile(n)
    n["entity_lanes"], n["gliner_only"] = reconcile_entities(n)
    for d in n["details"]:
        tag = "  RECALL_GAP" if d["recall_gap"] else ""
        print(f"  {d['detail_type']:<13} {str(d['normalized'])[:34]:<36} "
              f"owner={d['owner_ref']:<12} by={'+'.join(d['detected_by'])}{tag}")
    for g in n["gliner_only"][:10]:
        print(f"  entity_candidate {g['type']:<13} {g['text']!r} score={g['score']} "
              f"(gliner only, not loaded)")
    if len(n["gliner_only"]) > 10:
        print(f"  ... {len(n['gliner_only']) - 10} more gliner-only candidates")

gaps = [d for n in notes for d in n["details"] if d["recall_gap"]]
print(f"\nrecall gaps (pattern found, LLM missed): {len(gaps)}")
print("These would be lost silently without Lane A. They load as UNASSIGNED —")
print("a pattern match knows the value and nothing about who owns it.")


## 14 — Validation layer

Checks the schema could not express structurally: referential integrity, enum conformance,
format. Separate from the schema, and its failures mean different things.


In [ ]:
def validate_note(note):
    problems = []
    ids = {e["mention_id"] for e in note["lane_llm"]["entity_mentions"]}
    seen_mention_ids = [e["mention_id"] for e in note["lane_llm"]["entity_mentions"]]
    for mid in {m for m in seen_mention_ids if seen_mention_ids.count(m) > 1}:
        problems.append({"kind": "duplicate_mention_id", "value": mid})
    for e in note["lane_llm"]["entity_mentions"]:
        if e["type"] not in ENTITY_TYPES:
            problems.append({"kind": "enum_violation", "field": "entity.type",
                             "value": e["type"]})
    for d in note["details"]:
        if d["owner_ref"] not in ids and d["owner_ref"] != "UNASSIGNED":
            problems.append({"kind": "dangling_owner_ref", "value": d["owner_ref"]})
        if d["detail_type"] not in DETAIL_TYPES:
            problems.append({"kind": "enum_violation", "field": "detail_type",
                             "value": d["detail_type"]})
        if d["basis"] not in BASES:
            problems.append({"kind": "enum_violation", "field": "basis",
                             "value": d["basis"]})
        if d["normalized"] is None:
            problems.append({"kind": "unnormalizable", "value": d["raw_value"],
                             "detail_type": d["detail_type"]})
    for a in note["lane_llm"]["action_mentions"]:
        if not a.get("participants"):
            problems.append({"kind": "action_without_participant",
                             "value": a["action_id"]})
        for p in a.get("participants", []):
            if p["mention_id"] not in ids:
                problems.append({"kind": "dangling_participant",
                                 "action": a["action_id"], "value": p["mention_id"]})
        if a["stance"] not in STANCES:
            problems.append({"kind": "enum_violation", "field": "stance",
                             "value": a["stance"]})
    return problems

for n in notes:
    n["validation"] = validate_note(n)
    print(f"{n['note_key']}: {len(n['validation'])} problem(s)")
    for p in n["validation"]:
        print("   ", json.dumps(p))

print("\nenum_violation under structured output should be impossible.")
print("If it fires, the constraint was not in force — version skew or an")
print("unconstrained fallback path. That is a bug, not a data finding.")


## 15 — note_envelopes

Everything stamped with every version that could change the result.


In [ ]:
envelopes = []
for n in notes:
    env = {
      "note_id": n["note_id"], "note_key": n["note_key"],
      "claim_id": n["claim_id"], "client_id": n["client_id"],
      "occurrence_id": n["occurrence_id"], "coverage_code": n["coverage_code"],
      "versions": {"schema": SCHEMA_VERSION, "model": DEPLOYMENT,
                   "patterns": PATTERNS_VERSION,
                   "gliner": GLINER_MODEL if GLINER_ENABLED else None,
                   "clean_policy": CLEAN_POLICY, "chunk_policy": n["chunk_policy"],
                   "structured": USE_STRUCTURED, "offline": OFFLINE_MODE},
      "extraction_error": n["extraction_error"],
      "entity_mentions": [
        {**e, "clean_span": n["spans"].get(e["mention_id"]),
              "raw_span": n["raw_spans"].get(e["mention_id"]),
              "detected_by": n["entity_lanes"].get(e["mention_id"], ["llm"])}
        for e in n["lane_llm"]["entity_mentions"]],
      "entity_candidates": n["gliner_only"],
      "detail_mentions": [
        {**d, "raw_span": ([clean_to_raw(d["clean_span"][0], n["edits"], len(n["clean_text"])),
                            clean_to_raw(d["clean_span"][1], n["edits"], len(n["clean_text"]))]
                           if d.get("clean_span") else None)}
        for d in n["details"]],
      "action_mentions": [
        {**a, "clean_span": n["spans"].get(a["action_id"]),
              "raw_span": n["raw_spans"].get(a["action_id"])}
        for a in n["lane_llm"]["action_mentions"]],
      "review_items": (
        ([{"flag": "extraction_error", "detail": n["extraction_error"]}]
         if n["extraction_error"] else [])
        + [{"flag": f["reason"].split(":")[0], "id": f["id"], "quote": f["quote"],
            "reason": f["reason"]}
           for f in n["quote_failures"]]
        # placed, but worth a look: the position among identical copies is uncertain, or
        # the quote was edited and only the party's name could be placed
        + [{"flag": x["flag"], "id": x["id"], "quote": x["quote"],
            **({"count": x["count"]} if "count" in x else {})}
           for x in n.get("quote_notices", []) if x["id"] in n["spans"]]
        + [{"flag": "round_trip_fail", "id": r["id"], "method": r["method"]}
           for r in n["rt_fail"]]
        + [{"flag": "checksum_fail", "detail_type": d["detail_type"],
            "raw_value": d["raw_value"]}
           for d in n["details"] if str(d["checksum"]).startswith("FAIL")]
        + [{"flag": p["kind"], **p} for p in n["validation"]]),
    }
    envelopes.append(env)
    Path(OUT_DIR, f"envelope_{n['note_id']}.json").write_text(json.dumps(env, indent=2))

print(f"{len(envelopes)} envelopes written to {OUT_DIR}/")
for e in envelopes:
    print(f"  {e['note_key']:<14} entities={len(e['entity_mentions'])} "
          f"details={len(e['detail_mentions'])} actions={len(e['action_mentions'])} "
          f"review={len(e['review_items'])}")
print("\nsample:")
print(json.dumps(envelopes[0], indent=2)[:2000])


## 16 — Mention records and outside rarity

Identity is decided between **mentions**, never between pre-merged entities. Each entity
mention becomes one record: its fullest name (cell 16's filter over the note's own
occurrences), the name parsed into parts, the details the note says it owns, the roles it
plays in actions, and where it sits (note, claim, occurrence, client).

Rarity comes only from **outside references**, never from the corpus being linked. A corpus
of fraud claims over-represents exactly the names in question, so counting inside it would
call a ring member "common" because the ring is in the data.

| Table | Source | Used for |
|---|---|---|
| `surnames.csv.gz` | Census 2010, 162k surnames | chance two people share a surname |
| `first_names.csv.gz` | SSA births 1930–2005 | chance two people share a first name, or an initial |
| `org_tokens.csv.gz` | CMS NPPES, every organization provider | how many organizations use a name word |

`corpus/reference/fetch_reference.py` rebuilds them. If they are missing the linker still
runs, with a flat rarity that is stamped into every artifact so the run cannot be mistaken
for a real one.

Every mention also gets a **role class**, which sets its starting odds across claims (cell
18). Organizations and professionals (a credential, a "Dr." title, an NPI or bar number, or a
professional role in an action) recur across claims by nature. Private persons do not.


In [ ]:
import csv, gzip, math
from collections import defaultdict
from jellyfish import jaro_winkler_similarity as _jw

REFERENCE_DIR = "./corpus/reference"

PERSON_STOPWORDS = {"dr", "md", "do", "mr", "mrs", "ms", "esq", "jr", "sr", "the"}
ORG_STOPWORDS    = {"inc", "llc", "llp", "ltd", "corp", "co", "the", "and"}

def tokens(text):
    t = re.sub(r"[^a-z0-9\s]", " ", text.lower())
    return [w for w in t.split() if len(w) > 1]

# Words that describe a role, not a party. An occurrence made only of these ("Defendants",
# "the Answering Defendants") names nobody in particular and must never carry a match.
ROLE_WORDS = {"defendant", "defendants", "plaintiff", "plaintiffs", "answering",
              "claimant", "claimants", "insured", "counsel", "attorney", "respondent",
              "petitioner", "the", "dr", "mr", "mrs", "ms", "his", "her", "their"}

def name_tokens(text, etype):
    stop = PERSON_STOPWORDS if etype == "person" else ORG_STOPWORDS
    return {w for w in tokens(text) if w not in stop and w not in ROLE_WORDS}

# Lowercase words a name may contain. Anything else in lowercase ("is", "against",
# "billed") means the string is a phrase about a party, not the party's name.
NAME_PARTICLES = {"of", "and", "the", "de", "la", "del", "da", "di", "du", "van", "von",
                  "der", "le", "for", "at", "in", "on", "dba", "aka", "fka"}
_WORD = re.compile(r"[A-Za-z][A-Za-z'’\-]*")

def name_like(s, etype):
    """Could this string be a name, rather than a sentence fragment that contains one?

    Court text breaks the naive rule badly: the model's occurrence lists hold "Nexray is not
    engaged" and "Ninth Cause of Action against Nexray, Pierre, and Weiner", and a string
    sharing a rare word with two parties bridges them into one cluster."""
    # A trade name is two names: judge each side ("X, P.C. d/b/a Y" is one party)
    dba = re.split(r"\b(?:d\s*/\s*b\s*/\s*a|a\s*/\s*k\s*/\s*a|f\s*/\s*k\s*/\s*a)\b", s, flags=re.I)
    if len(dba) > 1:
        return all(p.strip(" ,") and name_like(p.strip(" ,"), etype) for p in dba)
    words = _WORD.findall(s)
    # single letters ("P.C.", initials) do not make a string long
    if not words or len([w for w in words if len(w) > 1]) > 10:
        return False
    if any(w[0].islower() and len(w) > 1 and w.lower() not in NAME_PARTICLES for w in words):
        return False
    if etype == "person" and re.search(r"\band\b|&|;", s, re.I):
        return False              # "Rutland, Pierre, and Moy" is three parties
    if etype != "person":
        # "Rutland and Nexray" is two parties; "Johnson & Johnson" and
        # "Schwartz, Conroy & Hack, PC" are one. A list has distinct parts and no word that
        # marks a single organization.
        parts = {p.strip().lower() for p in re.split(r"\s*(?:,|\band\b|&)\s*", s) if p.strip()}
        marked = {w.upper().strip(".") for w in words} & ORG_MARKERS
        if len(parts) > 1 and not marked:
            return False
    return True

ORG_MARKERS = {"PC", "PLLC", "LLC", "INC", "CORP", "CORPORATION", "CO", "LTD", "LLP", "LP", "PA",
               "COMPANY", "GROUP", "ASSOCIATES", "PARTNERS", "FIRM", "LAW", "SERVICES",
               "MEDICAL", "CENTER", "HEALTH", "INSURANCE", "BANK", "TRUST", "HOSPITAL", "CLINIC"}

def _capital_runs(s):
    """Maximal runs of capitalised words: 'billed American Transit for' -> ['American Transit']."""
    runs, cur = [], []

    def close():
        while cur and cur[-1][0].islower():      # "American Transit for" -> "American Transit"
            cur.pop()
        if cur:
            runs.append(" ".join(cur))
        cur.clear()

    for w in re.findall(r"[A-Za-z][A-Za-z'’\-\.]*|[,;]", s):
        if w[0].isupper() or (cur and w.lower() in NAME_PARTICLES - {"and"}):
            cur.append(w)
        else:
            close()
    close()
    return [r for r in runs if name_tokens(r, "organization")]

def fullest_name(m):
    """The most informative way this mention's own note names the party, or None.

    The model picks one `name` per mention and lists the rest in `occurrences`; sometimes
    the name it picks is the thinnest ("Mr. Pierre") while its own list holds the full one
    ("BRADLEY PIERRE"). An occurrence qualifies only if it is name-like and shares a real
    name word with the name, which keeps "Bradley Pierre" and drops "Defendants", "him",
    "which" and "Nexray is not engaged".

    When the model's own `name` is a phrase, a single capitalised run inside it is used
    ("billed American Transit for ..." -> "American Transit"). With several runs the
    mention names more than one party, and None is returned: better unlinked than a
    bridge between parties."""
    name = m.get("name") or m["quote"]
    if not name_like(name, m["type"]):
        runs = _capital_runs(name)
        if len(runs) != 1:
            return None
        name = runs[0]
    base = name_tokens(name, m["type"])
    best, best_toks = name, base
    for occ in m.get("occurrences", []):
        if not name_like(occ, m["type"]):
            continue
        toks = name_tokens(occ, m["type"])
        if same_name_family(toks, base) and len(toks) > len(best_toks):
            best, best_toks = occ, toks
    return best

def same_name_family(a, b):
    """One name is a short or long form of the other: its words are a subset.

    Sharing one word is not enough. The model sometimes lists sibling companies among one
    mention's occurrences ("Allstate Indemnity Company" as an occurrence of "Allstate
    Property and Casualty Insurance Company"); a shared "Allstate" would make each an alias
    of the other."""
    return bool(a) and bool(b) and (a <= b or b <= a)

# ---- outside frequency tables ---------------------------------------------------------
def _load_counts(fname):
    p = Path(REFERENCE_DIR, fname)
    if not p.exists():
        return None
    with gzip.open(p, "rt", encoding="utf-8") as f:
        rd = csv.reader(f); next(rd)
        return {k: int(v) for k, v in rd}

SURNAMES   = _load_counts("surnames.csv.gz")
FIRSTNAMES = _load_counts("first_names.csv.gz")
ORG_DF     = _load_counts("org_tokens.csv.gz")
_meta_p = Path(REFERENCE_DIR, "reference_meta.json")
REF_META = json.loads(_meta_p.read_text(encoding="utf-8")) if _meta_p.exists() else {}

# Floors for names below each table's publication cutoff: rare, but not impossible.
SURNAME_FLOOR, FIRSTNAME_FLOOR, ORG_DF_FLOOR = 50, 20, 1
FLAT_FREQ = 1e-3                     # used only when a table is missing

SURNAME_TOTAL = REF_META.get("census", {}).get("people") or (sum(SURNAMES.values()) if SURNAMES else 1)
FIRST_TOTAL   = REF_META.get("ssa", {}).get("births") or (sum(FIRSTNAMES.values()) if FIRSTNAMES else 1)
ORG_N         = REF_META.get("nppes", {}).get("organizations") or 1
_initials = defaultdict(int)
for _n, _c in (FIRSTNAMES or {}).items():
    _initials[_n[0]] += _c
INITIAL_SHARE = {k: v / FIRST_TOTAL for k, v in _initials.items()}

RARITY_SOURCE = {"surnames": "census2010" if SURNAMES else "FLAT",
                 "first_names": "ssa1930-2005" if FIRSTNAMES else "FLAT",
                 "org_tokens": "nppes" if (ORG_DF and ORG_N > 1) else "FLAT"}

def surname_freq(last):
    if not SURNAMES: return FLAT_FREQ
    parts = [last] + (last.split("-") if "-" in last else [])
    return min(SURNAMES.get(p, SURNAME_FLOOR) for p in parts) / SURNAME_TOTAL

def first_freq(first):
    if not FIRSTNAMES: return FLAT_FREQ
    return FIRSTNAMES.get(first, FIRSTNAME_FLOOR) / FIRST_TOTAL

def initial_share(letter):
    return INITIAL_SHARE.get(letter, 1 / 26) if FIRSTNAMES else 1 / 26

def org_idf(tok):
    """Bits of surprise in two organizations sharing this name word."""
    if RARITY_SOURCE["org_tokens"] == "FLAT": return 6.0
    return math.log2(ORG_N / ORG_DF.get(tok, ORG_DF_FLOOR))

# ---- name parsing ---------------------------------------------------------------------
TITLES      = {"DR", "DOCTOR", "MR", "MRS", "MS", "MISS", "HON", "JUDGE", "PROF"}
CREDENTIALS = {"MD", "DO", "DC", "DPM", "DDS", "DMD", "PHD", "PT", "DPT", "NP", "PA", "RN",
               "LPN", "LAC", "ESQ", "JD", "CPA", "OT", "OTR", "FACS", "FACP", "MPH", "LCSW",
               "PSYD"}
# A party holds one of these at most. Two different ones on a pair is a veto (cell 18).
PRIMARY_LICENSE = {"MD", "DO", "DC", "DPM", "DDS", "DMD"}
# Safe to read as a credential without a comma before it ("Marvin Moy MD"); "DO" and "PA"
# are also ordinary words and states, so they count only after a comma.
BARE_CREDENTIALS = CREDENTIALS - {"DO", "PA", "PT", "OT"}
NAME_SUFFIXES = {"JR", "SR", "II", "III", "IV"}
ROLE_WORDS_UP = {w.upper() for w in ROLE_WORDS}
ORG_LEGAL = {"PC", "PLLC", "LLC", "INC", "CORP", "CORPORATION", "CO", "LTD", "LLP", "LP",
             "PA", "COMPANY", "INCORPORATED", "THE", "AND", "OF"}

def _undot(s):
    """'M.D.' -> 'MD', 'P.C.' -> 'PC', then any remaining dot is a space ('A.' -> 'A')."""
    s = re.sub(r"\b((?:[A-Za-z]{1,2}\.){2,})", lambda m: m.group(1).replace(".", ""), s)
    return s.replace(".", " ")

def _words(s):
    return [w.upper() for w in re.findall(r"[A-Za-z][A-Za-z'\-]*", s)]

def parse_person(name):
    """Split a person's name into first / middle initial / last, with credentials and title.

    Handles 'Dr. A. Monroe', 'WILLIAM A. WEINER, D.O.', 'Moy, Marvin' and 'Mr. Pierre'.
    A single remaining word is taken as the surname."""
    s = _undot(name)
    head, _, tail = s.partition(",")
    hw, tw = _words(head), _words(tail)
    creds = {w for w in tw if w in CREDENTIALS} | {w for w in hw[1:] if w in BARE_CREDENTIALS}
    title = next((w for w in hw if w in TITLES), None)
    drop = TITLES | CREDENTIALS | NAME_SUFFIXES | ROLE_WORDS_UP
    core_h = [w.strip("'-") for w in hw if w not in drop]
    core_t = [w.strip("'-") for w in tw if w not in drop]
    if len(core_h) == 1 and core_t:          # "Moy, Marvin"
        core = core_t + core_h
    else:
        core = core_h
    core = [w for w in core if w]
    first = middle = last = ""
    if len(core) == 1:
        last = core[0]
    elif len(core) >= 2:
        first, last = core[0], core[-1]
        middle = core[1][0] if len(core) > 2 else ""
    return {"first": first, "middle": middle, "last": last,
            "creds": sorted(creds), "title": title}

def org_aliases(name):
    """Name variants of one organization: d/b/a parts split, legal suffixes stripped."""
    parts = re.split(r"\b(?:d\s*/\s*b\s*/\s*a|a\s*/\s*k\s*/\s*a|f\s*/\s*k\s*/\s*a|doing business as)\b",
                     name, flags=re.I)
    out = []
    for p in parts:
        # split on any punctuation: "IMANGING,PC." is two words, not "IMANGINGPC"
        toks = re.findall(r"[A-Z0-9]+", _undot(p).upper().replace("'", ""))
        # OCR splits a suffix too ("COMP ANY"); rejoin it so it is dropped like one
        joined, i = [], 0
        while i < len(toks):
            if i + 1 < len(toks) and toks[i] + toks[i + 1] in ORG_LEGAL:
                joined.append(toks[i] + toks[i + 1]); i += 2
            else:
                joined.append(toks[i]); i += 1
        toks = [ORG_ABBREV.get(t, t) for t in joined]
        toks = [t for t in toks if len(t) > 1 and t not in ORG_LEGAL and t not in ROLE_WORDS_UP]
        if toks and toks not in out:
            out.append(toks)
    return out

# Abbreviations of organization words, expanded before comparison so "AMERICAN TRANSIT INS.
# CO." meets "American Transit Insurance Company" word for word.
ORG_ABBREV = {"INS": "INSURANCE", "MED": "MEDICAL", "CTR": "CENTER", "CNTR": "CENTER",
              "SVCS": "SERVICES", "SVC": "SERVICES", "ASSOC": "ASSOCIATES", "ASSN": "ASSOCIATION",
              "HOSP": "HOSPITAL", "MGMT": "MANAGEMENT", "INTL": "INTERNATIONAL", "NATL": "NATIONAL",
              "DIAG": "DIAGNOSTIC", "REHAB": "REHABILITATION", "PHYS": "PHYSICAL", "THER": "THERAPY"}

# identifiers only a licensed professional holds
PROFESSIONAL_IDS = {"npi", "bar_number", "dea_number", "state_license"}

PROFESSIONAL_WORDS = {"physician", "doctor", "provider", "clinic", "attorney", "counsel",
                      "lawyer", "chiropractor", "therapist", "radiologist", "surgeon",
                      "nurse", "pharmacist", "practitioner", "treating", "dentist",
                      "acupuncturist", "examiner", "prescriber", "referring"}

def role_class(r):
    if r["type"] == "organization": return "organization"
    if r["type"] == "vehicle":      return "vehicle"
    if r.get("creds") or r.get("title") in ("DR", "DOCTOR"): return "professional"
    if any(t in PROFESSIONAL_IDS for t, _, _ in r["details"]): return "professional"
    if any(w in PROFESSIONAL_WORDS for role in r["roles"] for w in re.split(r"[_\s]+", role)):
        return "professional"
    return "private"

def parse_location(address):
    """City and state from an address as written, or None when no state can be read.

    '135-25F 79th Street, Suite 2B, Howard Beach, New York 11414' -> HOWARD BEACH, NY
    '4410 N Broadway, Chicago IL 60640'                             -> CHICAGO, IL
    Only a state that is spelled out or a two-letter code counts; a street line never does."""
    s = re.sub(r"\s+", " ", address or "").strip().rstrip(".")
    s = re.sub(r",?\s*\d{5}(?:-\d{4})?$", "", s)            # the ZIP, if any
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if not parts:
        return None
    last = re.sub(r"[^A-Za-z ]", "", parts[-1]).upper().split()
    state, city = "", ""
    for k in (3, 2, 1):                                   # "NEW YORK", "IL", "WEST VIRGINIA"
        if len(last) >= k:
            tail = " ".join(last[-k:])
            code = tail if (k == 1 and tail in STATE_CODES) else US_STATES.get(tail, "")
            if code:
                state, rest = code, last[:-k]
                city = " ".join(rest) if rest else (
                    re.sub(r"[^A-Za-z .'-]", "", parts[-2]).upper().strip() if len(parts) > 1 else "")
                break
    if not state:
        return None
    if re.search(r"\d", city) or len(city.split()) > 4:
        city = ""                                          # a street line, not a city
    return {"city": city, "state": state}

def build_mention(key, etype, name, *, claim_id, note_key, occurrences=(), details=(),
                  roles=(), chunk=0, name_as_extracted=None, spans=None, linkable=True,
                  locations=()):
    """One mention record. Used by the pipeline below and by the self-tests, so both
    exercise exactly the same parsing. `linkable=False` keeps a mention whose name is a
    phrase naming several parties: it stays in its claim as its own entity, visibly, and is
    never proposed as a candidate pair. `locations` are the cities and states of addresses
    the mention owns; only the watchlist comparison reads them (cell 18c)."""
    p = parse_claim_id(claim_id)
    r = {"key": key, "type": etype, "name": name, "name_as_extracted": name_as_extracted or name,
         "linkable": linkable,
         "claim_id": claim_id, "occurrence_id": p.get("occurrence_id"),
         "client_id": p.get("client_id"), "note_key": note_key, "chunk": chunk,
         "occurrences": list(occurrences), "details": [tuple(d) for d in details],
         "locations": [dict(x) for x in locations],
         "roles": sorted({x.lower() for x in roles}), **(spans or {})}
    base = name_tokens(name, etype)
    forms = [name] + [o for o in occurrences
                      if name_like(o, etype) and same_name_family(name_tokens(o, etype), base)]
    r["forms"] = list(dict.fromkeys(forms))
    if etype == "person":
        r.update(parse_person(name))
        r["creds"] = sorted(set(r["creds"]).union(*(parse_person(f)["creds"] for f in forms)))
        r["title"] = r["title"] or next((parse_person(f)["title"] for f in forms
                                         if parse_person(f)["title"]), None)
    elif etype == "organization":
        aliases = []
        for f in forms:
            for a in org_aliases(f):
                if a not in aliases: aliases.append(a)
        # a short form that is a subset of a longer alias adds nothing ("Nexray" in
        # "Nexray Medical Imaging")
        r["aliases"] = [a for a in aliases
                        if not any(set(a) < set(b) for b in aliases)] or aliases
    else:
        r["vehicle_key"] = " ".join(tokens(name))
    r["role_class"] = role_class(r)
    return r

mentions = []
for e in envelopes:
    roles = defaultdict(set)
    for a in e["action_mentions"]:
        for pp in a["participants"]:
            roles[pp["mention_id"]].add(pp["role"])
    for m in e["entity_mentions"]:
        details = [(d["detail_type"], d["normalized"], d["basis"]) for d in e["detail_mentions"]
                   if d["owner_ref"] == m["mention_id"] and d["normalized"]]
        full = fullest_name(m)
        locs = [loc for d in e["detail_mentions"]
                if d["owner_ref"] == m["mention_id"] and d["detail_type"] == "address"
                and (loc := parse_location(d["raw_value"]))]
        mentions.append(build_mention(
            f"{e['note_key']}:{m['mention_id']}", m["type"], full or m.get("name") or m["quote"],
            linkable=full is not None,
            claim_id=e["claim_id"], note_key=e["note_key"],
            occurrences=m.get("occurrences", []), details=details,
            roles=roles[m["mention_id"]], chunk=(m.get("_chunk") or [0])[0],
            name_as_extracted=m.get("name"), locations=locs,
            spans={"raw_span": m.get("raw_span"), "clean_span": m.get("clean_span")}))
mention_by_key = {r["key"]: r for r in mentions}

print(f"rarity tables: {RARITY_SOURCE}")
if "FLAT" in RARITY_SOURCE.values():
    print("!! a rarity table is missing: run corpus/reference/fetch_reference.py. Links scored")
    print("!! in this run treat every name as equally common, and are stamped that way.")
print(f"{len(mentions)} mentions in {len(claims)} claims")
unlinkable = [r for r in mentions if not r["linkable"]]
print(f"  unlinkable (name is a phrase naming several parties): {len(unlinkable)}")
for r in unlinkable[:8]:
    print(f"    {r['key']:<24} {r['name'][:70]!r}")
by_role = defaultdict(int)
for r in mentions: by_role[r["role_class"]] += 1
print(f"  role classes: {dict(by_role)}")
for r in mentions[:12]:
    parsed = (f"first={r['first']!r} mid={r['middle']!r} last={r['last']!r} creds={r['creds']}"
              if r["type"] == "person" else
              f"aliases={r.get('aliases', r.get('vehicle_key'))}")
    print(f"  {r['key']:<24} {r['role_class']:<12} {r['name'][:30]!r:<32} {parsed}")
if len(mentions) > 12:
    print(f"  ... {len(mentions) - 12} more")


### How `recordlinkage` is configured, and why

**One page on the linking strategy.** Cells 17–19 implement it. The agreed design is in
`AGENTS.md` rule 5.

**The shape.** Links sit underneath and a merged view sits on top. The linker never merges
anything. It writes one **link** per candidate pair of mentions, and every link carries a
probability, a **basis** and a **veto**. Merged entities are computed when something reads
them (cell 19, `goko/projection.py`), at one of three lenses. One mechanism serves every
distance: two chunks of one note, two notes of one claim, and two claims of different
insurers.

**What `recordlinkage` does, and what it does not.**

| Stage | Tool | Configuration |
|---|---|---|
| Candidate pairs | `recordlinkage.Index` | Persons: `Block` on the NYSIIS code of the surname, which lets Weiner/Wiener meet. Organizations: `Block` on the rarest name word plus `SortedNeighbourhood(window=5)` on the sorted name. Vehicles: `Block` on the normalized description. Identifiers: a plain join on `(type, value)`, because a mention can own many values and `Block` takes one column. |
| Field comparisons | `recordlinkage.Compare` | Jaro-Winkler on surname, first name and organization name; exact match on the phonetic code. |
| Scoring | **ours** (cell 18) | Fellegi-Sunter form with fixed parameters. `recordlinkage`'s classifiers use one coincidence rate per *field*; rarity needs one per *value* ("Pierre" is rarer than "Smith"), so the scorer is ours. |
| Learning the weights | `recordlinkage.ECMClassifier` | **Disabled** (cell 18b). Enable criteria are in that cell. |

A pair from the same chunk of the same note is never a candidate: the model already
resolved coreference there, and two mentions it kept apart are its assertion that they
differ.

**The score.** Each field that agrees adds `log2(m/u)` bits, where:

- **m** is how often the field agrees when the two *are* one party. It is a stated
  assumption, e.g. 0.93 for an exact surname.
- **u** is how often it agrees *by coincidence*. For names it comes from the outside tables,
  so "Pierre" agreeing is worth far more than "Smith". For identifiers it is a fixed small
  number.

Disagreement subtracts bits. The total is added to the prior log-odds, and the result is
turned into a probability `p`.

**Priors by distance × role.**

| Distance | Prior |
|---|---|
| Same note (different chunks) | 0.5 |
| Same claim | 0.25 |
| Same occurrence | 0.1 |
| Across claims | Depends on role (next table) |

| Role, across claims | Same client | Different client |
|---|---|---|
| Professionals and organizations | 1e-3 | 1e-4 |
| Private persons | 1e-6 | 1e-7 |

Recurring across claims is expected for a clinic and exceptional for a claimant.

| Against one watchlist record (cell 18c) | Prior | What it declares |
|---|---|---|
| Corpus professional or organization | 5e-7 | ~4% of such parties in a fraud file are on OIG LEIE, spread over its ~84,000 records |
| Corpus private person | 1e-7 | ~1%, spread the same way |

The corpus side's role decides: the question is how likely *this* party is to be listed.
Against a watchlist record, and only there, a **location** field is compared: the mention's
own addresses against the record's city and state. Same state adds
`log2(0.8 / the state's population share)`, the same city `log2(0.6 / 0.05)` more; a different
state counts against (`log2(0.2)` ≈ −2.3 bits), never vetoes; a missing location is no evidence.

**Basis, separate from score.** A link records which fields actually contributed. The basis
classes:

- `identifier`: NPI, TIN, SSN, VIN, bar number, DEA number, professional license, phone,
  email, license plate, or bank account.
- `address`
- `dob`: a person's name plus an agreeing date of birth. A date of birth adds weight but is
  never an identifier on its own: about 1 in 29,000 people share one.
- `co_party`: a name, plus an anchored co-party (next section).
- `location`: a name, plus an agreeing city or state (watchlist records only).
- `name_only`

**Identifiers.** `u` is the assumed chance of a coincidental match; `m` is 0.95 when both
owners are stated, 0.60 when either was inferred.

| Type | u | One per party (conflict vetoes) | Normalized as |
|---|---|---|---|
| SSN | 1e-8 | yes | digits |
| DEA number | 1e-8 | yes | `AB1234563`, check digit verified |
| bank account | 1e-8 | no | `ROUTING:ACCOUNT` (routing ABA-checked) or `ACCOUNT` |
| VIN | 1e-8 | yes | uppercase |
| NPI, TIN | 1e-7 | yes | digits (NPI Luhn-checked) |
| professional license | 1e-7 | no | `STATE:NUMBER` or `NUMBER` |
| bar number | 1e-6 | yes | letters and digits |
| email | 1e-6 | no | lowercase |
| license plate | 1e-6 | no | `STATE:PLATE`, uppercase, no spaces |
| phone | 1e-4 | no | `+1` and ten digits |
| address | 1e-3 | no | lowercase words |
| date of birth | 1/(365×80) | no, but a difference counts against (log2(0.05) ≈ −4.3 bits) | ISO date |

A license, plate or account number matches another when the numbers are equal and the
issuers are equal or one is unstated. Policy and claim numbers are not identifiers.

An exact match on a rare name can reach p = 0.99 and is still labelled `name_only`. The
score says how confident. The basis says what the confidence rests on. Neither is ever
silently upgraded.

**Vetoes.**

- Two different values of an identifier a party has only one of (NPI, SSN, TIN, VIN, bar
  number, DEA number), both stated.
- Two different primary licenses (M.D. vs D.O.).
- Two organization names that share a head but each keep a distinctive word the other
  lacks ("Allstate Indemnity Company" vs "Allstate Property & Casualty Insurance Company":
  sibling companies, not one party under two names). OCR splits are rejoined first.

A vetoed pair can never be in one merged entity, even transitively, at any lens.

**Co-parties.**

- **(A) Anchored.** A cross-claim link gains 3 bits per co-party pair that is itself linked on
  an identifier, up to 2. Evidence flows one way, from the anchored link to the other.
  Name-only matches never vouch for each other, so a loop of coincidences (the
  Garcia / Dr. Lee / City Medical case) cannot confirm itself.
- **(B) Displayed.** On every cross-claim link, the count of co-parties that also match by
  name is shown. It never changes the score or the basis.

**Lenses** (`goko/projection.py`).

| Lens | Admits |
|---|---|
| Strict | Identifier basis, p ≥ 0.90 |
| Default | Any basis, p ≥ 0.80 |
| Broad | Any basis, p ≥ 0.10 |

Clusters are built from the strongest link down. The confidence of a merged entity is its
**weakest link**. A union that would join a vetoed pair, or grow a cluster past 40 mentions,
is refused and recorded (the cluster-size alarm).

The same lens decides **Flagged for review** (`goko/watchlist.py`): an entity is flagged when
any of its mentions has a watchlist link the lens admits. There is no second threshold.

**Every link explains itself.** Each link in `links.json` and `watchlist_links.json` carries
`fields`: per compared field, the two values, the agreement level, `m`, `u` and what `u`
rests on (the surname's Census frequency, each organization word's NPPES share, an
identifier's assumed rate, a state's population share), and the bits the scorer added. The
rows add up to the link's `bits` (a self-test checks it). The app's decision card is built
from them.

**Known limits.**

- Fields are treated as independent, which they are not (first and last names correlate), so
  raw probabilities run high.
- The m-values are assumptions, not measurements. Phase 3 calibrates them against the docket
  party lists and registry records.


## 17 — Candidate pairs and field comparisons (`recordlinkage`)

Indexing proposes the pairs worth scoring; comparison measures string agreement on them.
Neither decides anything. Every candidate is printed with the index that proposed it, so a
pair that was never proposed — the one kind of miss no scorer can recover — is visible here.


In [ ]:
try:
    import pandas as pd
    import recordlinkage as rl
    from recordlinkage.preprocessing import phonetic
except ImportError as ex:
    raise ImportError("cells 17-18 need recordlinkage and pandas: "
                      "pip install -r requirements.txt") from ex

def _frame(recs):
    rows = []
    for r in recs:
        org = " ".join(r["aliases"][0]) if r["type"] == "organization" and r.get("aliases") else ""
        rare = (min(r["aliases"][0], key=lambda t: (-org_idf(t), t))
                if r["type"] == "organization" and r.get("aliases") else "")
        rows.append({"key": r["key"], "type": r["type"],
                     "last": r.get("last") or None, "first": r.get("first") or None,
                     "org": org or None, "org_sort": " ".join(sorted(org.split())) or None,
                     "org_rare": rare or None, "vehicle": r.get("vehicle_key") or None})
    cols = ["key", "type", "last", "first", "org", "org_sort", "org_rare", "vehicle"]
    df = pd.DataFrame(rows, columns=cols).set_index("key")     # columns survive zero mentions
    df["last_ph"] = None
    has_last = df["last"].notna()
    if has_last.any():
        df.loc[has_last, "last_ph"] = phonetic(df.loc[has_last, "last"].str.lower(),
                                               method="nysiis").values
    return df

def candidate_pairs(recs):
    """Returns {(a, b): {"proposed_by": set}} with a < b, same-chunk pairs dropped.
    Unlinkable mentions (a phrase naming several parties) are never proposed."""
    recs = [r for r in recs if r.get("linkable", True)]
    df = _frame(recs)
    by_key = {r["key"]: r for r in recs}
    found = defaultdict(set)

    def add(index, label):
        for a, b in index:
            if a == b: continue
            a, b = sorted((a, b))
            found[(a, b)].add(label)

    people = df[(df["type"] == "person") & df["last_ph"].notna()]
    if len(people) > 1:
        add(rl.Index().block("last_ph").index(people), "block:surname_nysiis")
    orgs = df[(df["type"] == "organization") & df["org"].notna()]
    if len(orgs) > 1:
        add(rl.Index().block("org_rare").index(orgs), "block:org_rarest_word")
        idx = rl.Index(); idx.sortedneighbourhood("org_sort", window=5)
        add(idx.index(orgs), "sorted_neighbourhood:org_name")
    veh = df[(df["type"] == "vehicle") & df["vehicle"].notna()]
    if len(veh) > 1:
        add(rl.Index().block("vehicle").index(veh), "block:vehicle")
    owners = defaultdict(list)                 # a mention can own many identifiers
    for r in recs:
        for t, v, _ in r["details"]:
            if t in JOIN_TYPES:                # a shared date of birth proposes nothing
                owners[(t, id_number(t, v))].append(r["key"])
    for (t, v), ks in owners.items():
        add([(x, y) for i, x in enumerate(ks) for y in ks[i + 1:]], f"shared:{t}")

    out = {}
    for (a, b), how in found.items():
        ra, rb = by_key[a], by_key[b]
        if ra["type"] != rb["type"]:
            continue                            # a hard constraint, never scored
        if ra["note_key"] == rb["note_key"] and ra["chunk"] == rb["chunk"]:
            continue                            # the model's own coreference call
        out[(a, b)] = {"proposed_by": sorted(how)}
    return out, df

def compare_pairs(cands, df):
    """Field agreement from recordlinkage.Compare, as {pair: {feature: value}}."""
    if not cands:
        return {}
    mi = pd.MultiIndex.from_tuples(list(cands), names=["a", "b"])
    cmp = rl.Compare()
    cmp.string("last", "last", method="jarowinkler", missing_value=0.0, label="last_jw")
    cmp.string("first", "first", method="jarowinkler", missing_value=0.0, label="first_jw")
    cmp.string("org", "org", method="jarowinkler", missing_value=0.0, label="org_jw")
    cmp.exact("last_ph", "last_ph", missing_value=0, label="last_phonetic")
    feats = cmp.compute(mi, df)
    return {pair: row.to_dict() for pair, row in feats.iterrows()}

cands, mention_frame = candidate_pairs(mentions)
features = compare_pairs(cands, mention_frame)

by_source = defaultdict(int)
for c in cands.values():
    for s in c["proposed_by"]: by_source[s] += 1
print(f"{len(cands)} candidate pairs from {len(mentions)} mentions "
      f"(of {len(mentions) * (len(mentions) - 1) // 2} possible)")
for s, n in sorted(by_source.items()):
    print(f"  {s:<34} {n}")
for pair in list(cands)[:10]:
    f = features[pair]
    print(f"  {pair[0]:<24} {pair[1]:<24} by={','.join(cands[pair]['proposed_by'])}  "
          f"last={f['last_jw']:.2f} first={f['first_jw']:.2f} org={f['org_jw']:.2f}")


## 18 — Link scoring (Fellegi-Sunter form, fixed parameters)

One formula for every pair at every distance. The weights and priors below are stated
assumptions, not fitted values — the page above cell 17 explains each, and cell 18b is where
they would be learned once there is enough data to learn them.

The result is a **link** per candidate pair: `p`, the `basis` it rests on, any `veto`, the
per-field bits that produced it, and the co-party evidence (A scored, B displayed). Links are
written to `links.json`; mentions (with their spans) to `mentions.json`. Nothing is merged.


In [ ]:
log2 = math.log2

# m: how often a field agrees when the two mentions ARE one party (assumed, not fitted)
M_LAST   = {"exact": 0.93, "close": 0.05, "differ": 0.02}
M_FIRST  = {"exact": 0.85, "close": 0.08, "differ": 0.04}
M_INITIAL_AGREE, M_MIDDLE_AGREE = 0.95, 0.90
JW_CLOSE = 0.92                               # Jaro-Winkler at or above: a spelling variant
# u: the chance two different parties show the same value by coincidence (assumed)
ID_U = {"npi": 1e-7, "tin": 1e-7, "ssn": 1e-8, "vin": 1e-8, "bar_number": 1e-6,
        "phone": 1e-4, "address": 1e-3,
        "email": 1e-6, "license_plate": 1e-6, "dea_number": 1e-8, "state_license": 1e-7,
        "bank_account": 1e-8,
        "dob": 1 / (365 * 80)}      # ~80 birth years in play, any day of each
ID_M = {"stated": 0.95, "inferred": 0.60}      # an inferred owner is weaker evidence
# A party has at most one of these: two different stated values veto the pair.
SINGULAR_TYPES = ("npi", "ssn", "tin", "vin", "bar_number", "dea_number")
# Agreement on one of these makes the link identifier-backed (the strict lens admits it).
# A date of birth is not among them: it adds weight to a person's name, never stands alone.
IDENTIFIER_BASIS = set(SINGULAR_TYPES) | {"phone", "email", "license_plate",
                                          "state_license", "bank_account"}
ORG_NO_OVERLAP_BITS = -8.0
CO_PARTY_BITS, CO_PARTY_MAX = 3.0, 2

PRIOR_P = {
    "same_note": 0.5, "same_claim": 0.25, "same_occurrence": 0.1,
    ("same_client", "recurring"): 1e-3, ("same_client", "private"): 1e-6,
    ("different_client", "recurring"): 1e-4, ("different_client", "private"): 1e-7,
    # A corpus mention against ONE watchlist record (cell 18c). Declared semantics: the
    # chance the corpus party is on the list at all, given its role, spread over the list's
    # ~84,000 records. Professionals and organizations named in a fraud file: ~4% are on
    # OIG LEIE (5e-7 x 84k). Private persons: ~1% (1e-7 x 84k); the list also excludes
    # owners, managers and employees after a conviction. Assumptions, not measurements.
    ("watchlist", "recurring"): 5e-7, ("watchlist", "private"): 1e-7,
}
SUSPICION_WEIGHT = {          # the same identity, read as a fraud signal
    "same_note": 0.0, "same_claim": 0.0,
    "same_occurrence": 0.0,   # one incident, several coverages: expected
    "same_client": 0.25, "different_client": 1.0,
    "watchlist": 0.0,         # a watchlist match is a flag for review, not a cross-claim signal
}

# Location, compared only against a watchlist record: a corpus mention's own addresses
# against the record's city and state. m: how often one party's two addresses agree (people
# move; a listed address can be a prison). u: the state's share of the US population, and
# for the city, a flat chance two addresses in one state share a city.
M_STATE, M_CITY, U_CITY = 0.80, 0.60, 0.05
STATE_POP_M = {   # 2020 Census, millions
    "CA": 39.54, "TX": 29.15, "FL": 21.54, "NY": 20.20, "PA": 13.00, "IL": 12.81, "OH": 11.80,
    "GA": 10.71, "NC": 10.44, "MI": 10.08, "NJ": 9.29, "VA": 8.63, "WA": 7.71, "AZ": 7.15,
    "MA": 7.03, "TN": 6.91, "IN": 6.79, "MD": 6.18, "MO": 6.15, "WI": 5.89, "CO": 5.77,
    "MN": 5.71, "SC": 5.12, "AL": 5.02, "LA": 4.66, "KY": 4.51, "OR": 4.24, "OK": 3.96,
    "CT": 3.61, "UT": 3.27, "IA": 3.19, "NV": 3.10, "AR": 3.01, "MS": 2.96, "KS": 2.94,
    "NM": 2.12, "NE": 1.96, "ID": 1.84, "WV": 1.79, "HI": 1.46, "NH": 1.38, "ME": 1.36,
    "RI": 1.10, "MT": 1.08, "DE": 0.99, "SD": 0.89, "ND": 0.78, "AK": 0.73, "DC": 0.69,
    "VT": 0.64, "WY": 0.58, "PR": 3.29, "GU": 0.15, "VI": 0.09}
_POP_TOTAL = sum(STATE_POP_M.values())

def state_share(st):
    return STATE_POP_M.get(st, 0.5) / _POP_TOTAL

def location_bits(a, b):
    """Best agreement between the mention's locations and the record's. None when either
    side has none: a missing location is no evidence either way."""
    best = None
    for x in a.get("locations") or []:
        for y in b.get("locations") or []:
            u = state_share(y["state"])
            if x["state"] == y["state"]:
                bits, level = log2(M_STATE / u), "same_state"
                if x.get("city") and y.get("city"):
                    if x["city"] == y["city"]:
                        bits, level = bits + log2(M_CITY / U_CITY), "same_city"
                    else:
                        bits, level = bits + log2((1 - M_CITY) / (1 - U_CITY)), "same_state_other_city"
            else:
                bits, level = log2((1 - M_STATE) / (1 - u)), "different_state"
            if best is None or bits > best[0]:
                best = (bits, level, x, y, u)
    return best

def distance_of(a, b):
    if a.get("source") == "oig_leie" or b.get("source") == "oig_leie":
        return "watchlist"
    if a["note_key"] == b["note_key"]:           return "same_note"
    if a["claim_id"] == b["claim_id"]:           return "same_claim"
    if a["occurrence_id"] == b["occurrence_id"]: return "same_occurrence"
    if a["client_id"] == b["client_id"]:         return "same_client"
    return "different_client"

def prior_of(a, b, dist):
    if dist in PRIOR_P:
        return PRIOR_P[dist], dist
    recurring = {"professional", "organization"}
    if dist == "watchlist":
        # the corpus side decides: the question is how likely THIS party is to be listed
        corpus = a if a.get("source") != "oig_leie" else b
        cls = "recurring" if corpus["role_class"] in recurring else "private"
    else:
        cls = "recurring" if (a["role_class"] in recurring or b["role_class"] in recurring) else "private"
    return PRIOR_P[(dist, cls)], f"{dist}/{cls}"

def person_name_bits(a, b, f):
    w, level = {}, {}
    la, lb = a["last"], b["last"]
    if la and lb:
        fr = max(surname_freq(la), surname_freq(lb))
        if la == lb:
            level["last"], w["last"] = "exact", log2(M_LAST["exact"] / fr)
        elif f["last_jw"] >= JW_CLOSE:
            level["last"], w["last"] = "close", log2(M_LAST["close"] / min(1.0, 5 * fr))
        else:
            level["last"], w["last"] = "differ", log2(M_LAST["differ"])
    else:
        level["last"] = "missing"
    fa, fb = a["first"], b["first"]
    if fa and fb:
        if len(fa) == 1 or len(fb) == 1:        # one side is an initial
            share = initial_share(fa[0])
            if fa[0] == fb[0]:
                level["first"], w["first"] = "initial", log2(M_INITIAL_AGREE / share)
            else:
                level["first"], w["first"] = "differ", log2((1 - M_INITIAL_AGREE) / (1 - share))
        elif fa == fb:
            level["first"], w["first"] = "exact", log2(M_FIRST["exact"] / first_freq(fa))
        elif f["first_jw"] >= JW_CLOSE:
            fr = max(first_freq(fa), first_freq(fb))
            level["first"], w["first"] = "close", log2(M_FIRST["close"] / min(1.0, 5 * fr))
        else:
            level["first"], w["first"] = "differ", log2(M_FIRST["differ"])
    else:
        level["first"] = "missing"
    ma, mb = a["middle"], b["middle"]
    if ma and mb:
        if ma == mb:
            level["middle"], w["middle"] = "exact", log2(M_MIDDLE_AGREE / (1 / 15))
        else:
            level["middle"], w["middle"] = "differ", log2(1 - M_MIDDLE_AGREE)
    else:
        level["middle"] = "missing"
    sims = [f["last_jw"]] + ([f["first_jw"]] if fa and fb else [])
    return w, level, round(sum(sims) / len(sims), 3)

ORG_RARE_IDF = 10.0      # a word fewer than ~1 in 1,000 organizations use

def _unmatched_cost(t):
    """Bits lost when one name has a word the other lacks.

    Short forms drop generic words ("Nexray" for "Nexray Medical Imaging"), so a common
    unmatched word costs little. They do not drop distinctive ones: "Rutland Medical Plaza"
    against "Rutland Medical" is probably a different business, and a rare word on one side
    only is real evidence of that."""
    idf = org_idf(t)
    return 0.6 * idf if idf >= ORG_RARE_IDF else min(4.0, 0.3 * idf)

def _org_alias_bits(x, y):
    """Token-level agreement between two organization names, weighted by NPPES rarity.

    Tokens pair by exact match first, then Jaro-Winkler >= JW_CLOSE ('IMANGING' finds
    'IMAGING'). The rarest shared word counts in full and the others at half, because the
    words of one name are not independent ('MEDICAL' and 'IMAGING' travel together).
    A word only one side has costs a little, scaled by its rarity."""
    x, y = _rejoin(x, y), _rejoin(y, x)
    left, matched = list(y), []
    for t in x:
        hit = t if t in left else max(left, key=lambda u: _jw(t, u), default=None)
        if hit is not None and (hit == t or _jw(t, hit) >= JW_CLOSE):
            matched.append((t, hit)); left.remove(hit)
    only_x = [t for t in x if t not in [m[0] for m in matched]]
    only = only_x + left
    if not matched:
        return ORG_NO_OVERLAP_BITS, 0.0, [], only_x, left
    idfs = sorted((min(org_idf(t), org_idf(u)) for t, u in matched), reverse=True)
    bits = idfs[0] + 0.5 * sum(idfs[1:]) - sum(_unmatched_cost(t) for t in only)
    union = sum(idfs) + sum(org_idf(t) for t in only)
    return bits, round(sum(idfs) / union, 3) if union else 0.0, [m[0] for m in matched], only_x, left

def _rejoin(x, y):
    """Undo OCR splits: 'CASUAL','TY' -> 'CASUALTY' when the other name has 'CASUALTY'."""
    ys, out, i = set(y), [], 0
    while i < len(x):
        if i + 1 < len(x) and x[i] + x[i + 1] in ys and x[i] not in ys:
            out.append(x[i] + x[i + 1]); i += 2
        else:
            out.append(x[i]); i += 1
    return out

def _distinctive(tokens):
    return [t for t in tokens if len(t) >= 4 and org_idf(t) >= ORG_RARE_IDF]

def declared_dbas(recs):
    """Name pairs that some mention declares to be one organization ('X d/b/a Y').

    A trade name shares no words with the legal name ('Soul Radiology Medical Imaging' is
    Nexray's), so without this the sibling veto below would split a party from its own
    d/b/a."""
    pairs = []
    for r in recs:
        al = r.get("aliases") or []
        pairs += [(set(al[i]), set(al[j])) for i in range(len(al)) for j in range(i + 1, len(al))]
    return pairs

DBA_PAIRS = []          # set by link_mentions for the corpus being linked

def _fuzzy_family(a, b):
    """same_name_family with spelling tolerance: 'MEDICA' (an OCR cut) counts as 'MEDICAL'."""
    small, big = (a, b) if len(a) <= len(b) else (b, a)
    return bool(small) and all(any(t == u or _jw(t, u) >= JW_CLOSE for u in big) for t in small)

def _declared_same(x, y):
    x, y = set(x), set(y)
    return any((_fuzzy_family(x, p) and _fuzzy_family(y, q)) or
               (_fuzzy_family(x, q) and _fuzzy_family(y, p)) for p, q in DBA_PAIRS)

def org_name_bits(a, b):
    """Best alias pair. Also returns a veto when each name keeps a distinctive word the
    other lacks: 'Allstate Indemnity Company' and 'Allstate Property & Casualty Insurance
    Company' are sibling companies, not one party under two names. Never between two
    names the corpus declares as one organization's d/b/a."""
    best = None
    for x in a.get("aliases") or []:
        for y in b.get("aliases") or []:
            got = _org_alias_bits(x, y)
            if best is None or got[0] > best[0]:
                best = got
    best = best or (ORG_NO_OVERLAP_BITS, 0.0, [], [], [])
    bits, sim, shared, only_a, only_b = best
    da, db = _distinctive(only_a), _distinctive(only_b)
    veto = None
    if shared and da and db and not any(_declared_same(x, y) for x in a.get("aliases") or []
                                        for y in b.get("aliases") or []):
        veto = f"distinct_org_names:{'+'.join(da)} vs {'+'.join(db)}"
    return ({"org_name": bits}, {"org_name": ("shared:" + ",".join(shared)) if shared else "differ"},
            sim, veto)

def identifier_bits(a, b):
    w, veto = {}, None
    va, vb = defaultdict(dict), defaultdict(dict)
    for t, v, basis in a["details"]: va[t][v] = basis
    for t, v, basis in b["details"]: vb[t][v] = basis
    for t in set(va) & set(vb):
        if t == "dob":
            continue                               # below: evidence, never an identifier
        # a plate or license number agrees when the numbers match and the issuers do not
        # contradict each other (one may be unstated)
        shared, seen = [], set()
        for x in sorted(va[t], key=len, reverse=True):
            for y in sorted(vb[t], key=len, reverse=True):
                if qualifiers_agree(t, x, y) and id_number(t, x) not in seen:
                    seen.add(id_number(t, x)); shared.append((x, y))
        for x, y in shared:
            weakest = "inferred" if "inferred" in (va[t][x], vb[t][y]) else "stated"
            v = x if len(x) >= len(y) else y       # the more specific form, with its issuer
            w[f"{t}:{v}"] = log2(ID_M[weakest] / ID_U.get(t, 1e-3))
        if t in SINGULAR_TYPES and not shared:
            both_stated = all(x == "stated" for x in list(va[t].values()) + list(vb[t].values()))
            if both_stated:
                veto = f"conflicting_{t}:{sorted(va[t])}/{sorted(vb[t])}"
            else:
                w[f"{t}:conflict"] = -8.0
    # Date of birth, persons only. Agreement adds weight; a difference counts against
    # (typos and transposed days happen), but never vetoes.
    if a["type"] == "person" and va.get("dob") and vb.get("dob"):
        shared = sorted(set(va["dob"]) & set(vb["dob"]))
        if shared:
            v = shared[0]
            weakest = "inferred" if "inferred" in (va["dob"][v], vb["dob"][v]) else "stated"
            w[f"dob:{v}"] = log2(ID_M[weakest] / ID_U["dob"])
        else:
            bases = list(va["dob"].values()) + list(vb["dob"].values())
            weakest = "inferred" if "inferred" in bases else "stated"
            w["dob:differ"] = log2((1 - ID_M[weakest]) / (1 - ID_U["dob"]))
    return w, veto

def score_pair(a, b, f, proposed_by=()):
    """One link. `f` holds the recordlinkage comparison features for the pair."""
    dist = distance_of(a, b)
    prior, prior_key = prior_of(a, b, dist)
    veto = None
    if a["type"] == "person":
        w, level, sim = person_name_bits(a, b, f)
        la = set(a["creds"]) & PRIMARY_LICENSE
        lb = set(b["creds"]) & PRIMARY_LICENSE
        if la and lb and not (la & lb):
            veto = f"credential_conflict:{'/'.join(sorted(la))} vs {'/'.join(sorted(lb))}"
    elif a["type"] == "organization":
        w, level, sim, veto = org_name_bits(a, b)
    else:
        same = a["vehicle_key"] == b["vehicle_key"]
        w, level, sim = {"vehicle": 6.0 if same else -3.0}, {"vehicle": "exact" if same else "differ"}, float(same)
    idw, idveto = identifier_bits(a, b)
    w.update(idw)
    veto = veto or idveto
    if dist == "watchlist":
        loc = location_bits(a, b)
        if loc:
            w["location"] = loc[0]
            level["location"] = loc[1]
    return _finish({"a": a["key"], "b": b["key"], "type": a["type"],
                    "distance": dist, "prior": prior, "prior_key": prior_key,
                    "weights": {k: round(v, 2) for k, v in w.items()},
                    "name_agreement": level, "name_similarity": sim,
                    "features": {k: round(float(v), 3) for k, v in f.items()},
                    "proposed_by": list(proposed_by), "veto": veto,
                    "co_party": {"anchored": [], "overlap": None}})

def _finish(link):
    w = link["weights"]
    bits = sum(w.values())
    logit = log2(link["prior"] / (1 - link["prior"])) + bits
    link["bits"] = round(bits, 2)
    link["p"] = round(1 / (1 + 2 ** -logit), 4) if logit > -60 else 0.0
    positive = {k.split(":")[0] for k, v in w.items() if v > 0}
    names = {"last", "first", "middle", "org_name", "vehicle"}
    link["basis"] = sorted(({"name"} if positive & names else set()) | (positive - names))
    if positive & IDENTIFIER_BASIS:   link["basis_class"] = "identifier"
    elif "address" in positive:       link["basis_class"] = "address"
    elif "dob" in positive:           link["basis_class"] = "dob"
    elif "co_party" in positive:      link["basis_class"] = "co_party"
    elif "location" in positive and positive & names:
                                      link["basis_class"] = "location"
    elif positive & names:            link["basis_class"] = "name_only"
    else:                             link["basis_class"] = "none"
    link["suspicion"] = round(SUSPICION_WEIGHT[link["distance"]] * link["p"], 4)
    return link

# ---- the decision card: every number a link rests on, in the order a reader needs it --------
FIELD_LABEL = {"last": "surname", "first": "first name", "middle": "middle initial",
               "org_name": "organization name", "vehicle": "vehicle description",
               "location": "city and state", "co_party": "anchored co-party"}

def _org_tokens_explained(a, b):
    """The alias pair org_name_bits chose, with each word's rarity: shared words, and the
    words only one side has."""
    best = None
    for x in a.get("aliases") or []:
        for y in b.get("aliases") or []:
            got = _org_alias_bits(x, y)
            if best is None or got[0] > best[0]:
                best = (got[0], got[2], got[3], got[4], x, y)
    if best is None:
        return None
    _, shared, only_a, only_b, x, y = best
    tok = lambda t: {"t": t, "bits": round(org_idf(t), 2),
                     "orgs_using": (ORG_DF or {}).get(t, ORG_DF_FLOOR), "orgs_total": ORG_N}
    return {"a": " ".join(x), "b": " ".join(y), "shared": [tok(t) for t in shared],
            "only_a": [tok(t) for t in only_a], "only_b": [tok(t) for t in only_b]}

def explain_link(a, b, link):
    """link["fields"]: one row per compared field, with the values compared, the agreement
    level, m and u where they apply, what u rests on, and the bits the scorer actually
    added (read from link["weights"], so the rows always sum to link["bits"])."""
    w, lvl, rows = link["weights"], link["name_agreement"], []

    def row(field, **kw):
        rows.append({"field": field, "label": FIELD_LABEL.get(field, field),
                     "bits": w.get(field), **kw})

    if link["type"] == "person":
        la, lb = a.get("last"), b.get("last")
        if lvl.get("last") == "missing":
            row("last", a=la or None, b=lb or None, level="missing")
        else:
            fa_, fb_ = surname_freq(la), surname_freq(lb)
            fr = max(fa_, fb_)
            m, u = {"exact": (M_LAST["exact"], fr), "close": (M_LAST["close"], min(1.0, 5 * fr)),
                    "differ": (M_LAST["differ"], 1.0)}[lvl["last"]]
            row("last", a=la, b=lb, level=lvl["last"], m=m, u=u, freq_a=fa_, freq_b=fb_,
                u_source=RARITY_SOURCE["surnames"], jw=link["features"].get("last_jw"))
        fa, fb = a.get("first"), b.get("first")
        if lvl.get("first") == "missing":
            row("first", a=fa or None, b=fb or None, level="missing")
        elif lvl["first"] == "initial" or (lvl["first"] == "differ" and (len(fa) == 1 or len(fb) == 1)):
            share = initial_share(fa[0])
            m, u = (M_INITIAL_AGREE, share) if lvl["first"] == "initial" else (1 - M_INITIAL_AGREE, 1 - share)
            row("first", a=fa, b=fb, level=lvl["first"], m=m, u=u, initial_share=share,
                u_source=RARITY_SOURCE["first_names"])
        else:
            f1, f2 = first_freq(fa), first_freq(fb)
            m, u = {"exact": (M_FIRST["exact"], f1), "close": (M_FIRST["close"], min(1.0, 5 * max(f1, f2))),
                    "differ": (M_FIRST["differ"], 1.0)}[lvl["first"]]
            row("first", a=fa, b=fb, level=lvl["first"], m=m, u=u, freq_a=f1, freq_b=f2,
                u_source=RARITY_SOURCE["first_names"], jw=link["features"].get("first_jw"))
        ma, mb = a.get("middle"), b.get("middle")
        if lvl.get("middle") == "missing":
            row("middle", a=ma or None, b=mb or None, level="missing")
        else:
            row("middle", a=ma, b=mb, level=lvl["middle"],
                m=M_MIDDLE_AGREE if lvl["middle"] == "exact" else 1 - M_MIDDLE_AGREE,
                u=1 / 15 if lvl["middle"] == "exact" else 1.0, u_source="assumed: 15 common initials")
    elif link["type"] == "organization":
        ex = _org_tokens_explained(a, b)
        row("org_name", level="shared" if ex and ex["shared"] else "differ",
            **(ex or {"a": a["name"], "b": b["name"], "shared": [], "only_a": [], "only_b": []}),
            u_source=RARITY_SOURCE["org_tokens"])
    else:
        row("vehicle", a=a.get("vehicle_key"), b=b.get("vehicle_key"), level=lvl.get("vehicle"),
            u_source="assumed")

    # identifiers and dates of birth, both sides' values, scored or not
    va, vb = defaultdict(dict), defaultdict(dict)
    for t, v, basis in a["details"]: va[t][v] = basis
    for t, v, basis in b["details"]: vb[t][v] = basis
    scored = set()
    for k, bits in w.items():
        t, _, v = k.partition(":")
        if t not in DETAIL_TYPES:
            continue
        scored.add(t)
        if v in ("conflict", "differ"):
            rows.append({"field": "dob" if t == "dob" else "identifier", "type": t,
                         "label": t.replace("_", " "), "level": "differ" if t == "dob" else "conflict",
                         "a": sorted(va[t]), "b": sorted(vb[t]), "bits": bits,
                         "note": ("a date of birth that differs counts against, never vetoes" if t == "dob"
                                  else "one side's owner was inferred, so the conflict counts against "
                                       "instead of vetoing")})
        else:
            u = ID_U.get(t, 1e-3)
            rows.append({"field": "dob" if t == "dob" else "identifier", "type": t,
                         "label": t.replace("_", " "), "level": "exact", "value": v,
                         "a": sorted(va[t]), "b": sorted(vb[t]), "m": round(u * 2 ** bits, 3), "u": u,
                         "u_source": "assumed", "bits": bits,
                         "identifier_basis": t in IDENTIFIER_BASIS})
    for t in sorted((set(va) & set(vb)) - scored):
        vetoing = t in SINGULAR_TYPES and (link.get("veto") or "").startswith(f"conflicting_{t}")
        rows.append({"field": "dob" if t == "dob" else "identifier", "type": t,
                     "label": t.replace("_", " "), "level": "conflict" if vetoing else "differ",
                     "a": sorted(va[t]), "b": sorted(vb[t]), "bits": None, "veto": vetoing,
                     "note": ("a party has only one: two stated values veto the pair" if vetoing
                              else "different values, not scored: a party can hold several")})
    for t in sorted(set(va) ^ set(vb)):
        rows.append({"field": "dob" if t == "dob" else "identifier", "type": t,
                     "label": t.replace("_", " "), "level": "missing",
                     "a": sorted(va.get(t, {})), "b": sorted(vb.get(t, {})), "bits": None})

    if link["distance"] == "watchlist":
        loc = location_bits(a, b)
        if loc:
            bits_, level_, x, y, u = loc
            row("location", a=f"{x['city'].title()}, {x['state']}".strip(", "),
                b=f"{y['city'].title()}, {y['state']}".strip(", "), level=level_,
                m=M_STATE, u=u, u_source="2020 Census state population share")
        else:
            fmt = lambda locs: (f"{locs[0]['city'].title()}, {locs[0]['state']}".strip(", ")
                                if locs else None)
            row("location", a=fmt(a.get("locations")), b=fmt(b.get("locations")), level="missing")
    if w.get("co_party"):
        row("co_party", level="anchored", pairs=link["co_party"]["anchored"],
            per_pair=CO_PARTY_BITS, max_pairs=CO_PARTY_MAX)
    link["fields"] = rows
    link["name_similarity_pct"] = round(100 * (link.get("name_similarity") or 0))
    return link

def link_mentions(recs, cands=None, feats=None):
    """Score every candidate pair, then add co-party evidence. Returns the link list."""
    if cands is None:
        cands, frame = candidate_pairs(recs)
        feats = compare_pairs(cands, frame)
    by_key = {r["key"]: r for r in recs}
    global DBA_PAIRS
    DBA_PAIRS = declared_dbas(recs)
    links = [score_pair(by_key[a], by_key[b], feats[(a, b)], c["proposed_by"])
             for (a, b), c in cands.items()]

    # Co-parties: the other mentions in the same claim.
    claim_of = {r["key"]: r["claim_id"] for r in recs}
    in_claim = defaultdict(set)
    for r in recs:
        if r.get("linkable", True) and name_tokens(r["name"], r["type"]):
            in_claim[r["claim_id"]].add(r["key"])
    adj = defaultdict(dict)
    for l in links:
        adj[l["a"]][l["b"]] = l; adj[l["b"]][l["a"]] = l
    # (A) anchors are decided on pass-1 scores and must rest on an identifier, so a
    # co-party boost can never create an anchor: evidence flows one way.
    anchors = {(l["a"], l["b"]) for l in links
               if l["basis_class"] == "identifier" and l["p"] >= 0.9 and not l["veto"]}
    anchor_adj = defaultdict(set)
    for x, y in anchors:
        anchor_adj[x].add(y); anchor_adj[y].add(x)
    for l in links:
        ca, cb = claim_of[l["a"]], claim_of[l["b"]]
        if ca == cb:
            continue
        co_a = in_claim[ca] - {l["a"]}
        found = sorted({tuple(sorted((x, y))) for x in co_a for y in anchor_adj[x]
                        if claim_of[y] == cb and y != l["b"]})
        name_bits = sum(v for k, v in l["weights"].items()
                        if k in ("last", "first", "middle", "org_name", "vehicle"))
        # co-parties strengthen a name that already agrees; they never make one
        if found and name_bits > 0:
            l["co_party"]["anchored"] = [list(p) for p in found]
            l["weights"]["co_party"] = CO_PARTY_BITS * min(CO_PARTY_MAX, len(found))
            _finish(l)
        # (B) displayed only: co-parties that also match by name, at any basis
        matched = sorted({x for x in co_a
                          if any(claim_of[y] == cb and y != l["b"] and not adj[x][y]["veto"]
                                 and adj[x][y]["p"] >= 0.5 for y in adj[x])})
        co_b = in_claim[cb] - {l["b"]}
        l["co_party"]["overlap"] = {
            "matched": len(matched), "of": min(len(co_a), len(co_b)),
            "names": [by_key[x]["name"] for x in matched[:6]]}
    for l in links:
        explain_link(by_key[l["a"]], by_key[l["b"]], l)
    links.sort(key=lambda l: (l["a"], l["b"]))
    return links

links = link_mentions(mentions, cands, features)

Path(OUT_DIR, "links.json").write_text(json.dumps(links, indent=1))
Path(OUT_DIR, "mentions.json").write_text(json.dumps(mentions, indent=1))

tally = defaultdict(int)
for l in links:
    tally[(l["distance"], "veto" if l["veto"] else l["basis_class"],
           "p>=.8" if l["p"] >= 0.8 else "p>=.1" if l["p"] >= 0.1 else "p<.1")] += 1
print(f"{len(links)} links scored  (rarity: {RARITY_SOURCE})")
for k in sorted(tally):
    print(f"  {k[0]:<17} {k[1]:<11} {k[2]:<6} {tally[k]}")

def show_link(l):
    a, b = mention_by_key[l["a"]], mention_by_key[l["b"]]
    print(f"  {a['name'][:28]!r:<30} <-> {b['name'][:28]!r:<30} p={l['p']:<6} "
          f"{l['basis_class']:<10} {l['distance']}"
          + (f"  VETO {l['veto']}" if l["veto"] else ""))
    print(f"      prior={l['prior_key']} bits={l['bits']} {l['weights']}")
    ov = l["co_party"]["overlap"]
    if ov and ov["matched"]:
        print(f"      co-parties also matching by name: {ov['matched']} of {ov['of']} {ov['names']}")
    if l["co_party"]["anchored"]:
        print(f"      anchored co-parties: {l['co_party']['anchored']}")

cross = sorted((l for l in links if l["distance"] in ("same_client", "different_client")),
               key=lambda l: -l["p"])
print(f"\ncross-claim links: {len(cross)} (top 15)")
for l in cross[:15]: show_link(l)
vetoes = [l for l in links if l["veto"]]
print(f"\nvetoes: {len(vetoes)}")
for l in vetoes[:10]: show_link(l)


## 18b — Learning the weights (disabled)

The m-values and priors in cell 18 are assumptions. Fellegi-Sunter's other half learns them
from the data with expectation-maximization; `recordlinkage.ECMClassifier` does this on
binary comparison vectors.

**Enable this cell when all of these hold:**
- **Enough pairs.** At least a few thousand candidate pairs with a real share of true
  matches. EM on the court corpus's few hundred pairs fits noise.
- **Gold to check against.** The docket party lists (Phase 3) or an SME-annotated set exist,
  so the learned weights can be shown to beat the fixed ones, not just differ from them.
- **Independence acknowledged.** EM assumes fields are conditionally independent, which is
  the same simplification the fixed weights make. Correlated fields (first and last name)
  will pull the learned m-values; compare them against the fixed ones before swapping.

**Requirements:** `recordlinkage` (already needed by cell 17); nothing else.

What it would change: only `M_LAST`, `M_FIRST`, `M_MIDDLE_AGREE` and the priors. The
per-value rarity (`u` from the outside tables) stays ours, because `ECMClassifier` learns one
`u` per field, not per name.


In [ ]:
# --- DISABLED: enable per the criteria above ------------------------------------------
# import numpy as np
# vec = pd.DataFrame({
#     "last_exact":  [int(l["name_agreement"].get("last") == "exact") for l in links],
#     "first_agree": [int(l["name_agreement"].get("first") in ("exact", "initial")) for l in links],
#     "middle_agree":[int(l["name_agreement"].get("middle") == "exact") for l in links],
#     "id_shared":   [int(l["basis_class"] == "identifier") for l in links],
# }, index=pd.MultiIndex.from_tuples([(l["a"], l["b"]) for l in links]))
# ecm = rl.ECMClassifier(binarize=None)
# ecm.fit(vec)
# print("learned m:", dict(zip(vec.columns, np.round(ecm.m_probs.values(), 3)))
#       if hasattr(ecm, "m_probs") else ecm.log_m_probs)
# print("learned u:", ecm.u_probs if hasattr(ecm, "u_probs") else ecm.log_u_probs)
# print("learned match share (prior):", ecm.p)
print("cell 18b: EM weight learning is disabled — see the criteria above.")


## 18c — Watchlist: OIG LEIE, "Flagged for review"

The federal health-care exclusion list (OIG LEIE, ~84,000 records) is linked against the
corpus with **the same machinery** as mention against mention: every record becomes a
mention record (`source: "oig_leie"`, no claim), `recordlinkage` proposes the pairs, and the
cell 18 scorer scores them with the same name fields, rarity tables, identifiers and vetoes.
What differs is declared, not hidden:

| | Between corpus mentions | Corpus mention against a watchlist record |
|---|---|---|
| Candidate pairs | every block | only corpus ↔ watchlist; two records are never paired |
| Distance | note / claim / occurrence / client | `watchlist` |
| Prior | by distance and role | by the corpus party's role: 5e-7 professional or organization, 1e-7 private (cell 18 states what these mean) |
| Location | not compared | the mention's own addresses against the record's city and state: same state adds `log2(0.8 / state's population share)`, same city more; a different state counts against (`log2(0.2)`), never vetoes |
| Identifier | as cell 18 | the record's NPI, when it has one (11% do). Agreement makes the link identifier-backed; a different stated NPI vetoes |
| Co-parties | scored when anchored | not applicable |

Blocking keeps 84,000 records cheap: persons block on the NYSIIS code of the surname,
organizations on their rarest name word and on their leading word, identifiers on the NPI,
and a record becomes a full mention record only once a block proposes it.

An entity is **Flagged for review** at a lens when any of its mentions has a watchlist link
that lens admits (`goko/watchlist.py`, which the app calls too). A flag is a lead for a person
to check, not a finding. Most rest on a name alone, and the basis travels with the flag.
Links below `WATCHLIST_KEEP_P` are not written; everything above it is, with the full
per-field breakdown, to `watchlist_links.json`.


In [ ]:
sys.path.insert(0, str(Path.cwd()))          # goko/ sits next to this notebook
from goko.watchlist import flags_for, near_flags
from goko.projection import LENSES as _LENSES, admits as _admits
import time as _time

WATCHLIST_FILE = "./corpus/registry/leie_records.csv.gz"
WATCHLIST_ENABLED = True
WATCHLIST_KEEP_P = 1e-3        # links below this are not written (still counted)

# LEIE "general" categories that are licensed professionals. Everyone else on the list who is
# a person (employees, owners, private citizens) is private. Informational: the prior reads
# the corpus side's role, not the record's.
LEIE_PROFESSIONAL = {"IND- LIC HC SERV PRO", "NURSING PROFESSION", "PHYSICIAN (MD, DO)",
                     "MEDICAL PRACTICE, MD", "CHIROPRACTIC PRACT", "DENTAL PRACTICE", "THERAPIST",
                     "PSYCHOLOGIC PRACTICE", "PODIATRY PRACTICE", "OSTEOPATHIC PRAC"}

def load_watchlist(path=WATCHLIST_FILE):
    """The watchlist table with a blocking key per record: the surname as parse_person reads
    the full name (so both sides of a person block agree on what 'surname' means) and its
    NYSIIS code; the organization's rarest and leading name words."""
    wl = pd.read_csv(path, dtype=str, keep_default_na=False)
    person = wl["last"] != ""
    names = [" ".join(x for x in (f, m, l) if x) for f, m, l in
             zip(wl["first"], wl["middle"], wl["last"])]
    wl["_name"] = [n if p else b for n, p, b in zip(names, person, wl["business"])]
    wl["_last"] = [parse_person(n)["last"] if p else "" for n, p in zip(wl["_name"], person)]
    wl["_last_ph"] = None
    has = wl["_last"] != ""
    wl.loc[has, "_last_ph"] = phonetic(wl.loc[has, "_last"].str.lower(), method="nysiis").values
    al = [org_aliases(b)[0] if (not p and b and org_aliases(b)) else [] for b, p in zip(wl["business"], person)]
    wl["_org_rare"] = [min(a, key=lambda t: (-org_idf(t), t)) if a else None for a in al]
    wl["_org_lead"] = [a[0] if a else None for a in al]
    return wl

def leie_mention(row):
    """One watchlist record as a mention record: build_mention, so it is parsed exactly like
    a corpus mention, plus the source, its city and state, and what the list says."""
    etype = "person" if row["last"] else "organization"
    details = [("npi", row["npi"], "stated")] if row["npi"] else []
    r = build_mention(row["record_id"], etype, row["_name"], claim_id="", note_key=None,
                      details=details)
    r.update(claim_id=None, source="oig_leie",
             locations=[{"city": row["city"].upper(), "state": row["state"].upper()}]
                       if row["state"] else [],
             watchlist={k: row[k] for k in ("general", "specialty", "excl_type", "excl_date",
                                             "npi", "city", "state")})
    if etype == "person":
        r["role_class"] = ("professional" if row["npi"] or row["general"] in LEIE_PROFESSIONAL
                           else "private")
    return r

def watchlist_candidates(recs, wl):
    """{(corpus_key, record_id): {"proposed_by": [...]}}. Every pair crosses corpus and
    watchlist: the indexes run in recordlinkage's two-frame linking mode, so two watchlist
    records (or two corpus mentions) can never be proposed here."""
    recs = [r for r in recs if r.get("linkable", True) and r.get("source") != "oig_leie"]
    by_key = {r["key"]: r for r in recs}
    found = defaultdict(set)

    def add(index, label):
        for a, b in index:
            found[(a, b)].add(label)

    cdf = _frame(recs)
    people = cdf[(cdf["type"] == "person") & cdf["last_ph"].notna()][["last_ph"]]
    wp = wl[wl["_last_ph"].notna()]
    if len(people) and len(wp):
        add(rl.Index().block("last_ph").index(
            people, pd.DataFrame({"last_ph": wp["_last_ph"].values}, index=wp["record_id"].values)),
            "block:surname_nysiis")
    orgs = cdf[(cdf["type"] == "organization") & cdf["org"].notna()].copy()
    wo = wl[wl["_org_rare"].notna()]
    if len(orgs) and len(wo):
        orgs["org_lead"] = [by_key[k]["aliases"][0][0] for k in orgs.index]
        wof = pd.DataFrame({"org_rare": wo["_org_rare"].values, "org_lead": wo["_org_lead"].values},
                           index=wo["record_id"].values)
        add(rl.Index().block("org_rare").index(orgs[["org_rare"]], wof[["org_rare"]]),
            "block:org_rarest_word")
        add(rl.Index().block("org_lead").index(orgs[["org_lead"]], wof[["org_lead"]]),
            "block:org_leading_word")
    npis = defaultdict(list)
    for rid, npi in zip(wl["record_id"], wl["npi"]):
        if npi:
            npis[npi].append(rid)
    for r in recs:
        for t, v, _ in r["details"]:
            if t == "npi":
                add([(r["key"], x) for x in npis.get(v, [])], "shared:npi")
    return {k: {"proposed_by": sorted(v)} for k, v in found.items()}

def link_watchlist(recs, wl):
    """Score every corpus-watchlist candidate with the cell 18 scorer. Returns (links kept,
    the watchlist records they point at, the number of candidates scored)."""
    cands = watchlist_candidates(recs, wl)
    by_key = {r["key"]: r for r in recs}
    ids = sorted({b for _, b in cands})
    rows = wl.set_index("record_id", drop=False).loc[ids] if ids else wl.iloc[:0]
    wrecs = {row["record_id"]: leie_mention(row) for _, row in rows.iterrows()}
    # a person pair needs a person record, an organization pair an organization one
    cands = {(a, b): c for (a, b), c in cands.items() if by_key[a]["type"] == wrecs[b]["type"]}
    frame = _frame([by_key[a] for a in sorted({a for a, _ in cands})] + list(wrecs.values()))
    feats = compare_pairs(cands, frame)
    kept = []
    for (a, b), c in cands.items():
        l = score_pair(by_key[a], wrecs[b], feats[(a, b)], c["proposed_by"])
        l["source"] = "oig_leie"
        if l["p"] >= WATCHLIST_KEEP_P:
            kept.append(explain_link(by_key[a], wrecs[b], l))
    kept.sort(key=lambda l: (-l["p"], l["a"], l["b"]))
    return kept, {l["b"]: wrecs[l["b"]] for l in kept}, len(cands)

watchlist_links, watchlist_records, watchlist_scored, watchlist_size = [], {}, 0, 0
if WATCHLIST_ENABLED and Path(WATCHLIST_FILE).exists():
    _t0 = _time.time()
    _wl = load_watchlist()
    watchlist_size = len(_wl)
    watchlist_links, watchlist_records, watchlist_scored = link_watchlist(mentions, _wl)
    print(f"watchlist: {watchlist_size:,} OIG LEIE records, {watchlist_scored} candidate pairs "
          f"scored, {len(watchlist_links)} kept (p >= {WATCHLIST_KEEP_P}) in {_time.time() - _t0:.1f}s")
else:
    print("!! watchlist disabled or corpus/registry/leie_records.csv.gz missing: run "
          "corpus/registry/fetch_registry.py. No entity can be flagged for review in this run.")

watchlist_by_mention = defaultdict(list)
for l in watchlist_links:
    watchlist_by_mention[l["a"]].append(l)

Path(OUT_DIR, "watchlist_links.json").write_text(json.dumps({
    "source": "OIG LEIE (List of Excluded Individuals/Entities)",
    "table": WATCHLIST_FILE, "records_on_list": watchlist_size,
    "candidates_scored": watchlist_scored, "keep_p": WATCHLIST_KEEP_P,
    "priors": {"professional_or_organization": PRIOR_P[("watchlist", "recurring")],
               "private": PRIOR_P[("watchlist", "private")]},
    "records": {rid: {"record_id": rid, "name": r["name"], "type": r["type"],
                      "role_class": r["role_class"], **r["watchlist"]}
                for rid, r in watchlist_records.items()},
    "links": watchlist_links}, indent=1))

for lens in _LENSES:
    hit = {l["a"] for l in watchlist_links if _admits(l, lens)}
    print(f"  mentions with a watchlist link admitted at {lens:<8}: {len(hit)}")
for l in watchlist_links[:12]:
    rec = watchlist_records[l["b"]]
    print(f"  {mention_by_key[l['a']]['name'][:30]!r:<32} ~ {rec['name'][:30]!r:<32} p={l['p']:<7} "
          f"{l['basis_class']:<10} {rec['watchlist']['city']}, {rec['watchlist']['state']}"
          f"  excl {rec['watchlist']['excl_type']} {rec['watchlist']['excl_date']}"
          + (f"  VETO {l['veto']}" if l["veto"] else ""))
    print(f"      prior={l['prior_key']} bits={l['bits']} {l['weights']}")


## 19 — Read-time projection

The merged view, computed from the links at a lens — here `DOSSIER_LENS` for the dossier,
while all three lenses are computed for comparison. `goko/projection.py` does the grouping
so the search app reads exactly the same thing.

A claim's entities are the global clusters restricted to that claim. Each entity carries its
members, the link that pulled each member in, the **weakest link** on the path connecting
them (its confidence), whether any of those links is name-only, and the stronger-than-broad
alternatives this lens left out (`uncertain_merge`).


In [ ]:
sys.path.insert(0, str(Path.cwd()))          # goko/ sits next to this notebook
from goko.projection import LENSES, MAX_CLUSTER, admits, project, subtree_weakest

DOSSIER_LENS = "default"

all_keys = [r["key"] for r in mentions]
projections = {lens: project(all_keys, links, lens) for lens in LENSES}
for lens, pr in projections.items():
    multi = [c for c in pr["clusters"] if len(c["members"]) > 1]
    spans = [c for c in multi if len({mention_by_key[k]["claim_id"] for k in c["members"]}) > 1]
    print(f"  {lens:<8} {len(pr['clusters'])} clusters, {len(multi)} with >1 mention, "
          f"{len(spans)} across claims, {len(pr['refused'])} refused unions")

def _link_ref(l, me=None):
    if l is None: return None
    other = l["b"] if me == l["a"] else l["a"] if me else None
    return {"a": l["a"], "b": l["b"], **({"with": other} if other else {}),
            "p": l["p"], "basis_class": l["basis_class"], "distance": l["distance"]}

proj = projections[DOSSIER_LENS]
id_maps, entity_sets, blocked_merges = {}, {}, {}
for claim in claims:
    ents, idmap = [], {}
    pieces = []
    for c in proj["clusters"]:
        ms = [k for k in c["members"] if mention_by_key[k]["claim_id"] == claim]
        if ms: pieces.append((c, ms))
    pieces.sort(key=lambda x: x[1][0])
    for i, (c, ms) in enumerate(pieces, start=1):
        eid = f"{claim}:E{i}"
        for k in ms: idmap[k] = eid
        recs = [mention_by_key[k] for k in ms]
        weakest = subtree_weakest(c, ms)
        used = [c["joined_by"][k] for k in ms if c["joined_by"].get(k)]
        outside = set(c["members"]) - set(ms)
        alts = [l for l in links
                if ((l["a"] in ms) != (l["b"] in ms)) and not l["veto"]
                and (l["a"] not in c["members"] or l["b"] not in c["members"])
                and l["p"] >= LENSES["broad"]["min_p"]]
        flags = []
        # Flagged for review: any mention of the whole entity (all claims) has a watchlist
        # link this lens admits. goko/watchlist.py; the app applies the same rule.
        wflags = flags_for(c["members"], watchlist_by_mention, DOSSIER_LENS)
        if wflags: flags.append("flagged_for_review")
        if any(l["basis_class"] == "name_only" for l in used): flags.append("name_only_merge")
        if alts: flags.append("uncertain_merge")
        if any(r["reason"] == "cluster_size" and {r["link"]["a"], r["link"]["b"]} & set(ms)
               for r in proj["refused"]):
            flags.append("cluster_size_alarm")
        ents.append({
            "entity_id": eid, "type": recs[0]["type"], "cluster_id": c["id"],
            "surface_forms": sorted({f for r in recs for f in r["forms"]}),
            "role_class": sorted({r["role_class"] for r in recs}),
            "confidence": weakest["p"] if weakest else None,
            "merge_provenance": {
                "lens": DOSSIER_LENS, "merged_mentions": len(ms), "members": ms,
                "joined_by": {k: _link_ref(c["joined_by"].get(k), k) for k in ms},
                "weakest_link": _link_ref(weakest),
                "also_in_other_claims": sorted({mention_by_key[k]["claim_id"] for k in outside}),
                "uncertain_merges": [_link_ref(l) for l in sorted(alts, key=lambda l: -l["p"])[:8]]},
            "watchlist": [{"record_id": l["b"], "mention": l["a"], "p": l["p"],
                           "basis_class": l["basis_class"],
                           "record": {k: watchlist_records[l["b"]][k] for k in ("name", "type")}
                                     | watchlist_records[l["b"]]["watchlist"]}
                          for l in wflags],
            "projection_flags": flags})
    id_maps[claim], entity_sets[claim] = idmap, ents
    blocked_merges[claim] = [
        {"edge": [r["link"]["a"], r["link"]["b"]], "reason": r["reason"],
         **({"would_merge": r["would_merge"]} if "would_merge" in r else {})}
        for r in proj["refused"] if {r["link"]["a"], r["link"]["b"]} & set(idmap)]
    print(f"\n{claim}: {len(idmap)} mentions -> {len(ents)} entities at lens '{DOSSIER_LENS}'")
    for e in ents:
        mp = e["merge_provenance"]
        if mp["merged_mentions"] > 1 or e["projection_flags"]:
            wl = mp["weakest_link"]
            print(f"  {e['entity_id']:<28} {mp['merged_mentions']} mentions  "
                  f"weakest={wl['p'] if wl else '-'} ({wl['basis_class'] if wl else '-'})  "
                  f"{e['surface_forms'][:3]} {e['projection_flags']}")


## 19b — Remap

The substitution where cross-note relationships materialize. Nothing detected them; identity
resolution did all of it.

The cell rebuilds each entity's details from the envelopes every time it runs rather than
appending to whatever was there. Appending makes the cell non-idempotent: a second run
doubles every detail, a third triples it, and nothing in the output says so — the dossier
just reports Dr. Monroe holding the same NPI three times.

The same value asserted in two notes is one detail with two pieces of evidence, not two
details, so identical `(type, value)` pairs are folded with their evidence kept.


In [ ]:
claim_unassigned, claim_actions, dangling_participants = {}, {}, {}

BASIS_RANK = {"stated": 2, "inferred": 1}

for claim in claims:
    idmap = id_maps[claim]
    ents = {e["entity_id"]: e for e in entity_sets[claim]}
    for e in ents.values():
        e["details"] = []          # rebuilt, not appended to — this cell is re-runnable
        e["flags"]   = list(e["projection_flags"])
    unassigned, actions, dangling = [], [], []

    for env in envelopes:
        if env["claim_id"] != claim: continue
        nk = env["note_key"]
        for d in env["detail_mentions"]:
            owner = idmap.get(f"{nk}:{d['owner_ref']}")
            rec = {k: d[k] for k in ("detail_type", "raw_value", "normalized", "basis",
                                     "detected_by", "checksum", "clean_span")}
            rec["evidence"] = [{"note_key": nk, "clean_span": d["clean_span"],
                                "raw_value": d["raw_value"]}]
            if owner:
                rec["owner_status"] = "assigned"
                ents[owner]["details"].append(rec)
            else:
                rec["owner_status"] = "unassigned"
                rec["note_key"] = nk
                unassigned.append(rec)
        for a in env["action_mentions"]:
            parts = []
            for p in a["participants"]:
                eid = idmap.get(f"{nk}:{p['mention_id']}")
                if eid is None:
                    dangling.append({"action": f"{nk}:{a['action_id']}",
                                     "mention_id": p["mention_id"]})
                parts.append({"entity_id": eid, "role": p["role"]})
            print(f"  {nk}:{a['action_id']}  {a['action_type']}")
            for p, q in zip(a["participants"], parts):
                mark = "" if q["entity_id"] else "   !! DANGLING"
                print(f"      {p['mention_id']} -> {q['entity_id']}  as {q['role']}{mark}")
            actions.append({"action_id": f"{nk}:{a['action_id']}",
                            "action_type": a["action_type"], "participants": parts,
                            "stance": a["stance"], "time_qualifier": a["time_qualifier"],
                            "quote": a["quote"], "raw_span": a.get("raw_span")})

    # fold identical (type, value) details; two notes saying the same thing is evidence,
    # not a second detail
    for e in ents.values():
        folded = {}
        for d in e["details"]:
            k = (d["detail_type"], d["normalized"])
            if k in folded:
                f = folded[k]
                f["detected_by"] = sorted(set(f["detected_by"]) | set(d["detected_by"]))
                f["evidence"].extend(d["evidence"])
                if BASIS_RANK.get(d["basis"], 0) > BASIS_RANK.get(f["basis"], 0):
                    f["basis"] = d["basis"]
                if f["checksum"] == "n/a":
                    f["checksum"] = d["checksum"]
            else:
                folded[k] = dict(d)
        e["details"] = list(folded.values())

    # contradictions on singular detail types
    for e in ents.values():
        for dt in SINGULAR_TYPES:
            vals = {d["normalized"] for d in e["details"] if d["detail_type"] == dt}
            if len(vals) > 1:
                e["flags"].append("contradictory_singular_detail")
                print(f"  !! {e['entity_id']} has {len(vals)} distinct {dt}: {vals}")

    entity_sets[claim] = list(ents.values())
    claim_unassigned[claim] = unassigned
    claim_actions[claim] = actions
    dangling_participants[claim] = dangling
    print(f"\n{claim}: {len(ents)} entities, {len(unassigned)} unassigned details, "
          f"{len(actions)} actions, {len(dangling)} dangling participant(s)")


## 20 — Category assignment

Six frozen values carry the score. `subcategory` is open and unscored — mined later for
promotion. The rule pre-pass removes the cases a model should never be asked about.

`insufficient_evidence` is in the *schema* enum even though it is not one of the six scored
categories. Constraining generation to the six and then treating "insufficient_evidence" as
a possible answer is a contradiction: the model cannot return a value the decoder forbids,
so every ambiguous party gets one of the six anyway, and the honest answer only exists as a
verdict the validator imposes after the fact. Giving the model the option is what makes the
refusal measurable.

The off-taxonomy fault is raised out of the loop rather than caught by it. An off-enum
return means the constraint was not in force — version skew, or an unconstrained fallback
path — and collapsing it into `insufficient_evidence` makes a pipeline bug look like a
recall problem in the evaluation.


In [ ]:
TAXONOMY_VERSION = "3"
CATEGORIES = ["medical", "legal", "repair_shop", "witness", "financier", "other"]
# what the model may return: the six scored values plus an explicit refusal
CATEGORY_ENUM = CATEGORIES + ["insufficient_evidence"]

def rule_prepass(entity):
    dts = {d["detail_type"] for d in entity["details"]}
    if entity["type"] == "person" and "npi" in dts:
        return "medical", "npi_present_on_person"
    if entity["type"] == "person" and "dea_number" in dts:
        return "medical", "dea_number_present_on_person"
    if "bar_number" in dts:
        return "legal", "bar_number_present"
    if entity["type"] == "vehicle":
        return "other", "vehicle_is_not_a_party"
    return None, None

CAT_SCHEMA = {
  "type": "object", "additionalProperties": False,
  "required": ["category", "subcategory", "basis", "evidence_ids"],
  "properties": {
    "category": {"type": "string", "enum": CATEGORY_ENUM,
      "description": "Choose insufficient_evidence rather than guessing."},
    "subcategory": {"type": ["string", "null"],
      "description": "Free text, lowercase_with_underscores. Unscored."},
    "basis": {"type": "string", "enum": ["stated", "inferred"]},
    "evidence_ids": {"type": "array", "items": {"type": "string"},
      "description": "Only ids present in the packet you were given."},
  }}
lint_strict_schema(CAT_SCHEMA)

CAT_SYSTEM = (
    "Assign a category to this claim party. Choose only from the enum. If the evidence "
    "does not support any of the six categories, choose 'other'. If the evidence does not "
    "support a decision at all, choose 'insufficient_evidence' — that is a real answer, "
    "not a failure. Cite only evidence_ids you were given.")

class PipelineFault(RuntimeError):
    """A bug in the pipeline, not a finding about the data."""

def categorize_llm(packet):
    kwargs = dict(model=DEPLOYMENT, temperature=TEMPERATURE, max_tokens=512,
                  messages=[{"role": "system", "content": CAT_SYSTEM},
                            {"role": "user", "content": json.dumps(packet, indent=2)}])
    if USE_STRUCTURED:
        kwargs["response_format"] = {"type": "json_schema",
                                     "json_schema": {"name": "category_v3", "strict": True,
                                                     "schema": CAT_SCHEMA}}
    else:
        kwargs["response_format"] = {"type": "json_object"}
        kwargs["messages"][0]["content"] += (
            "\n\nReturn ONLY JSON matching this schema:\n" + json.dumps(CAT_SCHEMA))
    r = client.chat.completions.create(**kwargs)
    if getattr(r.choices[0], "finish_reason", None) == "length":
        raise RuntimeError("category response truncated at max_tokens")
    return json.loads(r.choices[0].message.content)

pipeline_faults = []

for claim in claims:
    print(f"\n--- categorizing {claim} ---")
    acts = claim_actions[claim]
    for e in entity_sets[claim]:
        cat, rule = rule_prepass(e)
        if cat:
            e["category"] = {"value": cat, "basis": "rule", "rule_id": rule,
                             "taxonomy_version": TAXONOMY_VERSION}
            e["subcategory"] = None
            print(f"  {e['entity_id']:<28} RULE  {rule} -> {cat}")
            continue
        mine = [a for a in acts
                if any(p["entity_id"] == e["entity_id"] for p in a["participants"])]
        ev = [{"evidence_id": a["action_id"], "quote": a["quote"]} for a in mine]
        packet = {"entity_id": e["entity_id"], "type": e["type"],
                  "surface_forms": e["surface_forms"],
                  "details": [{"detail_type": d["detail_type"],
                               "raw_value": d["raw_value"]} for d in e["details"]],
                  "actions": [{"action_type": a["action_type"],
                               "role": next(p["role"] for p in a["participants"]
                                            if p["entity_id"] == e["entity_id"])}
                              for a in mine],
                  "evidence_quotes": ev, "taxonomy_version": TAXONOMY_VERSION}
        print(f"  {e['entity_id']:<28} LLM   packet={json.dumps(packet)[:120]}...")
        try:
            res = categorize_llm(packet)
        except Exception as ex:
            print(f"      !! category call failed: {type(ex).__name__}: {ex}")
            # A failed call is not an answer. Recording it as insufficient_evidence
            # would put an infrastructure error into the refusal count.
            e["category"] = {"value": "CATEGORY_CALL_FAILED", "basis": "error",
                             "taxonomy_version": TAXONOMY_VERSION,
                             "error": str(ex)}
            e["subcategory"] = None
            continue

        if res.get("category") not in CATEGORY_ENUM:
            # not a finding: the constraint was not in force
            fault = {"error": "off_taxonomy_value", "returned": res.get("category"),
                     "entity_id": e["entity_id"],
                     "taxonomy_version_in_call": TAXONOMY_VERSION}
            pipeline_faults.append(fault)
            e["category"] = {"value": "PIPELINE_FAULT", "basis": "error",
                             "taxonomy_version": TAXONOMY_VERSION, **fault}
            e["subcategory"] = None
            print(f"      !! PIPELINE FAULT {json.dumps(fault)}")
            continue

        valid_ev = {x["evidence_id"] for x in ev}
        unresolved = [x for x in res.get("evidence_ids", []) if x not in valid_ev]
        if unresolved:
            print(f"      !! unresolved evidence citation {unresolved} "
                  f"-> insufficient_evidence")
            e["category"] = {"value": "insufficient_evidence", "basis": "llm",
                             "taxonomy_version": TAXONOMY_VERSION,
                             "flag": "unresolved_evidence_citation",
                             "unresolved": unresolved}
            e["subcategory"] = None
        else:
            e["category"] = {"value": res["category"], "basis": res["basis"],
                             "taxonomy_version": TAXONOMY_VERSION,
                             "evidence_ids": res.get("evidence_ids", [])}
            e["subcategory"] = res.get("subcategory")
            print(f"      -> {res['category']}  sub={res.get('subcategory')}  "
                  f"basis={res['basis']}")

if pipeline_faults:
    raise PipelineFault(
        f"{len(pipeline_faults)} off-taxonomy return(s): {json.dumps(pipeline_faults)}\n"
        f"The enum constraint was not in force. Check the taxonomy version in the call "
        f"against the validator, and that USE_STRUCTURED took effect. This is a bug to "
        f"fix, not a data quality finding to record.")


## 21 — Claim dossier

The frozen, scoreable artifact. Nothing enters the graph until this exists, which keeps your
evaluation independent of graph state.


In [ ]:
dossiers = {}
for claim in claims:
    p = parse_claim_id(claim)
    d = {"claim_id": claim, "client_id": p["client_id"],
         "occurrence_id": p["occurrence_id"], "coverage_code": p["coverage_code"],
         "versions": {"schema": SCHEMA_VERSION, "taxonomy": TAXONOMY_VERSION,
                      "model": DEPLOYMENT, "patterns": PATTERNS_VERSION,
                      "clean_policy": CLEAN_POLICY,
                      "chunk_policy": sorted({n["chunk_policy"] for n in notes
                                              if n["claim_id"] == claim}),
                      "offline": OFFLINE_MODE},
         "watchlist": {"source": "OIG LEIE", "records_on_list": watchlist_size,
                       "rule": "flagged when any member mention has a watchlist link the "
                               "dossier lens admits (goko/watchlist.py)"},
         "identity": {"lens": DOSSIER_LENS,
                      "lenses": {k: {**v, "basis": sorted(v["basis"]) if v["basis"] else None}
                                 for k, v in LENSES.items()},
                      "rarity": RARITY_SOURCE, "max_cluster": MAX_CLUSTER},
         "entities": entity_sets[claim],
         "unassigned_details": claim_unassigned[claim],
         "actions": claim_actions[claim],
         "blocked_merges": blocked_merges[claim],
         "dangling_participants": dangling_participants[claim],
         "review_items": [r for e in envelopes if e["claim_id"] == claim
                          for r in e["review_items"]]}
    dossiers[claim] = d
    Path(OUT_DIR, f"dossier_{claim}.json").write_text(json.dumps(d, indent=2))
    print(f"\n=== {claim} ===")
    print(f"  entities={len(d['entities'])} unassigned={len(d['unassigned_details'])} "
          f"actions={len(d['actions'])} review={len(d['review_items'])}")
    for e in d["entities"]:
        conf = e["confidence"]
        print(f"  {e['entity_id']:<28} {e['type']:<13} "
              f"{e['category']['value']:<22} sub={e.get('subcategory')}  "
              f"mentions={e['merge_provenance']['merged_mentions']} "
              f"confidence={conf if conf is not None else 'single'}")
        print(f"      forms: {e['surface_forms']}")
        for dd in e["details"]:
            ev = f" x{len(dd['evidence'])}" if len(dd["evidence"]) > 1 else ""
            print(f"      {dd['detail_type']:<11} {dd['normalized']} "
                  f"({'+'.join(dd['detected_by'])}){ev}")
        if e["flags"]:
            print(f"      flags: {e['flags']}")
    for u in d["unassigned_details"]:
        print(f"  UNASSIGNED {u['detail_type']:<11} {u['normalized']}")

# What the search app needs to find this run's sources again (app/server.py reads it)
Path(OUT_DIR, "run_info.json").write_text(json.dumps({
    "notes_dir": str(Path(NOTES_DIR).resolve()),
    "notes": {n["note_key"]: {"file": str(Path(n["path"]).resolve()), "claim_id": n["claim_id"]}
              for n in notes},
    "provider": PROVIDER, "model": DEPLOYMENT, "offline": OFFLINE_MODE,
    "dossier_lens": DOSSIER_LENS, "rarity": RARITY_SOURCE}, indent=1))
print(f"\ndossiers written to {OUT_DIR}/")


## 22 — Cross-claim view, occurrence-aware

Cross-claim identity is not a separate scorer any more: it is the same links, read across
claim boundaries. What this cell adds is the second reading of the same evidence —
**suspicion** — which depends on distance. Two claims in one occurrence are one incident;
shared parties there are expected and mean nothing as a fraud signal. The same party across
insurers means a great deal.

Each cross-claim cluster is shown at every lens, so the difference between "linked on an
NPI" and "linked on a rare name alone" is in front of the reader, with the co-parties that
also match by name (displayed, never scored).


In [ ]:
def cross_claim_clusters(pr):
    out = []
    for c in pr["clusters"]:
        cls = sorted({mention_by_key[k]["claim_id"] for k in c["members"]})
        if len(cls) < 2:
            continue
        cross_edges = [l for l in c["edges"]
                       if mention_by_key[l["a"]]["claim_id"] != mention_by_key[l["b"]]["claim_id"]]
        out.append({
            "cluster_id": c["id"], "claims": cls, "lens": pr["lens"],
            "names": sorted({mention_by_key[k]["name"] for k in c["members"]}),
            "type": mention_by_key[c["id"]]["type"],
            "members": c["members"],
            "weakest_link": _link_ref(c["weakest"]),
            "cross_links": [{**_link_ref(l), "suspicion": l["suspicion"],
                             "co_party": l["co_party"]} for l in cross_edges],
            "basis_classes": sorted({l["basis_class"] for l in cross_edges}),
            "suspicion": max((l["suspicion"] for l in cross_edges), default=0.0)})
    return sorted(out, key=lambda x: (-x["suspicion"], x["cluster_id"]))

cross_view = {lens: cross_claim_clusters(pr) for lens, pr in projections.items()}
for lens, cl in cross_view.items():
    print(f"\n=== lens {lens}: {len(cl)} cluster(s) span more than one claim ===")
    for x in cl[:20]:
        wl = x["weakest_link"]
        print(f"  {x['names'][:3]}  claims={len(x['claims'])}  basis={x['basis_classes']}  "
              f"weakest p={wl['p']}  suspicion={x['suspicion']}")
        for cl_ in x["cross_links"][:3]:
            ov = cl_["co_party"]["overlap"]
            if ov and ov["matched"]:
                print(f"      co-parties also matching by name: {ov['matched']} of {ov['of']} "
                      f"{ov['names']}")

# Shared details are relationships, not identities: a clinic and a doctor who answer the same
# phone are two parties, and the identity links above rightly never cross entity types. The
# sharing itself is still evidence an investigator wants, read with the same distance rule.
owners = defaultdict(set)
for r in mentions:
    for t, v, _ in r["details"]:
        if t in JOIN_TYPES:
            owners[(t, v)].add(r["key"])
shared_details = []
for (t, v), ks in sorted(owners.items()):
    cls = sorted({mention_by_key[k]["claim_id"] for k in ks})
    if len(cls) < 2:
        continue
    dists = {distance_of(mention_by_key[x], mention_by_key[y])
             for x in ks for y in ks if mention_by_key[x]["claim_id"] != mention_by_key[y]["claim_id"]}
    far = max(dists, key=lambda d: SUSPICION_WEIGHT[d])
    shared_details.append({"detail_type": t, "value": v, "claims": cls, "distance": far,
                           "suspicion": SUSPICION_WEIGHT[far],
                           "holders": [{"mention": k, "name": mention_by_key[k]["name"],
                                        "type": mention_by_key[k]["type"]} for k in sorted(ks)]})
print(f"\nshared details across claims: {len(shared_details)}")
for s_ in shared_details:
    print(f"  {s_['detail_type']:<8} {s_['value']:<34} {len(s_['claims'])} claims  {s_['distance']:<16} "
          f"suspicion={s_['suspicion']}  {[h['name'] for h in s_['holders']]}")

Path(OUT_DIR, "cross_claim.json").write_text(json.dumps(
    {"clusters": cross_view, "shared_details": shared_details}, indent=1))
print(f"\ncross-claim view written to {OUT_DIR}/cross_claim.json")


## 23 — Self-tests

Invariants the pipeline is supposed to hold, checked against this run and against synthetic
inputs built to break them. Every one of these corresponds to a defect that a run can
otherwise complete cleanly while carrying: wrong spans that pass the round trip, a veto that
vetoes nothing, a scoring band no input can reach.

Run this cell before believing any number in the summary.


In [ ]:
_tests, _failures = [], []

def test(name):
    def deco(fn):
        _tests.append((name, fn)); return fn
    return deco

@test("cleaning records length-preserving substitutions, not just collapses")
def _t():
    raw = "UNITED\tSTATES\rDISTRICT  COURT"
    clean, edits = clean_with_map(raw)
    assert clean == "UNITED STATES\nDISTRICT COURT", repr(clean)
    kinds = [e["kind"] for e in edits]
    assert kinds == ["tab_to_space", "cr_to_lf", "collapse_ws"], kinds
    probe = {"raw_text": raw, "clean_text": clean, "edits": edits}
    assert not check_offset_map(probe), check_offset_map(probe)

@test("cleaning off: the text the model reads is the source text, byte for byte")
def _t():
    raw = "UNITED\tSTATES  COURT\r\nline two"
    clean, edits = clean_none(raw)
    assert clean == raw and edits == []

@test("chunking covers the note with no gaps and overlaps by whole sentences")
def _t():
    global CHUNKING
    saved, CHUNKING = CHUNKING, True     # test the splitter even when a run has it off
    try:
        text = ("First sentence here. Second one follows. " * 40 + "\n\n") * 6
        chunks = split_note(text, limit=900, overlap_sents=2)
    finally:
        CHUNKING = saved
    assert len(chunks) > 1, "a 10k-char note with a 900-char limit must split"
    assert chunks[0]["start"] == 0 and chunks[-1]["end"] == len(text)
    for a, b in zip(chunks, chunks[1:]):
        assert b["core_start"] == a["end"], "cores must tile the note with no gap"
        assert b["start"] < b["core_start"], "each later chunk must re-read an overlap"
        assert text[b["start"]] in "FS", "overlap must start on a sentence, not mid-word"

@test("chunking off: every note goes whole, however long")
def _t():
    global CHUNKING
    saved = CHUNKING
    try:
        CHUNKING = False
        chunks = split_note("x " * 50000, limit=900)
        assert len(chunks) == 1 and chunks[0]["end"] == 100000
    finally:
        CHUNKING = saved

@test("overlap duplicates collapse to the earlier chunk and references follow")
def _t():
    note = {"chunks": [{}, {}],
            "spans": {"c0.m1": [10, 20], "c1.m1": [10, 20], "c1.d1": [30, 40]},
            "lane_llm": {
                "entity_mentions": [
                    {"mention_id": "c0.m1", "name": "Dr. Monroe", "quote": "x", "_chunk": [0, 50]},
                    {"mention_id": "c1.m1", "name": "Dr. Monroe", "quote": "x", "_chunk": [5, 90]}],
                "detail_mentions": [
                    {"detail_id": "c1.d1", "owner_ref": "c1.m1", "detail_type": "npi",
                     "raw_value": "1548392012", "_chunk": [5, 90]}],
                "action_mentions": []}}
    assert dedupe_overlap(note) == 1
    assert [m["mention_id"] for m in note["lane_llm"]["entity_mentions"]] == ["c0.m1"]
    assert note["lane_llm"]["detail_mentions"][0]["owner_ref"] == "c0.m1"

@test("fullest name: use the note's own full form, never a role word")
def _t():
    m = {"name": "Mr. Pierre", "type": "person",
         "occurrences": ["BRADLEY PIERRE", "Defendants", "him", "Defendant Pierre"]}
    assert fullest_name(m) == "BRADLEY PIERRE", fullest_name(m)
    m2 = {"name": "Weiner", "type": "person",
          "occurrences": ["Answering Defendants", "Defendants"]}
    assert fullest_name(m2) == "Weiner", "role words must never become the name"

@test("fullest name: a sentence fragment is never a name, and a list names no one")
def _t():
    m = {"name": "Nexray Medical Imaging, P.C.", "type": "organization",
         "occurrences": ["Nexray is not engaged in the practice", "Ninth Cause of Action against Nexray, Pierre, and Weiner"]}
    assert fullest_name(m) == "Nexray Medical Imaging, P.C.", fullest_name(m)
    m2 = {"name": "billed American Transit for these examinations", "type": "organization", "occurrences": []}
    assert fullest_name(m2) == "American Transit", fullest_name(m2)
    m3 = {"name": "Rutland, Pierre, and Moy", "type": "person", "occurrences": []}
    assert fullest_name(m3) is None, "a list of parties must not become one party's name"
    assert name_like("Nexray Medical Imaging, P.C. d/b/a Soul Radiology", "organization")
    assert not name_like("Rutland and Nexray", "organization"), "two organizations, not one"
    assert name_like("Johnson & Johnson", "organization")
    assert name_like("Schwartz, Conroy & Hack, PC", "organization")
    assert name_like("WILLIAM A. WEINER, D.O.", "person")

@test("GLiNER windows cover the whole note and never cut a word")
def _t():
    text = " ".join(f"word{i}" for i in range(3000))
    wins = gliner_windows(text, size=500, overlap=100)
    assert wins[0][0] == 0 and wins[-1][1] == len(text)
    for (s1, e1), (s2, e2) in zip(wins, wins[1:]):
        assert s2 < e1, "consecutive windows must overlap"
        assert text[s2 - 1] == " ", "a window must start on a word boundary"

@test("offset map: every clean index maps to the matching raw character")
def _t():
    bad = [b for n in notes for b in check_offset_map(n)]
    assert not bad, f"{len(bad)} mismatched indices, first: {bad[:3]}"

@test("round trip: every retained span's raw slice matches the quote it came from")
def _t():
    for n in notes:
        for mid, span in n["spans"].items():
            rs  = clean_to_raw(span[0], n["edits"], len(n["clean_text"]))
            re_ = clean_to_raw(span[1], n["edits"], len(n["clean_text"]))
            raw_slice, quote = n["raw_text"][rs:re_], n["quotes"][mid]
            assert quote_matches(n["span_methods"][mid], quote, raw_slice), \
                f"{n['note_key']}:{mid} ({n['span_methods'][mid]}) -> {raw_slice!r} != {quote!r}"

@test("round trip rejects a span pointing at the wrong text")
def _t():
    n = notes[0]
    probe = {"spans": {"probe": [0, 10]}, "edits": n["edits"],
             "clean_text": n["clean_text"], "raw_text": n["raw_text"],
             "quotes": {"probe": "a quote that is definitely not at offset zero"},
             "span_methods": {"probe": "exact"}}
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):
        passes, fails = round_trip(probe)
    assert not passes and len(fails) == 1, "a wrong span was accepted"

@test("quote resolution: normalized retry returns clean-text coordinates")
def _t():
    text = "prefix\n\n\n\n\n\n\n\n\n\nthe target phrase here"
    span, how = resolve_quote("The Target  Phrase", text)
    assert how == "normalized", how
    assert text[span[0]:span[1]] == "the target phrase", repr(text[span[0]:span[1]])

@test("quote resolution: fuzzy retry aligns to the match, not to a window boundary")
def _t():
    text = "aaaa bbbb the claimant attended the appointment on friday morning zz tail"
    span, how = resolve_quote("the claimant attended the appointment on friday mornings",
                              text)
    assert how.startswith("fuzzy"), how
    got = text[span[0]:span[1]]
    assert got.startswith("the claimant") and got.endswith("morning"), repr(got)

@test("quote resolution: an unresolvable quote is refused, not placed")
def _t():
    span, how = resolve_quote("billed under NPI 9999999999",
                              "nothing in this text resembles that at all")
    assert span is None and how.startswith("unverifiable_quote"), (span, how)

@test("quote resolution: soft hyphens and hyphenated line breaks meet on both sides")
def _t():
    text = ("Plaintiffs Allstate Indemnity Com\xad\npany, and Allstate Fire (collectively, "
            "\"All\xad state\") bring this action under the Racketeer-\ning statute.")
    for q, want in [("Allstate Indemnity Com-\npany", "Allstate Indemnity Com\xad\npany"),
                    ("Allstate Indemnity Company", "Allstate Indemnity Com\xad\npany"),
                    ("\"Allstate\") bring", "\"All\xad state\") bring"),
                    ("the Racketeering statute", "the Racketeer-\ning statute")]:
        span, how = resolve_quote(q, text)
        assert span and how == "normalized", (q, how)
        assert text[span[0]:span[1]] == want, (q, text[span[0]:span[1]])
        assert quote_matches(how, q, text[span[0]:span[1]])

@test("quote resolution: identical repeats go to the nearest anchor, else are kept and flagged")
def _t():
    rep = "MRC Defendants lack the knowledge or information sufficient to form a belief."
    text = f"1. Pierre. {rep}\n2. Moy. {rep}\n3. Nexray and more words here. {rep}"
    hits = [i for i in range(len(text)) if text.startswith(rep, i)]
    span, how = resolve_quote(rep, text, [hits[2] - 2])
    assert span[0] == hits[2] and how == "proximity", (span, how)
    span, how = resolve_quote(rep, text)                      # no anchor: kept, flagged
    assert span == [hits[0], hits[0] + len(rep)], span
    assert how == "exact:multi_hit_identical:3", how
    span, how = resolve_quote(rep, text, [hits[0] + 5, hits[1] + 5])
    assert span == [hits[0], hits[0] + len(rep)] and how == "exact:multi_hit_identical:3", how
    # each participant is its own anchor: the nearest copy to ANY of them, not to their mean
    span, how = resolve_quote(rep, text, [0, hits[2] - 3])
    assert span[0] == hits[2] and how == "proximity", (span, how)
    span, how = resolve_quote(rep.replace(" the ", "  the\n"), text)   # repeats after normalizing
    assert span and how == "normalized:multi_hit_identical:3", how

@test("quote resolution: the guarded snap takes whitespace/punctuation/case slips only")
def _t():
    text = ("477. Rutland, in turn, referred patients to Nexray, a CT scan and x-ray provider. "
            "478. American Transit has been injured in its busi ness and property in that it "
            "has paid at least $1,600,000.00 because of the fraudulent bills submitted through "
            "Nexray. 514. Rutland, Pierre, and Moy intentionally made the above-described false "
            "and fraudulent statements to induce American Transit to pay charges submitted "
            "through Rutland. 520. Nexray did not bill for the services on 3/14/2019 at all.")
    for q in ["Rutland , in turn, referred patients to Nexray , a CT scan and x -ray provider.",
              "American Transit has been injured in its business and property",
              "RUTLAND, IN TURN, REFERRED PATIENTS"]:
        span, how = resolve_quote(q, text)
        assert span and how.startswith(("loose", "normalized")), (q, how)
        assert loose_equal(q, text[span[0]:span[1]]) and quote_matches(how, q, text[span[0]:span[1]])
    refused = [
        # a name swapped: an attribution error to surface, not a typo to fix
        "Rutland, Moy, and Pierre intentionally made the above-described false and fraudulent "
        "statements to induce American Transit to pay charges submitted through Rutland.",
        # a number changed
        "has paid at least $1,500,000.00 because of the fraudulent bills submitted through Nexray.",
        # a negation dropped
        "520. Nexray did bill for the services on 3/14/2019 at all.",
        # a date changed where the digits alone would still agree
        "520. Nexray did not bill for the services on 31/4/2019 at all."]
    for q in refused:
        span, how = resolve_quote(q, text)
        assert span is None and how.startswith("unverifiable_quote"), (q, span, how)
    assert _guard_diff("Moy and Weiner made", "Rutland and Moy made") == "name_differs"
    assert _guard_diff("seen on friday mornings", "seen on friday morning") is None

@test("quote resolution: an edited entity quote still places the party's own name")
def _t():
    text = "Defendants include Nexray Medical Imaging, P.C. d/b/a Soul Radiology (\"Nexray\"), and others."
    probe = {"clean_text": text, "lane_llm": {
        "entity_mentions": [{"mention_id": "m1", "type": "organization", "occurrences": [],
                             "quote": "and Nexray Medical Imaging, P.C. d/b/a Soul   Radiology and",
                             "name": "Nexray Medical Imaging, P.C."}],
        "detail_mentions": [], "action_mentions": []}}
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):
        spans, methods, quotes, failed = resolve_note(probe)
    assert not failed and methods["m1"] == "name_fallback", (failed, methods)
    assert text[slice(*spans["m1"])] == "Nexray Medical Imaging, P.C."
    assert probe["quote_notices"][0]["flag"] == "quote_edited_name_placed"

@test("gemini schema translation: nullable unions, no additionalProperties")
def _t():
    g = to_gemini_schema(EXTRACTION_SCHEMA)
    issuer = g["properties"]["detail_mentions"]["items"]["properties"]["issuer"]
    assert issuer == {"type": "string", "nullable": True,
                      "description": issuer.get("description")}, issuer

    def walk(n):
        if isinstance(n, dict):
            assert "additionalProperties" not in n, "Gemini rejects additionalProperties"
            assert not isinstance(n.get("type"), list), f"untranslated union: {n['type']}"
            for v in n.values():
                walk(v)
        elif isinstance(n, list):
            for v in n:
                walk(v)
    walk(g)
    ent = g["properties"]["entity_mentions"]["items"]
    assert ent["propertyOrdering"] == list(ent["properties"]), ent["propertyOrdering"]
    assert ent["properties"]["type"]["enum"] == ENTITY_TYPES

@test("gemini schema translation refuses a union it cannot express")
def _t():
    try:
        to_gemini_schema({"type": ["string", "integer"]})
    except ValueError:
        return
    raise AssertionError("a two-way non-null union should not translate silently")

@test("name narrowing survives a line break inside the name")
def _t():
    text = "Defendants BRADLEY PIERRE, WILLIAM A. WEINER, \nD.O., and others."
    probe = {"clean_text": text, "lane_llm": {
        "entity_mentions": [{"mention_id": "m1", "quote": "PIERRE, WILLIAM A. WEINER, \nD.O., and",
                             "name": "WILLIAM A. WEINER, D.O.", "type": "person",
                             "occurrences": []}],
        "detail_mentions": [], "action_mentions": []}}
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):
        spans, _, quotes, _ = resolve_note(probe)
    s0, e0 = spans["m1"]
    assert text[s0:e0] == "WILLIAM A. WEINER, \nD.O.", repr(text[s0:e0])

@test("bar numbers keep their letters")
def _t():
    assert normalize_detail("bar_number", "MW7455") == "MW7455"
    assert normalize_detail("bar_number", "MW7455") != normalize_detail("bar_number", "XY7455")
    assert normalize_detail("bar_number", "ARDC 6224417") == "ARDC6224417"

@test("entity span narrows from the padded quote to the bare name")
def _t():
    text = "Clmt seen 3/14 by Dr. Monroe (NPI 1548392012). Referred to Lakeshore PT."
    probe = {"clean_text": text, "lane_llm": {
        "entity_mentions": [{"mention_id": "m1", "quote": "seen 3/14 by Dr. Monroe (NPI",
                             "name": "Dr. Monroe", "type": "person", "occurrences": []}],
        "detail_mentions": [], "action_mentions": []}}
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):
        spans, methods, quotes, failed = resolve_note(probe)
    s0, e0 = spans["m1"]
    assert text[s0:e0] == "Dr. Monroe", repr(text[s0:e0])
    assert quotes["m1"] == "Dr. Monroe", "round trip must compare against the name"

@test("both response schemas pass the strict-mode lint")
def _t():
    lint_strict_schema(EXTRACTION_SCHEMA)
    lint_strict_schema(CAT_SCHEMA)

@test("the category enum lets the model refuse")
def _t():
    assert "insufficient_evidence" in CAT_SCHEMA["properties"]["category"]["enum"]
    assert "insufficient_evidence" not in CATEGORIES, "refusal must not be a scored value"

@test("claim ids are parsed, not sliced")
def _t():
    assert parse_claim_id("LEGACY-4471902")["valid"] is False
    assert parse_claim_id("123456-789012-ab-01")["coverage_code"] == "AB"
    assert parse_claim_id("123456-789012-AB-01")["occurrence_id"] == "123456-789012"

@test("note filenames with an over-long note id are rejected, not truncated")
def _t():
    assert NOTE_FILE_RE.match("123456-789012-AB-01_188213.txt")
    m = NOTE_FILE_RE.match("123456-789012-AB-01_1882130.txt")
    assert m is None or m.group("note") == "1882130", "note id silently truncated"

@test("pattern lane: a captured value's span covers the value, not the cue")
def _t():
    probe = {"clean_text": "Attorney of record is J. Whitfield, ARDC 6224417."}
    finds = pattern_scan(probe)
    bar = [f for f in finds if f["detail_type"] == "bar_number"]
    assert bar, finds
    s, e = bar[0]["clean_span"]
    assert probe["clean_text"][s:e] == "6224417", repr(probe["clean_text"][s:e])

@test("pattern lane: a punctuated phone is not swallowed by the npi pattern")
def _t():
    probe = {"clean_text": "call (312) 555-0101 or NPI 1548392012 for records"}
    finds = {f["detail_type"]: f["raw_value"] for f in pattern_scan(probe)}
    assert finds.get("phone") == "(312) 555-0101", finds
    assert finds.get("npi") == "1548392012", finds

@test("checksum: a transposed NPI digit fails Luhn")
def _t():
    assert luhn_npi("1548392012") == "pass"
    assert luhn_npi("1548392018") == "FAIL_LUHN"
    assert luhn_npi("15483920") == "malformed"

@test("checksums: DEA check digit and ABA routing number")
def _t():
    assert dea_check("AK1234563") == "pass"
    assert dea_check("AK1234564") == "FAIL_DEA"
    assert dea_check("A1234563") == "malformed"
    assert aba_check("021000021") == "pass"
    assert aba_check("021000022") == "FAIL_ABA"
    assert aba_check("02100002") == "malformed"

@test("pattern lane: email, dob, plate, DEA, license and bank account, value spans only")
def _t():
    text = ("Claimant Maria Lopez (DOB 04/12/1979) saw Dr. Kessler, DEA AK1234563, NY medical "
            "license 212345, akessler@kesslerpain.com. Deposit: routing 021000021, account "
            "4417229108. NY plate KLM-4821. Her driver's license 55512345 and policy no. 7734412 "
            "are on file; the plate was dented; case CV1234567 was filed.")
    finds = pattern_scan({"clean_text": text})
    got = {f["detail_type"]: f for f in finds}
    for t, v in [("dob", "04/12/1979"), ("dea_number", "AK1234563"), ("state_license", "212345"),
                 ("email", "akessler@kesslerpain.com"), ("bank_account", "4417229108"),
                 ("license_plate", "KLM-4821")]:
        assert t in got and got[t]["raw_value"] == v, (t, got.get(t))
        s, e = got[t]["clean_span"]
        assert text[s:e] == v, (t, text[s:e])
    assert got["bank_account"]["issuer"] == "021000021" and got["bank_account"]["checksum"] == "pass"
    assert got["license_plate"]["issuer"] == "NY"
    assert got["dea_number"]["checksum"] == "pass" and got["dea_number"]["cue"]
    vals = [f["raw_value"] for f in finds]
    assert "55512345" not in vals, "a driver's license is not a professional license"
    assert "7734412" not in vals, "a policy number is not an identifier"
    assert "CV1234567" not in vals, "an uncued run whose DEA check digit fails is not a DEA number"
    assert sum(f["detail_type"] == "license_plate" for f in finds) == 1, "'plate was dented' is not a plate"
    assert not any(f["detail_type"] == "npi" for f in finds), "the account number must not read as an NPI"

@test("normalization: email, dob, plate, license, DEA and bank account")
def _t():
    today = _dt.date(2026, 9, 1)
    assert normalize_detail("email", "Mailto:AKessler@KesslerPain.com.") == "akessler@kesslerpain.com"
    assert normalize_detail("email", "not an address") is None
    for raw in ("04/12/1979", "4-12-1979", "1979-04-12", "19790412", "April 12, 1979",
                "12 Apr 1979", "Apr. 12 1979"):
        assert parse_date_iso(raw, today) == "1979-04-12", raw
    assert parse_date_iso("4/12/79", today) == "1979-04-12"
    assert parse_date_iso("4/12/05", today) == "2005-04-12"
    assert parse_date_iso("02/30/1980", today) is None, "not a real date"
    assert parse_date_iso("01/01/2031", today) is None, "a birth date cannot be in the future"
    assert normalize_detail("license_plate", "klm 4821", "New York") == "NY:KLM4821"
    assert normalize_detail("license_plate", "KLM-4821") == "KLM4821"
    assert normalize_detail("state_license", "MD-212345", "N.Y. State Education Dept.") == "NY:MD212345"
    assert normalize_detail("dea_number", "ak 123456-3") == "AK1234563"
    assert normalize_detail("dea_number", "1234563") is None
    assert normalize_detail("bank_account", "4417 2291 08", "021000021") == "021000021:4417229108"
    assert normalize_detail("bank_account", "routing 021000021 account 4417229108") == "021000021:4417229108"
    assert normalize_detail("bank_account", "4417229108") == "4417229108"
    assert checksum_of("bank_account", "021000022:4417229108") == "FAIL_ABA"
    assert checksum_of("dea_number", "AK1234563") == "pass"
    assert qualifiers_agree("license_plate", "NY:KLM4821", "KLM4821")
    assert not qualifiers_agree("license_plate", "NY:KLM4821", "NJ:KLM4821")

@test("scorer: email, plate, license and bank account are identifiers; a DEA conflict vetoes")
def _t():
    c2 = "200002-222222-AB-01"
    for t, v1, v2 in [("email", "akessler@kesslerpain.com", "akessler@kesslerpain.com"),
                      ("state_license", "NY:212345", "212345"),
                      ("bank_account", "021000021:4417229108", "4417229108"),
                      ("dea_number", "AK1234563", "AK1234563")]:
        l = _link(_mk("a", "Alan Kessler", details=[(t, v1, "stated")]),
                  _mk("b", "Alan Kessler", claim=c2, details=[(t, v2, "stated")]))
        assert l["basis_class"] == "identifier" and admits(l, "strict"), (t, l["basis_class"], l["p"])
    car = _link(_mk("a", "2019 Honda Civic", "vehicle", details=[("license_plate", "NY:KLM4821", "stated")]),
                _mk("b", "Honda Civic", "vehicle", claim=c2, details=[("license_plate", "KLM4821", "stated")]))
    assert car["basis_class"] == "identifier", car
    other_state = _link(_mk("a", "Alan Kessler", details=[("state_license", "NY:212345", "stated")]),
                        _mk("b", "Alan Kessler", claim=c2, details=[("state_license", "NJ:212345", "stated")]))
    assert other_state["basis_class"] != "identifier" and not other_state["veto"], other_state
    dea = _link(_mk("a", "Alan Kessler", details=[("dea_number", "AK1234563", "stated")]),
                _mk("b", "Alan Kessler", details=[("dea_number", "BK7654325", "stated")]))
    assert dea["veto"] and dea["veto"].startswith("conflicting_dea_number"), dea["veto"]
    assert "dea_number" in SINGULAR_TYPES and "dob" not in IDENTIFIER_BASIS

@test("date of birth: agreement adds weight, a difference counts against, neither vetoes")
def _t():
    c2 = "200002-222222-AB-01"
    base = _link(_mk("a", "Maria Lopez"), _mk("b", "Maria Lopez", claim=c2))
    same = _link(_mk("a", "Maria Lopez", details=[("dob", "1979-04-12", "stated")]),
                 _mk("b", "Maria Lopez", claim=c2, details=[("dob", "1979-04-12", "stated")]))
    diff = _link(_mk("a", "Maria Lopez", details=[("dob", "1979-04-12", "stated")]),
                 _mk("b", "Maria Lopez", claim=c2, details=[("dob", "1981-07-02", "stated")]))
    assert same["p"] > base["p"] > diff["p"], (same["p"], base["p"], diff["p"])
    assert same["basis_class"] == "dob" and "dob" in same["basis"], same
    assert not admits(same, "strict"), "a date of birth is not an identifier"
    assert not diff["veto"], "a differing date of birth never vetoes"
    assert diff["basis_class"] == "name_only"
    org = _link(_mk("a", "Alpha Clinic", "organization", details=[("dob", "1979-04-12", "stated")]),
                _mk("b", "Alpha Clinic", "organization", claim=c2, details=[("dob", "1981-07-02", "stated")]))
    assert not any(k.startswith("dob") for k in org["weights"]), "dates of birth are for persons"

@test("fixture note: every new identifier reaches its envelope normalized, with its owner")
def _t():
    env = next((e for e in envelopes if e["note_id"] == 190470), None)
    if env is None:
        return                               # a run over other notes
    got = {d["detail_type"]: d for d in env["detail_mentions"]}
    want = {"dob": "1979-04-12", "ssn": "123456789", "dea_number": "AK1234563",
            "state_license": "NY:212345", "email": "akessler@kesslerpain.com",
            "bank_account": "021000021:4417229108", "license_plate": "NY:KLM4821"}
    for t, v in want.items():
        assert t in got and got[t]["normalized"] == v, (t, got.get(t, {}).get("normalized"))
        assert got[t]["owner_ref"] != "UNASSIGNED", t
    assert got["dea_number"]["checksum"] == "pass" and got["bank_account"]["checksum"] == "pass"
    assert all("pattern" in got[t]["detected_by"] for t in want), \
        {t: got[t]["detected_by"] for t in want}

@test("reconcile: the same value twice in one note is not a phantom recall gap")
def _t():
    probe = {"clean_text": "call (312) 555-0101 today, or (312) 555-0101 after five",
             "lane_llm": {"entity_mentions": [], "action_mentions": [], "detail_mentions": [
                 {"detail_id": "d1", "quote": "call (312) 555-0101 today",
                  "raw_value": "(312) 555-0101", "detail_type": "phone",
                  "issuer": None, "owner_ref": "UNASSIGNED", "basis": "stated"},
                 {"detail_id": "d2", "quote": "or (312) 555-0101 after five",
                  "raw_value": "(312) 555-0101", "detail_type": "phone",
                  "issuer": None, "owner_ref": "UNASSIGNED", "basis": "stated"}]},
             "lane_gliner": []}
    probe["lane_pattern"] = pattern_scan(probe)
    probe["spans"] = {d["detail_id"]: resolve_quote(d["quote"], probe["clean_text"])[0]
                      for d in probe["lane_llm"]["detail_mentions"]}
    out = reconcile(probe)
    assert len(out) == 2, [d["raw_value"] for d in out]
    assert not any(d["recall_gap"] for d in out), out

def _mk(key, name, etype="person", claim="100001-111111-AB-01", note=None, details=(),
        roles=(), occurrences=(), chunk=0):
    return build_mention(key, etype, name, claim_id=claim, note_key=note or f"note:{key}",
                         occurrences=occurrences, details=details, roles=roles, chunk=chunk)

def _link(a, b):
    ls = link_mentions([a, b])
    return ls[0] if ls else None

@test("person names parse: titles, credentials, initials and 'Last, First'")
def _t():
    p = parse_person("WILLIAM A. WEINER, D.O.")
    assert (p["first"], p["middle"], p["last"], p["creds"]) == ("WILLIAM", "A", "WEINER", ["DO"]), p
    assert parse_person("Moy, Marvin")["first"] == "MARVIN"
    assert parse_person("Dr. A. Monroe")["first"] == "A" and parse_person("Dr. A. Monroe")["title"] == "DR"
    assert parse_person("Mr. Pierre") == {"first": "", "middle": "", "last": "PIERRE",
                                          "creds": [], "title": "MR"}

@test("organization aliases: d/b/a split, legal suffixes dropped")
def _t():
    assert org_aliases("Nexray Medical Imaging, P.C. d/b/a Soul Radiology") == \
        [["NEXRAY", "MEDICAL", "IMAGING"], ["SOUL", "RADIOLOGY"]]
    assert org_aliases("AMERICAN TRANSIT INS. CO.") == org_aliases("American Transit Insurance Company")

@test("a long d/b/a name is still a name, judged part by part")
def _t():
    full = "Nexray Medical Imaging, P.C. d/b/a Soul Radiology Medical Imaging"
    assert name_like(full, "organization")
    assert fullest_name({"name": full, "type": "organization", "occurrences": []}) == full
    assert not name_like("Nexray is not engaged d/b/a something", "organization")

@test("candidates: a short org form and a phonetic surname variant are proposed")
def _t():
    recs = [_mk("a", "Lakeshore PT", "organization"),
            _mk("b", "Lakeshore Physical Therapy", "organization"),
            _mk("c", "William Weiner"), _mk("d", "William Wiener")]
    cands, _ = candidate_pairs(recs)
    assert ("a", "b") in cands, sorted(cands)
    assert ("c", "d") in cands, sorted(cands)

@test("candidates: two mentions from one chunk of one note are never paired")
def _t():
    recs = [_mk("a", "Dr. Monroe", note="note:1"), _mk("b", "Dr. Monroe", note="note:1")]
    assert not candidate_pairs(recs)[0]
    recs[1]["chunk"] = 1
    assert candidate_pairs(recs)[0], "different chunks of one note must be compared"

@test("rarity: a rare surname agreeing outweighs a common one")
def _t():
    if RARITY_SOURCE["surnames"] == "FLAT":
        return
    rare = _link(_mk("a", "Pierre"), _mk("b", "Pierre", claim="100001-111111-AB-02"))
    common = _link(_mk("a", "Smith"), _mk("b", "Smith", claim="100001-111111-AB-02"))
    assert rare["weights"]["last"] > common["weights"]["last"] + 3, (rare["weights"], common["weights"])

@test("a name-only link keeps its basis however high it scores")
def _t():
    l = _link(_mk("a", "William A. Weiner, D.O."),
              _mk("b", "William A. Weiner, D.O.", claim="200002-222222-AB-01"))
    assert l["basis_class"] == "name_only" and l["basis"] == ["name"], l
    assert l["p"] > 0.5, f"a rare full professional name across insurers should score: {l['p']}"
    assert not admits(l, "strict"), "strict must admit identifier-backed links only"

@test("the same full name scores far lower for a private person than a professional")
def _t():
    pro = _link(_mk("a", "Maria Garcia", roles=["treating_physician"]),
                _mk("b", "Maria Garcia", claim="200002-222222-AB-01", roles=["treating_physician"]))
    priv = _link(_mk("a", "Maria Garcia", roles=["claimant"]),
                 _mk("b", "Maria Garcia", claim="200002-222222-AB-01", roles=["claimant"]))
    assert priv["p"] < 0.1 < pro["p"], (priv["p"], pro["p"])

@test("vetoes: two stated NPIs, or M.D. against D.O.")
def _t():
    l = _link(_mk("a", "Dr. Monroe", details=[("npi", "1548392012", "stated")]),
              _mk("b", "Dr. Monroe", details=[("npi", "1999999999", "stated")]))
    assert l["veto"] and l["veto"].startswith("conflicting_npi"), l["veto"]
    l2 = _link(_mk("a", "William Weiner, M.D."), _mk("b", "William Weiner, D.O."))
    assert l2["veto"] and "credential_conflict" in l2["veto"], l2["veto"]
    l3 = _link(_mk("a", "Dr. Monroe", details=[("npi", "1548392012", "inferred")]),
               _mk("b", "Dr. Monroe", details=[("npi", "1999999999", "stated")]))
    assert not l3["veto"], "an inferred owner is evidence against, not a veto"

@test("identifiers: a shared NPI reaches strict; a shared address alone never merges")
def _t():
    npi = _link(_mk("a", "Alpha Clinic", "organization", details=[("npi", "1548392012", "stated")]),
                _mk("b", "Alpha Clinic LLC", "organization", claim="200002-222222-AB-01",
                    details=[("npi", "1548392012", "stated")]))
    assert npi["basis_class"] == "identifier" and admits(npi, "strict"), npi
    addr = _link(_mk("a", "Alpha Clinic", "organization", details=[("address", "1100|w|lawrence", "stated")]),
                 _mk("b", "Zeta Holdings", "organization", claim="200002-222222-AB-01",
                     details=[("address", "1100|w|lawrence", "stated")]))
    assert not admits(addr, "default"), addr

@test("org names: a typo in a common word still matches; an extra rare word costs")
def _t():
    a = _mk("a", "Nexray Medical Imaging, P.C.", "organization")
    typo = _link(a, _mk("b", "NEXRAY MEDICAL IMANGING,PC.", "organization"))
    assert typo["name_similarity"] > 0.95, typo
    r1 = _link(_mk("c", "Rutland Medical P.C.", "organization"),
               _mk("d", "Rutland Medical", "organization", claim="200002-222222-AB-01"))
    r2 = _link(_mk("c", "Rutland Medical P.C.", "organization"),
               _mk("d", "Rutland Medical Plaza", "organization", claim="200002-222222-AB-01"))
    assert r1["p"] > r2["p"], (r1["p"], r2["p"])

@test("org names: sibling companies veto each other; an OCR split does not")
def _t():
    sib = _link(_mk("a", "Allstate Indemnity Company", "organization"),
                _mk("b", "Allstate Property & Casualty Insurance Company", "organization"))
    assert sib["veto"] and sib["veto"].startswith("distinct_org_names"), sib
    ocr = _link(_mk("a", "ALLSTATE FIRE & CASUAL TY INSURANCE COMP ANY", "organization"),
                _mk("b", "Allstate Fire & Casualty Insurance Company", "organization"))
    assert not ocr["veto"] and ocr["p"] > 0.99, ocr
    m = {"name": "Allstate Property and Casualty Insurance Company", "type": "organization",
         "occurrences": ["Allstate Indemnity Company", "Allstate"]}
    assert fullest_name(m) == m["name"]
    # a declared trade name is the same organization, never a sibling
    recs = [_mk("a", "Nexray Medical Imaging, P.C. d/b/a Soul Radiology Medical Imaging", "organization"),
            _mk("b", "Nexray Medical Imaging, P.C.", "organization"),
            _mk("c", "Soul Radiology Medical Imaging", "organization")]
    assert not any(l["veto"] for l in link_mentions(recs)), [l["veto"] for l in link_mentions(recs)]

@test("co-party A: only an identifier-backed co-party adds weight; loops cannot")
def _t():
    c1, c2 = "100001-111111-AB-01", "200002-222222-AB-01"
    base = [_mk("m1", "Dr. Monroe", claim=c1), _mk("m2", "Dr. Monroe", claim=c2)]
    anchored = base + [
        _mk("o1", "Lakeshore Physical Therapy", "organization", claim=c1,
            details=[("phone", "3125550101", "stated")]),
        _mk("o2", "Lakeshore Physical Therapy", "organization", claim=c2,
            details=[("phone", "3125550101", "stated")])]
    loop = base + [_mk("o1", "Lakeshore Physical Therapy", "organization", claim=c1),
                   _mk("o2", "Lakeshore Physical Therapy", "organization", claim=c2)]
    la = next(l for l in link_mentions(anchored) if {l["a"], l["b"]} == {"m1", "m2"})
    ll = next(l for l in link_mentions(loop) if {l["a"], l["b"]} == {"m1", "m2"})
    assert la["co_party"]["anchored"] and la["basis_class"] == "co_party", la
    assert not ll["co_party"]["anchored"] and ll["basis_class"] == "name_only", ll
    assert ll["co_party"]["overlap"]["matched"] == 1, "B must still display the name match"
    assert la["p"] > ll["p"]

def _leie(*rows):
    """A tiny watchlist table, loaded through load_watchlist like the real one."""
    import tempfile
    cols = ["record_id", "last", "first", "middle", "business", "general", "specialty", "npi",
            "city", "state", "excl_type", "excl_date"]
    with tempfile.TemporaryDirectory() as d:
        p = Path(d, "wl.csv")
        pd.DataFrame([{c: r.get(c, "") for c in cols} for r in rows], columns=cols).to_csv(p, index=False)
        return load_watchlist(p)

_PIERRE = {"record_id": "leie:t1", "last": "PIERRE", "first": "BRADLEY", "city": "LEWISBURG",
           "state": "PA", "general": "EMPLOYEE - PRIVATE S", "excl_type": "1128a3"}

def _wl_link(rec, wl):
    got = link_watchlist([rec], wl)[0]
    return got[0] if got else None

@test("watchlist prior: declared by the corpus party's role, private below professional")
def _t():
    assert PRIOR_P[("watchlist", "private")] < PRIOR_P[("watchlist", "recurring")]
    wl = _leie(_PIERRE)
    priv = _wl_link(_mk("m1", "Bradley Pierre", roles=["defendant"]), wl)
    pro = _wl_link(_mk("m1", "Dr. Bradley Pierre"), wl)
    assert priv["distance"] == "watchlist" and priv["prior_key"] == "watchlist/private", priv["prior_key"]
    assert pro["prior_key"] == "watchlist/recurring" and pro["p"] > priv["p"], (pro["p"], priv["p"])
    assert priv["basis_class"] == "name_only" and priv["b"] == "leie:t1" and priv["source"] == "oig_leie"
    assert admits(priv, "broad") and not admits(priv, "default"), \
        f"a rare full name alone, private person: a weak flag (p={priv['p']})"

@test("watchlist location: same state adds, a different state subtracts, neither vetoes")
def _t():
    assert parse_location("135-25F 79th Street, Suite 2B, Howard Beach, New York 11414") == \
        {"city": "HOWARD BEACH", "state": "NY"}
    assert parse_location("4410 N Broadway, Chicago IL 60640") == {"city": "CHICAGO", "state": "IL"}
    assert parse_location("100 Ring Road, Suite 211") is None
    wl = _leie(_PIERRE)
    def at(city, state):
        m = _mk("m1", "Bradley Pierre", roles=["defendant"])
        m["locations"] = [{"city": city, "state": state}] if state else []
        return _wl_link(m, wl)
    none, pa, city, ny = at("", ""), at("HARRISBURG", "PA"), at("LEWISBURG", "PA"), at("BROOKLYN", "NY")
    assert "location" not in none["weights"], "a missing location is no evidence"
    assert city["p"] > pa["p"] > none["p"] > ny["p"], (city["p"], pa["p"], none["p"], ny["p"])
    assert ny["weights"]["location"] < 0 and not ny["veto"], "a different state counts against, never vetoes"
    assert ny["name_agreement"]["location"] == "different_state"
    assert city["basis_class"] == "location" and "location" in city["basis"]
    c2 = "200002-222222-AB-01"
    a, b = _mk("a", "Bradley Pierre"), _mk("b", "Bradley Pierre", claim=c2)
    a["locations"], b["locations"] = [{"city": "", "state": "PA"}], [{"city": "", "state": "NY"}]
    assert "location" not in _link(a, b)["weights"], "location is compared only against a watchlist record"

@test("watchlist candidates: only corpus against watchlist, never two records or two mentions")
def _t():
    wl = _leie({"record_id": "leie:s1", "last": "SMITH", "first": "JOHN"},
               {"record_id": "leie:s2", "last": "SMITH", "first": "JOHN"},
               {"record_id": "leie:o1", "business": "NEXRAY MEDICAL IMAGING PC"})
    recs = [_mk("m1", "John Smith"), _mk("m2", "John Smith", claim="200002-222222-AB-01"),
            _mk("m3", "Nexray Medical Imaging, P.C.", "organization")]
    cands = watchlist_candidates(recs, wl)
    assert cands, "the corpus Smiths must meet the listed Smiths"
    ids, keys = set(wl["record_id"]), {r["key"] for r in recs}
    assert all(a in keys and b in ids for a, b in cands), sorted(cands)
    assert {("m1", "leie:s1"), ("m1", "leie:s2"), ("m2", "leie:s1"), ("m2", "leie:s2"),
            ("m3", "leie:o1")} <= set(cands), sorted(cands)

@test("watchlist identifiers: a shared NPI is identifier-backed; a different stated NPI vetoes")
def _t():
    same = _wl_link(_mk("m1", "Dr. A. Monroe", details=[("npi", "1548392012", "stated")]),
                    _leie({"record_id": "leie:n1", "last": "MONROE", "first": "ALBERT",
                           "npi": "1548392012", "general": "PHYSICIAN (MD, DO)"}))
    assert same["basis_class"] == "identifier" and admits(same, "strict"), same
    assert "shared:npi" in same["proposed_by"]
    other = _wl_link(_mk("m1", "Dr. Albert Monroe", details=[("npi", "1548392012", "stated")]),
                     _leie({"record_id": "leie:n2", "last": "MONROE", "first": "ALBERT",
                            "npi": "1999999999", "general": "PHYSICIAN (MD, DO)"}))
    assert other["veto"] and other["veto"].startswith("conflicting_npi"), other
    assert not any(admits(other, lens) for lens in LENSES)

@test("flagged for review: an admitted watchlist link flags the whole entity, at that lens only")
def _t():
    weak = {"a": "m2", "b": "leie:x", "p": 0.36, "basis_class": "name_only", "veto": None}
    npi = {"a": "m3", "b": "leie:y", "p": 0.99, "basis_class": "identifier", "veto": None}
    vetoed = {"a": "m1", "b": "leie:z", "p": 0.99, "basis_class": "name_only", "veto": "conflicting_npi"}
    by = {"m1": [vetoed], "m2": [weak], "m3": [npi]}
    assert flags_for(["m1", "m2"], by, "broad") == [weak]
    assert flags_for(["m1", "m2"], by, "default") == [] and flags_for(["m1", "m2"], by, "strict") == []
    assert flags_for(["m3"], by, "strict") == [npi]
    assert near_flags(["m1", "m2"], by, "default") == [vetoed, weak], "what the lens left out stays visible"
    # the run's own dossiers follow the same rule
    for es in entity_sets.values():
        for e in es:
            assert ("flagged_for_review" in e["flags"]) == bool(e["watchlist"]), e["entity_id"]

@test("decision card: every link's field rows add up to its bits, with m, u and values")
def _t():
    for l in links + watchlist_links:
        total = sum(r["bits"] for r in l["fields"] if r.get("bits") is not None)
        assert abs(total - l["bits"]) < 0.05, (l["a"], l["b"], total, l["bits"], l["weights"])
        assert 0 <= l["name_similarity_pct"] <= 100
    rare = _link(_mk("a", "Bradley Pierre"), _mk("b", "Bradley Pierre", claim="200002-222222-AB-01"))
    last = next(r for r in rare["fields"] if r["field"] == "last")
    assert last["level"] == "exact" and last["a"] == "PIERRE" and 0 < last["u"] < 1e-3, last
    assert abs(math.log2(last["m"] / last["u"]) - last["bits"]) < 0.05, last
    npi = _link(_mk("a", "Dr. Monroe", details=[("npi", "1548392012", "stated")]),
                _mk("b", "Dr. Monroe", details=[("npi", "1999999999", "stated")]))
    row = next(r for r in npi["fields"] if r.get("type") == "npi")
    assert row["level"] == "conflict" and row["veto"], row
    org = _link(_mk("a", "Nexray Medical Imaging, P.C.", "organization"),
                _mk("b", "Nexray Medical Imaging", "organization", claim="200002-222222-AB-01"))
    orow = next(r for r in org["fields"] if r["field"] == "org_name")
    assert {t["t"] for t in orow["shared"]} == {"NEXRAY", "MEDICAL", "IMAGING"}, orow

@test("projection: a veto blocks a transitive merge at every lens")
def _t():
    L = [{"a": "x", "b": "y", "p": 0.99, "basis_class": "identifier", "veto": None},
         {"a": "y", "b": "z", "p": 0.98, "basis_class": "identifier", "veto": None},
         {"a": "x", "b": "z", "p": 0.0, "basis_class": "none", "veto": "conflicting_npi"}]
    for lens in LENSES:
        pr = project(["x", "y", "z"], L, lens)
        assert sorted(len(c["members"]) for c in pr["clusters"]) == [1, 2], (lens, pr["clusters"])
        assert pr["refused"] and pr["refused"][0]["reason"] == "veto"

@test("projection: a link with no agreeing field is never admitted, whatever its prior")
def _t():
    l = {"a": "x", "b": "y", "p": 0.5, "basis_class": "none", "veto": None}
    assert not any(admits(l, lens) for lens in LENSES)

@test("projection: lenses nest — every strict cluster sits inside a default one, and so on")
def _t():
    for tight, loose in (("strict", "default"), ("default", "broad")):
        home = {k: c["id"] for c in projections[loose]["clusters"] for k in c["members"]}
        for c in projections[tight]["clusters"]:
            assert len({home[k] for k in c["members"]}) == 1, (tight, c["members"])

@test("projection: confidence is the weakest link on the connecting path")
def _t():
    L = [{"a": "a", "b": "b", "p": 0.99, "basis_class": "name_only", "veto": None},
         {"a": "b", "b": "c", "p": 0.85, "basis_class": "name_only", "veto": None},
         {"a": "c", "b": "d", "p": 0.95, "basis_class": "name_only", "veto": None}]
    c = project(list("abcd"), L, "default")["clusters"][0]
    assert c["weakest"]["p"] == 0.85
    assert subtree_weakest(c, ["c", "d"])["p"] == 0.95, "c-d does not pass through b-c"
    assert subtree_weakest(c, ["a", "d"])["p"] == 0.85

@test("projection: the cluster-size alarm refuses a runaway chain")
def _t():
    L = [{"a": f"k{i:03d}", "b": f"k{i + 1:03d}", "p": 0.99, "basis_class": "identifier",
          "veto": None} for i in range(60)]
    pr = project([f"k{i:03d}" for i in range(61)], L, "default", max_size=40)
    assert max(len(c["members"]) for c in pr["clusters"]) <= 40
    assert any(r["reason"] == "cluster_size" for r in pr["refused"])

@test("remap is idempotent: no entity carries a duplicated (type, value)")
def _t():
    for claim, es in entity_sets.items():
        for e in es:
            seen = [(d["detail_type"], d["normalized"]) for d in e["details"]]
            assert len(seen) == len(set(seen)), f"{e['entity_id']} duplicates: {seen}"

@test("entity ids are stable: numbering follows the lowest member key")
def _t():
    for claim, es in entity_sets.items():
        order = [e["merge_provenance"]["members"][0] for e in es]
        assert order == sorted(order), f"{claim}: numbering not member-ordered: {order}"

@test("every mention lands in exactly one entity of its claim")
def _t():
    for claim in claims:
        keys = [r["key"] for r in mentions if r["claim_id"] == claim]
        assert sorted(keys) == sorted(id_maps[claim]), claim

for name, fn in _tests:
    try:
        fn()
        print(f"  PASS  {name}")
    except Exception as ex:
        _failures.append((name, f"{type(ex).__name__}: {ex}"))
        print(f"  FAIL  {name}\n          {type(ex).__name__}: {ex}")

print(f"\n{len(_tests) - len(_failures)}/{len(_tests)} self-tests passed")
if _failures:
    raise AssertionError(f"{len(_failures)} self-test failure(s): "
                         + "; ".join(n for n, _ in _failures))


## 24 — What this notebook does not implement

The architecture trace describes more than this POC runs. Without this list a clean run
summary reads as a complete implementation.

| Architecture stage | State here |
|---|---|
| **UNASSIGNED retry** (2A) — one batched LLM call per claim that re-attempts ownership for unresolved details with full-claim context | **Absent.** Unresolved details stay `UNASSIGNED` in the dossier. The trace marks the retry optional in v1; the consequence is that a detail whose owner is only determinable from a *second* note is never attributed. |
| **Person projection** (2C) — canonical `Person` nodes recomputed from `IdentityLink`s | **Implemented, at read time** (cell 19, `goko/projection.py`). Nothing is stored merged; strict / default / broad lenses, weakest-link confidence. The search app calls the same code. |
| **Cluster-size alarm** (2C) — suspends a projection whose member count is implausible | **Implemented** as a refused union at `MAX_CLUSTER` mentions, recorded in `blocked_merges` and flagged `cluster_size_alarm` on the entity. |
| **Watchlist matching** (2C) — match a Person against the watchlist, alert above threshold | **Implemented against OIG LEIE** (cell 18c) as "Flagged for review": each exclusion record is a mention record, linked with the same candidates, scorer, rarity and vetoes, and flagged at the reader's lens. **Not implemented:** alert records, review decisions on a flag, a refresh schedule for the list (the table is a dated snapshot), other lists (SAM, state Medicaid exclusions), and address or date-of-birth comparison against the list (street address, ZIP and DOB are left out of the committed table). Only 11% of LEIE records carry an NPI, so most flags rest on a name alone, and say so. |
| **Decision review** — an investigator accepts or rejects a link or flag, and a rerun respects it | **Absent.** The app is read-only. Every link and flag is inspectable (decision card) but not correctable. |
| **Graph load** (2C) — `MERGE` of Detail / Entity / Note / Evidence nodes into the store | **Absent.** The cross-claim query is done in memory over the dossiers, which is equivalent at this scale and not equivalent at archive scale. |
| **Oversized-note chunking** | **Implemented, optional** (cell 9b, `CHUNKING`). Structure-first splitting with whole-sentence overlap; spans in note coordinates; overlap duplicates removed. With it off, a note too long to extract is counted as a failure, never a crash. |
| **Rarity** | **Implemented from outside references** (Census, SSA, NPPES), never from the corpus. |
| **Weight calibration** | The m-values and priors in cell 18 are stated assumptions. Cell 18b (disabled) would learn them with EM once gold data and volume exist. |
| **Collective resolution** (co-party option C) | **Absent by design.** Co-party evidence counts only when anchored on an identifier (A) and is otherwise displayed (B). Joint resolution can confirm a loop of coincidences. |

One structural point: **two mentions from the same chunk of one note are never compared.**
Within a chunk, coreference is the model's job, and two mentions it kept apart are its
assertion that they differ. If the model is wrong about that, nothing downstream notices.
Mentions from different chunks of one split note *are* compared (distance `same_note`).


## 25 — Run summary

What to look at first: whether the run is valid at all, then review-queue volume, recall
gaps, and any round-trip failure.


In [ ]:
print("=" * 68)
print("RUN SUMMARY")
print("=" * 68)
print(f"mode                 : {'OFFLINE REPLAY (fixtures)' if OFFLINE_MODE else 'LIVE'}")
print(f"model / deployment   : {DEPLOYMENT}")
print(f"notes processed      : {len(notes)}")
print(f"claims               : {len(claims)}")
print(f"occurrences          : {len(occurrences)}")
print(f"entities resolved    : {sum(len(d['entities']) for d in dossiers.values())}")
print(f"actions              : {sum(len(d['actions']) for d in dossiers.values())}")
print(f"unassigned details   : {sum(len(d['unassigned_details']) for d in dossiers.values())}")
print(f"identity links       : {len(links)}  "
      f"({sum(1 for l in links if l['basis_class'] == 'identifier')} identifier, "
      f"{sum(1 for l in links if l['basis_class'] == 'name_only')} name-only, "
      f"{sum(1 for l in links if l['veto'])} vetoed)")
for _lens in LENSES:
    print(f"  cross-claim @{_lens:<8}: {len(cross_view[_lens])} cluster(s)")
print(f"rarity               : {RARITY_SOURCE}")
_fl = sorted({e["cluster_id"] for d in dossiers.values() for e in d["entities"]
              if "flagged_for_review" in e["flags"]})
print(f"flagged for review   : {len(_fl)} entit{'y' if len(_fl) == 1 else 'ies'} at lens "
      f"'{DOSSIER_LENS}' ({len(watchlist_links)} watchlist links kept of "
      f"{watchlist_scored} scored against {watchlist_size:,} OIG LEIE records)")

ef = len(extraction_failures)
rt = sum(len(n["rt_fail"]) for n in notes)
qf = sum(len(n["quote_failures"]) for n in notes)
rg = len([d for n in notes for d in n["details"] if d["recall_gap"]])
ck = len([d for n in notes for d in n["details"] if str(d["checksum"]).startswith("FAIL")])
vp = sum(len(n["validation"]) for n in notes)
bm = sum(len(v) for v in blocked_merges.values())
dp = sum(len(v) for v in dangling_participants.values())
ie = len([e for d in dossiers.values() for e in d["entities"]
          if e["category"]["value"] == "insufficient_evidence"])
cf = len([e for d in dossiers.values() for e in d["entities"]
          if e["category"]["value"] == "CATEGORY_CALL_FAILED"])

print("\nvalidity")
print(f"  extraction failures  : {ef}   <- non-zero means this run is PARTIAL")
if "_tests" in dir():
    print(f"  self-tests           : {len(_tests) - len(_failures)}/{len(_tests)} passed")
else:
    print( "  self-tests           : NOT RUN — run cell 23 before trusting anything here")
print(f"  category call failures: {cf}  <- non-zero means categories are PARTIAL")

print("\nquality signals")
print(f"  round-trip failures  : {rt}   <- span dropped, mention sent to review")
print(f"  unresolved quotes    : {qf}   <- watch this; no overlap means no second chance")
print(f"  recall gaps (Lane A) : {rg}   <- what the LLM missed and patterns caught")
print(f"  checksum failures    : {ck}   <- data quality in the source notes")
print(f"  validation problems  : {vp}   <- enum violations here mean a pipeline bug")
print(f"  blocked merges       : {bm}   <- must_not_link refusals, with reasons")
print(f"  dangling participants: {dp}   <- action pointing at a mention that resolved to nothing")
print(f"  insufficient_evidence: {ie}   <- the model declining to guess; a real answer")

acts = [a["action_type"] for d in dossiers.values() for a in d["actions"]]
subs = [e.get("subcategory") for d in dossiers.values() for e in d["entities"]
        if e.get("subcategory")]
print("\noverflow channels — cluster these to decide what to promote")
print(f"  action_type values   : {sorted(set(acts))}")
print(f"  subcategory values   : {sorted(set(subs))}")

print()
if ef or cf:
    if ef:
        print(f"!! {ef} of {len(notes)} notes produced nothing. Read nothing above as a")
        print("!! measurement: the corpus this run scored is not the corpus you loaded.")
    if cf:
        print(f"!! {cf} category call(s) failed. Those entities have no category, which is")
        print("!! not the same as the model declining to assign one.")
elif OFFLINE_MODE:
    print("Offline replay: the plumbing ran end to end. No model was called, so none of")
    print("this measures extraction quality. Set OFFLINE_MODE = False for that.")
else:
    print("Run complete. See cell 24 for the architecture stages this POC does not cover.")
print(f"\nall artifacts in {OUT_DIR}/")
